# Code: *When do entropy constraints add information? An energy–entropy residual audit for phase-change operator learning*

R. Musah, S. S. Abukari and Jude Bayor Simons — submitted to the *International Journal for Numerical Methods in Fluids*.

This notebook is self-contained. It writes every solver, training, analysis, and plotting script to disk from the cells below, then runs them in order to regenerate all results of the paper: Figures 1–8, Tables I–V, the numbers quoted in the text, and the values in Appendices A–H.

**Folder layout.** Place this notebook in the root of the reproduction package (the folder that contains `data_eq.npz`, `runs/`, and `v2/`). All paths are relative to that root.

**Run modes.** Training all models and running the fine-grid solvers takes many CPU-hours, so the stages are switched by the flags in the next cell.

| Flag | What it does | Approximate cost (2 CPU cores) |
|---|---|---|
| default (all flags `False` except `RUN_FAST_CHECKS`) | recomputes every statistic, audit, table, and figure from the saved per-run records, checkpoints, and solver outputs shipped with the package; reruns the fast solver checks | about 5 min |
| `RUN_FAST_CHECKS` | manufactured solution, mobility and seed sweeps, corruption test, velocity-reconstruction audit, discrete-gradient checks | included above |
| `RUN_SOLVERS` | one-dimensional verification, de Vahl Davis cavity, all two-dimensional melting grids including $192^2$ (restartable) | 3–4 h |
| `RUN_DATA` | regenerates the one- and two-dimensional training datasets | about 1 h |
| `RUN_TRAINING` | retrains all 1D and 2D models (96 1D runs, 62 2D runs) | about 15–20 h |

With every flag set to `True` the notebook reproduces the study from scratch. Training is deterministic on CPU for a given seed; for example, retraining two of the two-dimensional runs reproduced their stored test errors exactly.

## 0. Configuration

In [1]:
import os, sys, json, glob, time, shlex, shutil, subprocess
ROOT = os.path.abspath(os.environ.get('POF_ROOT', os.getcwd()))   # folder with data_eq.npz, runs/, v2/
V = os.path.join(ROOT, 'v2')

RUN_FAST_CHECKS = True    # quick solver checks (~10 min)
RUN_SOLVERS     = False   # long solver runs (hours)
RUN_DATA        = False   # regenerate datasets (~1 h)
RUN_TRAINING    = False   # retrain every model (~15-20 h)
N_PARALLEL      = 2       # concurrent training jobs

for d in [ROOT, V, os.path.join(V, 'runs'), os.path.join(V, 'runs2d'), os.path.join(V, 'runs2d_active'),
          os.path.join(V, 'runs2d_r4'), os.path.join(V, 'runs2d_r4check'), os.path.join(ROOT, 'runs'),
          os.path.join(ROOT, 'figures'), os.path.join(ROOT, 'tables')]:
    os.makedirs(d, exist_ok=True)
print('ROOT =', ROOT)
missing = [f for f in ['data_eq.npz', 'data_kin.npz', 'v2/data_2d.npz'] if not os.path.exists(os.path.join(ROOT, f))]
print('datasets missing:', missing or 'none')

ROOT = /home/rabiu/Downloads/Musah_IJNMF
datasets missing: ['data_eq.npz', 'data_kin.npz', 'v2/data_2d.npz']


In [2]:
import platform, importlib
print('python', platform.python_version())
for m in ['numpy', 'scipy', 'jax', 'jaxlib', 'optax', 'matplotlib', 'pandas']:
    try: print(f'{m:11s}', importlib.import_module(m).__version__)
    except Exception as e: print(f'{m:11s} NOT INSTALLED -> pip install {m}')
# versions used for the paper: numpy 2.5.3, scipy 1.18.1, jax/jaxlib 0.12.0.1, optax 0.2.8, matplotlib 3.11.2

python 3.10.19
numpy       2.0.2


scipy       1.15.3


jax         0.6.2
jaxlib      0.6.2
optax       0.2.8


matplotlib  3.10.8


pandas      2.3.3


In [ ]:
ENV = dict(os.environ, XLA_FLAGS='--xla_cpu_multi_thread_eigen=false intra_op_parallelism_threads=1',
           OMP_NUM_THREADS='1', PYTHONWARNINGS='ignore', JAX_PLATFORMS='cpu')

def writefile(rel, src):
    # write an embedded script; the original absolute prefix is mapped onto ROOT
    path = os.path.join(ROOT, rel)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, 'w') as fh:
        fh.write(src.replace('/home/claude/pof', ROOT))

def run_py(rel, args='', cwd=None, tail=12, check=True):
    # run one script of the package and show the end of its output
    t0 = time.time()
    p = subprocess.run([sys.executable, os.path.join(ROOT, rel)] + shlex.split(args.replace('{ROOT}', ROOT)),
                       cwd=cwd or V, env=ENV, capture_output=True, text=True)
    lines = [l for l in (p.stdout + p.stderr).splitlines() if 'Warning' not in l and l.strip()]
    print(f'[{rel} {args[:60]}] {time.time() - t0:.0f} s, exit {p.returncode}')
    print('\n'.join(lines[-tail:]))
    if check and p.returncode != 0:
        raise RuntimeError(f'{rel} failed')

def run_jobs(jobs, n=None):
    # run (output, script, args) jobs in parallel, skipping outputs that already exist
    n = n or N_PARALLEL
    todo = [j for j in jobs if not os.path.exists(os.path.join(ROOT, j[0]))]
    print(f'{len(jobs) - len(todo)} of {len(jobs)} outputs present; running {len(todo)}')
    running = []
    while todo or running:
        while todo and len(running) < n:
            out, script, args = todo.pop(0)
            cmd = [sys.executable, os.path.join(ROOT, script)] + shlex.split(args.replace('{ROOT}', ROOT))
            running.append((out, subprocess.Popen(cmd, cwd=V, env=ENV, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)))
        for item in list(running):
            if item[1].poll() is not None:
                running.remove(item); print('done' if item[1].returncode == 0 else 'FAILED', item[0])
        time.sleep(2)

## 1. Source code

Each cell below writes one script of the study. Scripts in the package root are the original one-dimensional solvers and verification; scripts in `v2/` are the solvers, training, statistics, audits, and figures used for the final paper. The `make_*.py` scripts regenerate summary files (`analysis2.json`, `active2d.json`, `floors_skip.json`, `sharp_limit.json`, `dissipation.json`, `vdd.json`) whose original producers were interactive; each was checked to reproduce the stored file exactly.

**`solvers1d.py`** — One-dimensional equilibrium and internal-variable solvers, Neumann solution, discrete residuals.

In [ ]:
writefile('solvers1d.py', r'''"""1D reference solvers and discrete thermodynamic residuals for the PoF study.

Nondimensional units: rho = c_p = k = 1 (alpha = 1). Absolute temperature scale
with melting temperature T_m = 1. Domain x in [0, 1].

Equilibrium closure:  f_eq(T) = 0.5*(1 + tanh((T - T_m)/w)),  h = (T - T_ref) + L*f_eq(T)
Internal-variable closure:  h = (T - T_ref) + L*f,  df/dt = M(T,f) * X(T),
    X(T) = L*(1/T_m - 1/T),  M = M0*f*(1-f) + Meps.
"""
import numpy as np
from scipy.special import erf
from scipy.optimize import brentq

T_M = 1.0
T_REF = 1.0


# ----------------------------------------------------------------------------- equilibrium closure
def f_eq(T, w):
    return 0.5 * (1.0 + np.tanh((T - T_M) / w))


def df_eq(T, w):
    c = np.cosh((T - T_M) / w)
    return 0.5 / (w * c * c)


def h_eq(T, L, w):
    return (T - T_REF) + L * (f_eq(T, w) - f_eq(T_REF, w))


def dh_eq(T, L, w):
    return 1.0 + L * df_eq(T, w)


def make_s_eq_table(L, w, Tmin=0.6, Tmax=1.8, n=800001):
    """Compatible entropy s_eq(T) = int_{T_ref}^{T} h'(th)/th dth, tabulated.

    The table (piecewise-linear interpolation) is the *independent* discrete
    entropy constitutive operator used by r_S; it is NOT algebraically tied to
    the enthalpy used by r_E.
    """
    Tg = np.linspace(Tmin, Tmax, n)
    integrand = dh_eq(Tg, L, w) / Tg
    S = np.concatenate([[0.0], np.cumsum(0.5 * (integrand[1:] + integrand[:-1]) * np.diff(Tg))])
    S -= np.interp(T_REF, Tg, S)
    return Tg, S


def s_eq(T, table):
    Tg, S = table
    return np.interp(T, Tg, S)


def T_of_h(H, L, w, Tmin=0.5, Tmax=2.5):
    """Invert h_eq (monotone) cell-wise via a precomputed fine table."""
    Tg = np.linspace(Tmin, Tmax, 40001)
    Hg = h_eq(Tg, L, w)
    return np.interp(H, Hg, Tg)


# ----------------------------------------------------------------------------- equilibrium enthalpy solver
def solve_equilibrium(N, T_h, L, w, t_end, cfl=0.25, T_init=T_M, store_every=None):
    """Explicit FV enthalpy method. Left wall Dirichlet T_h, right wall adiabatic.

    Update: H^{n+1}_i = H^n_i + dt*(F_{i-1/2} - F_{i+1/2})/dx with F = -dT/dx at faces.
    Returns dict with times, T snapshots, dx, dt.
    """
    dx = 1.0 / N
    x = (np.arange(N) + 0.5) * dx
    dt = cfl * dx * dx
    nsteps = int(np.ceil(t_end / dt))
    dt = t_end / nsteps

    T = np.full(N, T_init, dtype=float)
    H = h_eq(T, L, w)
    snaps_T, snaps_t = [], []
    if store_every is None:
        store_every = max(1, nsteps // 200)

    # cumulative boundary heat input for the global energy defect
    Qin = 0.0
    qin_hist = [0.0]

    Tg_inv = np.linspace(0.4, 3.0, 60001)
    Hg_inv = h_eq(Tg_inv, L, w)

    for n in range(nsteps):
        # face fluxes F = -dT/dx; ghost: left Dirichlet (T_h at wall), right adiabatic
        q = np.empty(N + 1)
        q[1:-1] = -(T[1:] - T[:-1]) / dx
        q[0] = -(T[0] - T_h) / (0.5 * dx)     # wall at x=0, half-cell distance
        q[-1] = 0.0
        H = H - dt * (q[1:] - q[:-1]) / dx
        Qin += dt * q[0]                      # heat entering at left wall (+x inflow)
        T = np.interp(H, Hg_inv, Tg_inv)
        if (n + 1) % store_every == 0 or n == nsteps - 1:
            snaps_T.append(T.copy())
            snaps_t.append((n + 1) * dt)
            qin_hist.append(Qin)

    return dict(x=x, dx=dx, dt=dt, t=np.array(snaps_t), T=np.array(snaps_T),
                Qin=np.array(qin_hist[1:]), T_h=T_h, L=L, w=w)


def stefan_lambda(Ste):
    """One-phase Stefan: sqrt(pi)*lam*exp(lam^2)*erf(lam) = Ste."""
    g = lambda lam: np.sqrt(np.pi) * lam * np.exp(lam ** 2) * erf(lam) - Ste
    return brentq(g, 1e-8, 5.0)


def stefan_front(t, Ste):
    lam = stefan_lambda(Ste)
    return 2.0 * lam * np.sqrt(t)


def front_from_T(x, T, level=T_M, w=None, from_f=None):
    """Interface = f=1/2 contour, i.e. T = T_M for the tanh closure."""
    prof = f_eq(T, w) - 0.5 if from_f is None else from_f - 0.5
    sign = np.sign(prof)
    idx = np.where(np.diff(sign) < 0)[0]
    if len(idx) == 0:
        return 0.0 if prof[0] < 0 else x[-1]
    i = idx[0]
    x0, x1, p0, p1 = x[i], x[i + 1], prof[i], prof[i + 1]
    return x0 + (x1 - x0) * p0 / (p0 - p1)


# ----------------------------------------------------------------------------- discrete residuals (numpy)
def discrete_residuals_eq(T0, T1, dt, dx, L, w, s_table, T_h=None):
    """Finite-volume residuals over one step, implicit-Euler time quotient.

    r_E uses the enthalpy closure; r_S is discretized *independently*:
    tabulated s_eq, arithmetic-mean face temperature in the entropy flux,
    face-gradient conductive production. Returns (r_E, r_S, Tc) on interior cells.
    """
    N = len(T0)
    def face_grad(T):
        g = np.empty(N + 1)
        g[1:-1] = (T[1:] - T[:-1]) / dx
        g[0] = (T[0] - T_h) / (0.5 * dx) if T_h is not None else 0.0
        g[-1] = 0.0
        return g

    g1 = face_grad(T1)
    divq = -(g1[1:] - g1[:-1]) / dx                      # div(-K grad T)
    r_E = (h_eq(T1, L, w) - h_eq(T0, L, w)) / dt + divq

    Tf = np.empty(N + 1)
    Tf[1:-1] = 0.5 * (T1[1:] + T1[:-1])
    Tf[0] = T_h if T_h is not None else T1[0]
    Tf[-1] = T1[-1]
    qf = -g1                                             # q = -grad T at faces
    ent_flux_div = ((qf / Tf)[1:] - (qf / Tf)[:-1]) / dx
    sigma = 0.5 * ((g1[:-1] ** 2) + (g1[1:] ** 2)) / T1 ** 2
    r_S = (s_eq(T1, s_table) - s_eq(T0, s_table)) / dt + ent_flux_div - sigma
    return r_E, r_S


# ----------------------------------------------------------------------------- internal-variable solver
def X_drive(T, L):
    return L * (1.0 / T_M - 1.0 / T)


def mobility(T, f, M0, Meps):
    return M0 * f * (1.0 - f) + Meps


def solve_kinetic(N, T_h, L, M0, Meps, t_end, cfl=0.25, T_init=T_M, f_init=0.0, store_every=None):
    """Explicit coupled solver: c_p dT/dt = Txx - L df/dt;  df/dt = M X, f clipped-free.

    Bound preservation: mobility vanishes at f=0,1 up to Meps floor; a conservative
    limiter caps df so f stays in [0,1] and the energy update uses the *limited* df.
    """
    dx = 1.0 / N
    x = (np.arange(N) + 0.5) * dx
    dt = cfl * dx * dx
    nsteps = int(np.ceil(t_end / dt))
    dt = t_end / nsteps

    T = np.full(N, T_init, dtype=float)
    f = np.full(N, f_init, dtype=float)
    snaps_T, snaps_f, snaps_t = [], [], []
    if store_every is None:
        store_every = max(1, nsteps // 200)
    Qin = 0.0
    qin_hist = []

    for n in range(nsteps):
        q = np.empty(N + 1)
        q[1:-1] = -(T[1:] - T[:-1]) / dx
        q[0] = -(T[0] - T_h) / (0.5 * dx)
        q[-1] = 0.0
        lap = -(q[1:] - q[:-1]) / dx
        rate = mobility(T, f, M0, Meps) * X_drive(T, L)
        df = dt * rate
        df = np.clip(df, -f, 1.0 - f)        # conservative limiter (enters energy too)
        T = T + dt * lap - L * df
        f = f + df
        Qin += dt * q[0]
        if (n + 1) % store_every == 0 or n == nsteps - 1:
            snaps_T.append(T.copy()); snaps_f.append(f.copy()); snaps_t.append((n + 1) * dt)
            qin_hist.append(Qin)

    return dict(x=x, dx=dx, dt=dt, t=np.array(snaps_t), T=np.array(snaps_T),
                f=np.array(snaps_f), Qin=np.array(qin_hist), T_h=T_h, L=L, M0=M0, Meps=Meps)


def discrete_residuals_kin(T0, f0, T1, f1, dt, dx, L, M0, Meps, T_h=None):
    """r_E, r_f, sigma_ph over one macro step (implicit-Euler quotients)."""
    N = len(T0)
    g1 = np.empty(N + 1)
    g1[1:-1] = (T1[1:] - T1[:-1]) / dx
    g1[0] = (T1[0] - T_h) / (0.5 * dx) if T_h is not None else 0.0
    g1[-1] = 0.0
    divq = -(g1[1:] - g1[:-1]) / dx
    dfdt = (f1 - f0) / dt
    r_E = (T1 - T0) / dt + L * dfdt + divq
    r_f = dfdt - mobility(T1, f1, M0, Meps) * X_drive(T1, L)
    sigma_ph = (X_drive(T1, L) / 1.0) * dfdt * 1.0  # rho L (1/Tm - 1/T) df/dt = X * df/dt
    return r_E, r_f, sigma_ph


# ----------------------------------------------------------------------------- two-phase Neumann (equal properties)
def neumann_lambda(T_h, T_i, L):
    """Two-phase Stefan (c_p=k=rho=1): wall T_h, melting T_m=1, initial solid T_i<T_m.
    sqrt(pi)*lam*L = (T_h-T_m)*exp(-lam^2)/erf(lam) - (T_m-T_i)*exp(-lam^2)/erfc(lam)."""
    from scipy.special import erfc
    g = lambda lam: ((T_h - T_M) * np.exp(-lam ** 2) / erf(lam)
                     - (T_M - T_i) * np.exp(-lam ** 2) / erfc(lam)
                     - np.sqrt(np.pi) * lam * L)
    return brentq(g, 1e-9, 3.0)


def neumann_front(t, T_h, T_i, L):
    return 2.0 * neumann_lambda(T_h, T_i, L) * np.sqrt(t)


def neumann_T(x, t, T_h, T_i, L):
    from scipy.special import erfc
    lam = neumann_lambda(T_h, T_i, L)
    eta = x / (2.0 * np.sqrt(t))
    Tl = T_h - (T_h - T_M) * erf(eta) / erf(lam)
    Ts = T_i + (T_M - T_i) * erfc(eta) / erfc(lam)
    return np.where(eta <= lam, Tl, Ts)
''')

**`datagen.py`** — One-dimensional training datasets (Table II).

In [ ]:
writefile('datagen.py', r'''"""Generate 1D trajectory datasets (equilibrium + kinetic closures)."""
import numpy as np, sys, json
sys.path.insert(0, '/home/claude/pof')
from solvers1d import solve_equilibrium, solve_kinetic

rng = np.random.default_rng(0)
N = 128; T_i = 0.95; t_end = 0.1; w = 0.02; M0 = 400.0
NMACRO = 25

def macro_states(sol):
    # subsample stored snapshots to NMACRO states incl. initial
    T = sol['T']; t = sol['t']
    idx = np.linspace(0, len(t) - 1, NMACRO).round().astype(int)
    return t[idx], idx

# ---------------- equilibrium dataset: 90 (T_h, L) combos
Th_vals = np.linspace(1.10, 1.40, 9)
L_vals = np.linspace(1.0, 3.0, 10)
combos = [(th, l) for th in Th_vals for l in L_vals]
rng.shuffle(combos)
split = dict(train=combos[:60], val=combos[60:75], test=combos[75:])

def gen_eq(combolist):
    X = []
    for (th, l) in combolist:
        sol = solve_equilibrium(N=N, T_h=th, L=l, w=w, t_end=t_end, T_init=T_i, store_every=50)
        t, idx = macro_states(sol)
        states = np.concatenate([np.full((1, N), T_i), sol['T'][idx[1:] - 0]], axis=0) \
            if False else np.vstack([np.full((1, N), T_i), sol['T'][idx]])[-NMACRO:]
        # simpler: prepend exact initial state then take stored macro states 1..end
        T0 = np.full((1, N), T_i)
        Ts = np.vstack([T0, sol['T'][np.linspace(0, len(sol['t']) - 1, NMACRO - 1).round().astype(int)]])
        X.append(dict(T=Ts, T_h=th, L=l, t=np.concatenate([[0.0], sol['t'][np.linspace(0, len(sol['t']) - 1, NMACRO - 1).round().astype(int)]])))
    return X

def pack_eq(traj):
    T = np.stack([d['T'] for d in traj])            # (n, NMACRO, N)
    th = np.array([d['T_h'] for d in traj])
    L = np.array([d['L'] for d in traj])
    t = np.stack([d['t'] for d in traj])
    return T, th, L, t

data = {}
for k, cl in split.items():
    T, th, L, t = pack_eq(gen_eq(cl))
    data[f'{k}_T'] = T; data[f'{k}_th'] = th; data[f'{k}_L'] = L; data[f'{k}_t'] = t
np.savez_compressed('/home/claude/pof/data_eq.npz', **data, w=w, T_i=T_i, N=N)
dtm = data['train_t'][0]
print("equilibrium dataset:", {k: v.shape for k, v in data.items() if hasattr(v, 'shape')})
print("macro dt =", np.diff(dtm)[1:].mean())

# ---------------- kinetic dataset: 72 combos
Th_vals = np.linspace(1.15, 1.40, 8)
L_vals = np.linspace(1.5, 2.5, 9)
combos = [(th, l) for th in Th_vals for l in L_vals]
rng.shuffle(combos)
split = dict(train=combos[:48], val=combos[48:60], test=combos[60:])

def gen_kin(combolist):
    out = []
    for (th, l) in combolist:
        sol = solve_kinetic(N=N, T_h=th, L=l, M0=M0, Meps=0.0, t_end=t_end,
                            T_init=T_i, f_init=0.01, store_every=50)
        sel = np.linspace(0, len(sol['t']) - 1, NMACRO - 1).round().astype(int)
        Ts = np.vstack([np.full((1, N), T_i), sol['T'][sel]])
        fs = np.vstack([np.full((1, N), 0.01), sol['f'][sel]])
        out.append(dict(T=Ts, f=fs, T_h=th, L=l,
                        t=np.concatenate([[0.0], sol['t'][sel]])))
    return out

data = {}
for k, cl in split.items():
    traj = gen_kin(cl)
    data[f'{k}_T'] = np.stack([d['T'] for d in traj])
    data[f'{k}_f'] = np.stack([d['f'] for d in traj])
    data[f'{k}_th'] = np.array([d['T_h'] for d in traj])
    data[f'{k}_L'] = np.array([d['L'] for d in traj])
    data[f'{k}_t'] = np.stack([d['t'] for d in traj])
np.savez_compressed('/home/claude/pof/data_kin.npz', **data, M0=M0, T_i=T_i, N=N)
print("kinetic dataset:", {k: v.shape for k, v in data.items() if hasattr(v, 'shape')})
''')

**`verify1d.py`** — One-dimensional verification and discrete identity (Fig. 1a–c, Fig. 2a,b).

In [ ]:
writefile('verify1d.py', r'''"""Verification of the 1D equilibrium enthalpy solver and the discrete identity."""
import json
import numpy as np
import sys
sys.path.insert(0, '/home/claude/pof')
from solvers1d import (solve_equilibrium, front_from_T, h_eq, f_eq, make_s_eq_table,
                       discrete_residuals_eq, neumann_front, neumann_lambda, T_M)

out = {}
T_i = 0.95          # subcooled solid initial temperature
w = 0.01
t_end = 0.0625

# ---------------------------------------------------------------- 1. Neumann comparison
cases = [dict(T_h=1.10, L=2.0), dict(T_h=1.20, L=2.0), dict(T_h=1.30, L=1.0)]
stef = []
for c in cases:
    Ste = (c['T_h'] - T_M) / c['L']
    sol = solve_equilibrium(N=256, T_h=c['T_h'], L=c['L'], w=w, t_end=t_end,
                            T_init=T_i, store_every=40)
    fr_num = np.array([front_from_T(sol['x'], Ti, w=w) for Ti in sol['T']])
    fr_ana = neumann_front(sol['t'], c['T_h'], T_i, c['L'])
    rel = np.abs(fr_num[-1] - fr_ana[-1]) / fr_ana[-1]
    stef.append(dict(Ste=Ste, T_h=c['T_h'], L=c['L'],
                     lam=neumann_lambda(c['T_h'], T_i, c['L']),
                     t=sol['t'].tolist(), front_num=fr_num.tolist(),
                     front_ana=fr_ana.tolist(), rel_err_final=float(rel)))
    print(f"Ste={Ste:.3f}: front num={fr_num[-1]:.4f} ana={fr_ana[-1]:.4f} rel={rel:.2e}")
out['stefan'] = stef

# ---------------------------------------------------------------- 2. refinement
conv = []
T_h0, L0 = 1.2, 2.0
fa = neumann_front(t_end, T_h0, T_i, L0)
for N in [32, 64, 128, 256, 512]:
    sol = solve_equilibrium(N=N, T_h=T_h0, L=L0, w=w, t_end=t_end, T_init=T_i,
                            store_every=10 ** 9)
    fr = front_from_T(sol['x'], sol['T'][-1], w=w)
    conv.append(dict(N=N, rel_front_err=float(abs(fr - fa) / fa)))
    print(f"N={N}: front rel err {conv[-1]['rel_front_err']:.3e}")
out['grid_refinement'] = conv

wconv = []
for ww in [0.04, 0.02, 0.01, 0.005]:
    sol = solve_equilibrium(N=512, T_h=T_h0, L=L0, w=ww, t_end=t_end, T_init=T_i,
                            store_every=10 ** 9)
    fr = front_from_T(sol['x'], sol['T'][-1], w=ww)
    wconv.append(dict(w=ww, rel_front_err=float(abs(fr - fa) / fa)))
    print(f"w={ww}: front rel err {wconv[-1]['rel_front_err']:.3e}")
out['width_refinement'] = wconv

# ---------------------------------------------------------------- 3. solver energy defect
sol = solve_equilibrium(N=256, T_h=T_h0, L=L0, w=w, t_end=t_end, T_init=T_i, store_every=40)
dx = sol['dx']
E = np.array([np.sum(h_eq(Ti, L0, w)) * dx for Ti in sol['T']])
E0 = np.sum(h_eq(np.full(256, T_i), L0, w)) * dx
defect = (E - E0) - sol['Qin']
scale = np.abs(sol['Qin'][-1])
out['energy_defect'] = dict(t=sol['t'].tolist(), defect_rel=(defect / scale).tolist(),
                            max_rel=float(np.max(np.abs(defect)) / scale))
print("max relative energy defect of solver:", out['energy_defect']['max_rel'])

# ---------------------------------------------------------------- 4. identity, manufactured fields
def T_manu(x, t):
    return 1.0 + 0.25 * np.sin(np.pi * x) * np.exp(-t) + 0.05 * np.cos(2 * np.pi * x) * (1 + t)

L_id, w_id = 2.0, 0.02
s_table = make_s_eq_table(L_id, w_id)
ident = []
t0 = 0.10
for N in [32, 64, 128, 256, 512, 1024]:
    dxn = 1.0 / N
    dtn = 0.5 * dxn * dxn
    x = (np.arange(N) + 0.5) * dxn
    T0, T1 = T_manu(x, t0), T_manu(x, t0 + dtn)
    r_E, r_S = discrete_residuals_eq(T0, T1, dtn, dxn, L_id, w_id, s_table, T_h=None)
    sl = slice(2, -2)
    num = np.linalg.norm((r_S - r_E / T1)[sl])
    den = np.linalg.norm((r_E / T1)[sl])
    ident.append(dict(N=N, dx=dxn, eps_id_rel=float(num / den)))
    print(f"N={N}: eps_id (rel) = {num/den:.3e}")
e = np.array([d['eps_id_rel'] for d in ident]); hh = np.array([d['dx'] for d in ident])
p = np.polyfit(np.log(hh[:5]), np.log(e[:5]), 1)[0]
out['identity_manufactured'] = dict(points=ident, observed_order=float(p))
print("observed order of eps_id (first 5 pts):", p)

# ---------------------------------------------------------------- 5. identity on solver trajectories
# normalized by the entropy-storage term scale (r_E on solver fields is near zero,
# so a residual-relative normalization is ill-posed there)
from solvers1d import s_eq
id_sol = []
for N in [128, 256, 512]:
    sol = solve_equilibrium(N=N, T_h=T_h0, L=L0, w=0.02, t_end=t_end, T_init=T_i, store_every=5)
    s_tab = make_s_eq_table(L0, 0.02)
    vals = []
    for k in range(5, len(sol['t']) - 1, 10):
        dts = sol['t'][k + 1] - sol['t'][k]
        T0s, T1s = sol['T'][k], sol['T'][k + 1]
        r_E, r_S = discrete_residuals_eq(T0s, T1s, dts, sol['dx'], L0, 0.02, s_tab, T_h=T_h0)
        term_scale = np.linalg.norm((s_eq(T1s, s_tab) - s_eq(T0s, s_tab)) / dts)
        vals.append(np.linalg.norm(r_S - r_E / T1s) / term_scale)
    id_sol.append(dict(N=N, eps_id_termrel_median=float(np.median(vals))))
    print(f"solver trajectory N={N}: median eps_id/term = {np.median(vals):.3e}")
out['identity_on_solver'] = id_sol

json.dump(out, open('/home/claude/pof/verify1d.json', 'w'), indent=1)
print("saved verify1d.json")
''')

**`solver2d.py`** — First two-dimensional solver (used for the $C_m$ sweep and two-dimensional identity points).

In [ ]:
writefile('solver2d.py', r'''"""2D enthalpy-porosity Boussinesq melting in a square cavity (MAC projection).

Nondimensionalization: length H, time H^2/alpha, velocity alpha/H,
theta = (T - T_m)/(T_h - T_m). Model:

  du/dt + u.grad u = -grad p + Pr lap u + Ra Pr theta e_y - A(f) u
  div u = 0
  dHt/dt + div(u Ht) = lap theta,   Ht = theta + f/Ste,  f = 0.5(1+tanh(theta/w_s))
  A(f) = Cm (1-f)^2 / (f^3 + eps_m)

Left wall theta = 1, right wall theta = theta_c, top/bottom adiabatic; no-slip walls;
initial subcooled quiescent solid theta_i. Staggered MAC grid, first-order upwind
advection, explicit diffusion, semi-implicit face drag, Chorin projection with a
DCT-II Neumann Poisson solve. The energy update is in conservative flux form with
Ht as the primary variable, so the discrete global energy balance is exact up to
round-off; Qin is accumulated from the same discrete wall fluxes.
"""
import numpy as np
from scipy.fft import dctn, idctn

DTH = 0.2  # absolute DeltaT for mapping to absolute temperature: T = 1 + DTH*theta


def f_eq(th, ws):
    return 0.5 * (1.0 + np.tanh(th / ws))


class NeumannPoisson:
    def __init__(self, nc, h):
        i = np.arange(nc)
        lam = 2.0 * (np.cos(np.pi * i / nc) - 1.0) / h ** 2
        self.den = lam[:, None] + lam[None, :]
        self.den[0, 0] = 1.0

    def solve(self, rhs):
        w = dctn(rhs, type=2, norm='ortho')
        w = w / self.den
        w[0, 0] = 0.0
        return idctn(w, type=2, norm='ortho')


def run(nc=96, Ra=1e5, Pr=1.0, Ste=0.2, ws=0.02, Cm=1e6, eps_m=1e-3,
        theta_i=-0.1, theta_c=-0.1, t_end=0.1, out_times=None, cfl=0.3):
    h = 1.0 / nc
    xc = (np.arange(nc) + 0.5) * h
    u = np.zeros((nc + 1, nc))     # x-faces
    v = np.zeros((nc, nc + 1))     # y-faces
    th = np.full((nc, nc), theta_i)
    f = f_eq(th, ws)
    Ht = th + f / Ste
    pois = NeumannPoisson(nc, h)

    thg = np.linspace(-4.0, 4.0, 32001)
    Hg = thg + f_eq(thg, ws) / Ste

    def theta_of(Htf):
        return np.interp(Htf, Hg, thg)

    def ghosts(th):
        gl = 2.0 * 1.0 - th[0, :]         # left Dirichlet theta=1
        gr = 2.0 * theta_c - th[-1, :]    # right Dirichlet
        return gl, gr

    if out_times is None:
        out_times = list(np.linspace(0, t_end, 7)[1:])
    else:
        out_times = list(out_times)
    snaps = []
    capture_next = False
    Qin = 0.0
    hist = dict(t=[], Nu=[], fbar=[], umax=[], Qin=[], E=[], usolid=[], uliq=[])
    E0 = Ht.sum() * h * h
    dt_dif = cfl * h * h / (4.0 * max(1.0, Pr))
    t = 0.0; step = 0

    while t < t_end - 1e-13:
        umax = max(np.abs(u).max(), np.abs(v).max(), 1e-9)
        dt = min(dt_dif, cfl * h / umax, t_end - t)

        # ---- face liquid fraction and drag
        fu = np.ones((nc + 1, nc)); fu[1:-1] = 0.5 * (f[1:] + f[:-1]); fu[0] = f[0]; fu[-1] = f[-1]
        fv = np.ones((nc, nc + 1)); fv[:, 1:-1] = 0.5 * (f[:, 1:] + f[:, :-1]); fv[:, 0] = f[:, 0]; fv[:, -1] = f[:, -1]
        Au = Cm * (1 - fu) ** 2 / (fu ** 3 + eps_m)
        Av = Cm * (1 - fv) ** 2 / (fv ** 3 + eps_m)

        # ---- momentum predictor (padded fields carry no-slip ghosts)
        upad = np.zeros((nc + 3, nc + 2))
        upad[1:-1, 1:-1] = u
        upad[:, 0] = -upad[:, 1]; upad[:, -1] = -upad[:, -2]    # no-slip at y walls
        lap_u = (upad[2:, 1:-1] - 2 * u + upad[:-2, 1:-1] +
                 upad[1:-1, 2:] - 2 * u + upad[1:-1, :-2]) / h ** 2
        dudx_m = (u - upad[:-2, 1:-1]) / h
        dudx_p = (upad[2:, 1:-1] - u) / h
        dudy_m = (u - upad[1:-1, :-2]) / h
        dudy_p = (upad[1:-1, 2:] - u) / h
        v_at_u = np.zeros_like(u)
        v_at_u[1:-1, :] = 0.25 * (v[1:, :-1] + v[1:, 1:] + v[:-1, :-1] + v[:-1, 1:])
        adv_u = np.where(u > 0, u * dudx_m, u * dudx_p) + \
                np.where(v_at_u > 0, v_at_u * dudy_m, v_at_u * dudy_p)
        u_star = (u + dt * (-adv_u + Pr * lap_u)) / (1.0 + dt * Au)
        u_star[0, :] = 0.0; u_star[-1, :] = 0.0

        vpad = np.zeros((nc + 2, nc + 3))
        vpad[1:-1, 1:-1] = v
        vpad[0, 1:-1] = -vpad[1, 1:-1]; vpad[-1, 1:-1] = -vpad[-2, 1:-1]  # no-slip x walls
        lap_v = (vpad[2:, 1:-1] - 2 * v + vpad[:-2, 1:-1] +
                 vpad[1:-1, 2:] - 2 * v + vpad[1:-1, :-2]) / h ** 2
        dvdx_m = (v - vpad[:-2, 1:-1]) / h
        dvdx_p = (vpad[2:, 1:-1] - v) / h
        dvdy_m = (v - vpad[1:-1, :-2]) / h
        dvdy_p = (vpad[1:-1, 2:] - v) / h
        u_at_v = np.zeros_like(v)
        u_at_v[:, 1:-1] = 0.25 * (u[:-1, 1:] + u[1:, 1:] + u[:-1, :-1] + u[1:, :-1])
        th_at_v = np.zeros_like(v)
        th_at_v[:, 1:-1] = 0.5 * (th[:, 1:] + th[:, :-1])
        adv_v = np.where(u_at_v > 0, u_at_v * dvdx_m, u_at_v * dvdx_p) + \
                np.where(v > 0, v * dvdy_m, v * dvdy_p)
        v_star = (v + dt * (-adv_v + Pr * lap_v + Ra * Pr * th_at_v)) / (1.0 + dt * Av)
        v_star[:, 0] = 0.0; v_star[:, -1] = 0.0

        # ---- projection
        div = (u_star[1:, :] - u_star[:-1, :]) / h + (v_star[:, 1:] - v_star[:, :-1]) / h
        phi = pois.solve(div / dt)
        u = u_star; v = v_star
        u[1:-1, :] = u[1:-1, :] - dt * (phi[1:, :] - phi[:-1, :]) / h
        v[:, 1:-1] = v[:, 1:-1] - dt * (phi[:, 1:] - phi[:, :-1]) / h

        # ---- energy (conservative flux form, Ht primary)
        gl, gr = ghosts(th)
        qx = np.zeros((nc + 1, nc))
        qx[1:-1] = -(th[1:] - th[:-1]) / h
        qx[0] = -(th[0] - gl) / h
        qx[-1] = -(gr - th[-1]) / h
        qy = np.zeros((nc, nc + 1))
        qy[:, 1:-1] = -(th[:, 1:] - th[:, :-1]) / h
        Fx = np.zeros((nc + 1, nc))
        Fx[1:-1] = np.where(u[1:-1] > 0, Ht[:-1], Ht[1:]) * u[1:-1]
        Fy = np.zeros((nc, nc + 1))
        Fy[:, 1:-1] = np.where(v[:, 1:-1] > 0, Ht[:, :-1], Ht[:, 1:]) * v[:, 1:-1]
        Gx = Fx + qx; Gy = Fy + qy
        Ht = Ht - dt * ((Gx[1:, :] - Gx[:-1, :]) / h + (Gy[:, 1:] - Gy[:, :-1]) / h)
        Qin += dt * (qx[0].sum() * h - qx[-1].sum() * h)
        th = theta_of(Ht)
        f = f_eq(th, ws)

        t += dt; step += 1
        if step % 100 == 0 or t >= t_end - 1e-13:
            glh, _ = ghosts(th)
            Nu = np.sum(-(th[0, :] - glh) / h) * h
            liq = f > 0.95; sol = f < 0.05
            uc = 0.5 * (u[1:, :] + u[:-1, :]); vc = 0.5 * (v[:, 1:] + v[:, :-1])
            spd = np.hypot(uc, vc)
            hist['t'].append(t); hist['Nu'].append(Nu)
            hist['fbar'].append(f.mean()); hist['umax'].append(umax)
            hist['Qin'].append(Qin); hist['E'].append(Ht.sum() * h * h)
            hist['usolid'].append(spd[sol].max() if sol.any() else 0.0)
            hist['uliq'].append(spd[liq].max() if liq.any() else 0.0)
        if capture_next and snaps:
            snaps[-1]['next'] = dict(t=t, th=th.copy(), f=f.copy(), Ht=Ht.copy(),
                                     uf=u.copy(), vf=v.copy(), dt=dt)
            capture_next = False
        if out_times and t >= out_times[0] - 1e-13:
            uc = 0.5 * (u[1:, :] + u[:-1, :]); vc = 0.5 * (v[:, 1:] + v[:, :-1])
            snaps.append(dict(t=t, th=th.copy(), f=f.copy(), u=uc, v=vc, Ht=Ht.copy(),
                              uf=u.copy(), vf=v.copy()))
            out_times.pop(0)
            capture_next = True

    hist = {k: np.array(vv) for k, vv in hist.items()}
    return dict(xc=xc, h=h, snaps=snaps, E0=E0, steps=step, params=dict(
        nc=nc, Ra=Ra, Pr=Pr, Ste=Ste, ws=ws, Cm=Cm, eps_m=eps_m,
        theta_i=theta_i, theta_c=theta_c, t_end=t_end), **hist)
''')

**`run2d.py`** — First two-dimensional production runs: $C_m$ sensitivity and identity on solver steps.

In [ ]:
writefile('run2d.py', r'''"""Production 2D runs: main case, resolution study, Cm sensitivity, identity and
corrupted-surrogate diagnostics."""
import sys, json, pickle
import numpy as np
sys.path.insert(0, '/home/claude/pof')
from solver2d import run, f_eq, DTH
from scipy.ndimage import gaussian_filter

OUT = {}
SNAP_T = [0.05, 0.10, 0.15, 0.20, 0.25]
BASE = dict(Ra=1e5, Pr=1.0, Ste=0.2, ws=0.02, Cm=1e6, eps_m=1e-3,
            theta_i=-0.1, theta_c=-0.1, t_end=0.25)

# -------------------------------------------------- absolute-variable residuals on a pair
def residuals_pair(s, L_eff, Ste, ws, h, theta_c):
    """r_E and r_S over one solver step, absolute variables T=1+DTH*theta.
    Flux-form material derivative matching the solver for r_E; r_S discretized
    independently (upwind entropy advection, arithmetic face T in q/T)."""
    th0, Ht0 = s['th'], s['Ht']
    nx = s['next']
    th1, Ht1, dt = nx['th'], nx['Ht'], nx['dt']
    uf, vf = nx['uf'], nx['vf']
    nc = th0.shape[0]
    T0 = 1.0 + DTH * th0; T1 = 1.0 + DTH * th1
    H0 = DTH * Ht0; H1 = DTH * Ht1                     # absolute enthalpy (c=1, h = DTH*Ht)
    # faces of T1 with Dirichlet ghosts
    gl = 2 * (1 + DTH * 1.0) - T1[0, :]; gr = 2 * (1 + DTH * theta_c) - T1[-1, :]
    qx = np.zeros((nc + 1, nc)); qx[1:-1] = -(T1[1:] - T1[:-1]) / h
    qx[0] = -(T1[0] - gl) / h; qx[-1] = -(gr - T1[-1]) / h
    qy = np.zeros((nc, nc + 1)); qy[:, 1:-1] = -(T1[:, 1:] - T1[:, :-1]) / h
    Fx = np.zeros((nc + 1, nc)); Fx[1:-1] = np.where(uf[1:-1] > 0, H1[:-1], H1[1:]) * uf[1:-1]
    Fy = np.zeros((nc, nc + 1)); Fy[:, 1:-1] = np.where(vf[:, 1:-1] > 0, H1[:, :-1], H1[:, 1:]) * vf[:, 1:-1]
    divE = ((Fx + qx)[1:, :] - (Fx + qx)[:-1, :]) / h + ((Fy + qy)[:, 1:] - (Fy + qy)[:, :-1]) / h
    r_E = (H1 - H0) / dt + divE

    # independent entropy discretization: s = ln T + L_eff * g(T), g = int f'(T)/T dT
    Tg = np.linspace(0.5, 1.6, 400001)
    fg = 0.5 / (ws * DTH) / np.cosh((Tg - 1.0) / (ws * DTH)) ** 2   # df/dT in absolute T
    integ = fg / Tg
    G = np.concatenate([[0.0], np.cumsum(0.5 * (integ[1:] + integ[:-1]) * np.diff(Tg))])
    G -= np.interp(1.0, Tg, G)
    def s_of(T): return np.log(T) + L_eff * np.interp(T, Tg, G)
    s0, s1 = s_of(T0), s_of(T1)
    Tfx = np.zeros((nc + 1, nc)); Tfx[1:-1] = 0.5 * (T1[1:] + T1[:-1])
    Tfx[0] = 1 + DTH * 1.0; Tfx[-1] = 1 + DTH * theta_c
    Tfy = np.ones((nc, nc + 1)); Tfy[:, 1:-1] = 0.5 * (T1[:, 1:] + T1[:, :-1])
    Tfy[:, 0] = T1[:, 0]; Tfy[:, -1] = T1[:, -1]
    Sx = np.zeros((nc + 1, nc)); Sx[1:-1] = np.where(uf[1:-1] > 0, s1[:-1], s1[1:]) * uf[1:-1]
    Sy = np.zeros((nc, nc + 1)); Sy[:, 1:-1] = np.where(vf[:, 1:-1] > 0, s1[:, :-1], s1[:, 1:]) * vf[:, 1:-1]
    GxS = Sx + qx / Tfx; GyS = Sy + qy / Tfy
    divS = (GxS[1:, :] - GxS[:-1, :]) / h + (GyS[:, 1:] - GyS[:, :-1]) / h
    gx2 = 0.5 * ((qx[:-1, :] ** 2) + (qx[1:, :] ** 2))
    gy2 = 0.5 * ((qy[:, :-1] ** 2) + (qy[:, 1:] ** 2))
    sigma = (gx2 + gy2) / T1 ** 2
    r_S = (s1 - s0) / dt + divS - sigma
    term = np.linalg.norm((s1 - s0) / dt)
    return r_E, r_S, T1, term

def ident_metrics(r_E, r_S, T1, term):
    d = np.linalg.norm(r_S - r_E / T1)
    return float(d / term)

# -------------------------------------------------- main case nc=96
print("main nc=96 ...")
r96 = run(nc=96, out_times=SNAP_T, **BASE)
d = (r96['E'] - r96['E0']) - r96['Qin']
OUT['main'] = dict(nc=96, steps=r96['steps'],
                   energy_defect_rel=float(np.abs(d).max() / abs(r96['Qin'][-1])),
                   fbar_final=float(r96['fbar'][-1]), Nu_final=float(r96['Nu'][-1]),
                   usolid_max=float(r96['usolid'].max()), uliq_max=float(r96['uliq'].max()),
                   t=r96['t'].tolist(), Nu=r96['Nu'].tolist(), fbar=r96['fbar'].tolist())
print(" fbar(t_end)=", OUT['main']['fbar_final'], " Nu=", OUT['main']['Nu_final'],
      " defect=", OUT['main']['energy_defect_rel'],
      " usolid/uliq=", OUT['main']['usolid_max'], OUT['main']['uliq_max'])
pickle.dump(dict(snaps=r96['snaps'], xc=r96['xc'], h=r96['h'], t=r96['t'],
                 Nu=r96['Nu'], fbar=r96['fbar']), open('/home/claude/pof/r96.pkl', 'wb'))

# identity on solver pairs at each resolution
L_eff = DTH / BASE['Ste']
ids = []
for nc in [48, 96]:
    r = r96 if nc == 96 else run(nc=nc, out_times=SNAP_T, **BASE)
    vals = []
    for s in r['snaps']:
        if 'next' not in s: continue
        rE, rS, T1, term = residuals_pair(s, L_eff, BASE['Ste'], BASE['ws'], r['h'], BASE['theta_c'])
        vals.append(ident_metrics(rE, rS, T1, term))
    ids.append(dict(nc=nc, eps_id_termrel_median=float(np.median(vals)), all=vals))
    print(f" identity nc={nc}: median {np.median(vals):.3e}")
    if nc == 48:
        r48 = r
OUT['identity_2d'] = ids

# resolution check on observables
fr96 = [float(np.interp(0.5, s['f'][:, 48][::-1], r96['xc'][::-1])) for s in r96['snaps']]
fr48 = [float(np.interp(0.5, s['f'][:, 24][::-1], r48['xc'][::-1])) for s in r48['snaps']]
OUT['resolution'] = dict(front_midheight_96=fr96, front_midheight_48=fr48,
                         Nu_final_96=float(r96['Nu'][-1]), Nu_final_48=float(r48['Nu'][-1]),
                         fbar_final_96=float(r96['fbar'][-1]), fbar_final_48=float(r48['fbar'][-1]))
print(" resolution:", OUT['resolution'])

# -------------------------------------------------- Cm sensitivity at nc=64
sens = []
for Cm in [1e4, 1e5, 1e6]:
    p = dict(BASE); p['Cm'] = Cm
    r = run(nc=64, out_times=[0.25], **p)
    s = r['snaps'][-1]
    frontmid = float(np.interp(0.5, s['f'][:, 32][::-1], r['xc'][::-1]))
    sens.append(dict(Cm=Cm, fbar=float(r['fbar'][-1]), Nu=float(r['Nu'][-1]),
                     front_mid=frontmid, usolid=float(r['usolid'].max()),
                     uliq=float(r['uliq'].max())))
    print(f" Cm={Cm:.0e}: fbar={sens[-1]['fbar']:.4f} Nu={sens[-1]['Nu']:.3f} front_mid={frontmid:.4f} usolid={sens[-1]['usolid']:.2f}")
OUT['cm_sensitivity'] = sens

# -------------------------------------------------- corrupted-surrogate diagnostics (nc=96, last snap)
s = [sn for sn in r96['snaps'] if 'next' in sn][-1]
rE0, rS0, T1, term = residuals_pair(s, L_eff, BASE['Ste'], BASE['ws'], r96['h'], BASE['theta_c'])
rows = []
for sig in [0.0, 1.0, 2.0, 4.0]:
    s2 = {k: (v.copy() if hasattr(v, 'copy') else v) for k, v in s.items()}
    nx = dict(s['next'])
    th_c = gaussian_filter(nx['th'], sigma=sig) if sig > 0 else nx['th']
    f_c = f_eq(th_c, BASE['ws'])
    nx2 = dict(nx); nx2['th'] = th_c; nx2['Ht'] = th_c + f_c / BASE['Ste']
    s2['next'] = nx2
    rE, rS, T1c, termc = residuals_pair(s2, L_eff, BASE['Ste'], BASE['ws'], r96['h'], BASE['theta_c'])
    ratio = float(np.linalg.norm(rS) / np.linalg.norm(rE / T1c))
    rows.append(dict(sigma_cells=sig,
                     rE_rms_over_term=float(np.linalg.norm(rE) / term),
                     rS_rms_over_term=float(np.linalg.norm(rS) / term),
                     rS_over_rEoverT=ratio,
                     ident_defect=float(np.linalg.norm(rS - rE / T1c) / np.linalg.norm(rE / T1c))))
    print(f" sigma={sig}: |rE|/term={rows[-1]['rE_rms_over_term']:.3e} |rS|/|rE/T|={ratio:.4f} ident={rows[-1]['ident_defect']:.3e}")
OUT['corruption'] = rows

json.dump(OUT, open('/home/claude/pof/run2d.json', 'w'), indent=1)
print("saved run2d.json")
''')

**`v2/solver2d_v2.py`** — Two-dimensional MAC enthalpy–porosity solver (Sec. III B).

In [ ]:
writefile('v2/solver2d_v2.py', r'''"""2D enthalpy-porosity Boussinesq melting in a square cavity (MAC projection).

Nondimensionalization: length H, time H^2/alpha, velocity alpha/H,
theta = (T - T_m)/(T_h - T_m). Model:

  du/dt + u.grad u = -grad p + Pr lap u + Ra Pr theta e_y - A(f) u
  div u = 0
  dHt/dt + div(u Ht) = lap theta,   Ht = theta + f/Ste,  f = 0.5(1+tanh(theta/w_s))
  A(f) = Cm (1-f)^2 / (f^3 + eps_m)

Left wall theta = 1, right wall theta = theta_c, top/bottom adiabatic; no-slip walls;
initial subcooled quiescent solid theta_i. Staggered MAC grid, first-order upwind
advection, explicit diffusion, semi-implicit face drag, Chorin projection with a
DCT-II Neumann Poisson solve. The energy update is in conservative flux form with
Ht as the primary variable, so the discrete global energy balance is exact up to
round-off; Qin is accumulated from the same discrete wall fluxes.
"""
import numpy as np
from scipy.fft import dctn, idctn

DTH = 0.2  # absolute DeltaT for mapping to absolute temperature: T = 1 + DTH*theta


def f_eq(th, ws):
    return 0.5 * (1.0 + np.tanh(th / ws))


def _vanleer(a, b):
    ab = a * b
    return np.where(ab > 0, 2.0 * ab / (a + b + 1e-300), 0.0)


def muscl_faces(q, axis):
    """Second-order MUSCL (van Leer limiter) left/right states on interior faces along axis.
    Cells adjacent to walls use zero slope (first order there)."""
    q = np.moveaxis(q, axis, 0)
    d = q[1:] - q[:-1]                       # differences at interior faces
    slope = np.zeros_like(q)
    slope[1:-1] = _vanleer(d[:-1], d[1:])
    qL = (q + 0.5 * slope)[:-1]              # left state at face i+1/2
    qR = (q - 0.5 * slope)[1:]               # right state
    return np.moveaxis(qL, 0, axis), np.moveaxis(qR, 0, axis)


class NeumannPoisson:
    def __init__(self, nc, h):
        i = np.arange(nc)
        lam = 2.0 * (np.cos(np.pi * i / nc) - 1.0) / h ** 2
        self.den = lam[:, None] + lam[None, :]
        self.den[0, 0] = 1.0

    def solve(self, rhs):
        w = dctn(rhs, type=2, norm='ortho')
        w = w / self.den
        w[0, 0] = 0.0
        return idctn(w, type=2, norm='ortho')


def run(nc=96, Ra=1e5, Pr=1.0, Ste=0.2, ws=0.02, Cm=1e6, eps_m=1e-3,
        theta_i=-0.1, theta_c=-0.1, t_end=0.1, out_times=None, cfl=0.3, adv='upwind',
        init_state=None, wall_limit=None):
    import time as _time
    _t0 = _time.time()
    h = 1.0 / nc
    xc = (np.arange(nc) + 0.5) * h
    u = np.zeros((nc + 1, nc))     # x-faces
    v = np.zeros((nc, nc + 1))     # y-faces
    th = np.full((nc, nc), theta_i)
    f = f_eq(th, ws)
    Ht = th + f / Ste
    pois = NeumannPoisson(nc, h)

    thg = np.linspace(-4.0, 4.0, 32001)
    Hg = thg + f_eq(thg, ws) / Ste

    def theta_of(Htf):
        return np.interp(Htf, Hg, thg)

    def ghosts(th):
        gl = 2.0 * 1.0 - th[0, :]         # left Dirichlet theta=1
        gr = 2.0 * theta_c - th[-1, :]    # right Dirichlet
        return gl, gr

    if out_times is None:
        out_times = list(np.linspace(0, t_end, 7)[1:])
    else:
        out_times = list(out_times)
    snaps = []
    capture_next = False
    Qin = 0.0
    hist = dict(t=[], Nu=[], fbar=[], umax=[], Qin=[], E=[], usolid=[], uliq=[])
    E0 = Ht.sum() * h * h
    dt_dif = cfl * h * h / (4.0 * max(1.0, Pr))
    t = 0.0; step = 0
    if init_state is not None:
        st = init_state
        u, v, th, f, Ht = st['u'], st['v'], st['th'], st['f'], st['Ht']
        t, step, Qin, E0 = st['t'], st['step'], st['Qin'], st['E0']
        snaps, hist, out_times, capture_next = st['snaps'], st['hist'], st['out_times'], st['capture_next']

    while t < t_end - 1e-13:
        if wall_limit is not None and _time.time() - _t0 > wall_limit:
            return dict(partial=True, state=dict(u=u, v=v, th=th, f=f, Ht=Ht, t=t, step=step, Qin=Qin, E0=E0,
                        snaps=snaps, hist=hist, out_times=out_times, capture_next=capture_next))
        umax = max(np.abs(u).max(), np.abs(v).max(), 1e-9)
        dt = min(dt_dif, cfl * h / umax, t_end - t)

        # ---- face liquid fraction and drag
        fu = np.ones((nc + 1, nc)); fu[1:-1] = 0.5 * (f[1:] + f[:-1]); fu[0] = f[0]; fu[-1] = f[-1]
        fv = np.ones((nc, nc + 1)); fv[:, 1:-1] = 0.5 * (f[:, 1:] + f[:, :-1]); fv[:, 0] = f[:, 0]; fv[:, -1] = f[:, -1]
        Au = Cm * (1 - fu) ** 2 / (fu ** 3 + eps_m)
        Av = Cm * (1 - fv) ** 2 / (fv ** 3 + eps_m)

        # ---- momentum predictor (padded fields carry no-slip ghosts)
        upad = np.zeros((nc + 3, nc + 2))
        upad[1:-1, 1:-1] = u
        upad[:, 0] = -upad[:, 1]; upad[:, -1] = -upad[:, -2]    # no-slip at y walls
        lap_u = (upad[2:, 1:-1] - 2 * u + upad[:-2, 1:-1] +
                 upad[1:-1, 2:] - 2 * u + upad[1:-1, :-2]) / h ** 2
        dudx_m = (u - upad[:-2, 1:-1]) / h
        dudx_p = (upad[2:, 1:-1] - u) / h
        dudy_m = (u - upad[1:-1, :-2]) / h
        dudy_p = (upad[1:-1, 2:] - u) / h
        v_at_u = np.zeros_like(u)
        v_at_u[1:-1, :] = 0.25 * (v[1:, :-1] + v[1:, 1:] + v[:-1, :-1] + v[:-1, 1:])
        adv_u = np.where(u > 0, u * dudx_m, u * dudx_p) + \
                np.where(v_at_u > 0, v_at_u * dudy_m, v_at_u * dudy_p)
        u_star = (u + dt * (-adv_u + Pr * lap_u)) / (1.0 + dt * Au)
        u_star[0, :] = 0.0; u_star[-1, :] = 0.0

        vpad = np.zeros((nc + 2, nc + 3))
        vpad[1:-1, 1:-1] = v
        vpad[0, 1:-1] = -vpad[1, 1:-1]; vpad[-1, 1:-1] = -vpad[-2, 1:-1]  # no-slip x walls
        lap_v = (vpad[2:, 1:-1] - 2 * v + vpad[:-2, 1:-1] +
                 vpad[1:-1, 2:] - 2 * v + vpad[1:-1, :-2]) / h ** 2
        dvdx_m = (v - vpad[:-2, 1:-1]) / h
        dvdx_p = (vpad[2:, 1:-1] - v) / h
        dvdy_m = (v - vpad[1:-1, :-2]) / h
        dvdy_p = (vpad[1:-1, 2:] - v) / h
        u_at_v = np.zeros_like(v)
        u_at_v[:, 1:-1] = 0.25 * (u[:-1, 1:] + u[1:, 1:] + u[:-1, :-1] + u[1:, :-1])
        th_at_v = np.zeros_like(v)
        th_at_v[:, 1:-1] = 0.5 * (th[:, 1:] + th[:, :-1])
        adv_v = np.where(u_at_v > 0, u_at_v * dvdx_m, u_at_v * dvdx_p) + \
                np.where(v > 0, v * dvdy_m, v * dvdy_p)
        v_star = (v + dt * (-adv_v + Pr * lap_v + Ra * Pr * th_at_v)) / (1.0 + dt * Av)
        v_star[:, 0] = 0.0; v_star[:, -1] = 0.0

        # ---- projection
        div = (u_star[1:, :] - u_star[:-1, :]) / h + (v_star[:, 1:] - v_star[:, :-1]) / h
        phi = pois.solve(div / dt)
        u = u_star; v = v_star
        u[1:-1, :] = u[1:-1, :] - dt * (phi[1:, :] - phi[:-1, :]) / h
        v[:, 1:-1] = v[:, 1:-1] - dt * (phi[:, 1:] - phi[:, :-1]) / h

        # ---- energy (conservative flux form, Ht primary)
        gl, gr = ghosts(th)
        qx = np.zeros((nc + 1, nc))
        qx[1:-1] = -(th[1:] - th[:-1]) / h
        qx[0] = -(th[0] - gl) / h
        qx[-1] = -(gr - th[-1]) / h
        qy = np.zeros((nc, nc + 1))
        qy[:, 1:-1] = -(th[:, 1:] - th[:, :-1]) / h
        Fx = np.zeros((nc + 1, nc))
        Fy = np.zeros((nc, nc + 1))
        if adv == 'muscl':
            HL, HR = muscl_faces(Ht, 0)      # face states on interior x-faces
            Fx[1:-1] = np.where(u[1:-1] > 0, HL, HR) * u[1:-1]
            HL, HR = muscl_faces(Ht, 1)
            Fy[:, 1:-1] = np.where(v[:, 1:-1] > 0, HL, HR) * v[:, 1:-1]
        else:
            Fx[1:-1] = np.where(u[1:-1] > 0, Ht[:-1], Ht[1:]) * u[1:-1]
            Fy[:, 1:-1] = np.where(v[:, 1:-1] > 0, Ht[:, :-1], Ht[:, 1:]) * v[:, 1:-1]
        Gx = Fx + qx; Gy = Fy + qy
        Ht = Ht - dt * ((Gx[1:, :] - Gx[:-1, :]) / h + (Gy[:, 1:] - Gy[:, :-1]) / h)
        Qin += dt * (qx[0].sum() * h - qx[-1].sum() * h)
        th = theta_of(Ht)
        f = f_eq(th, ws)

        t += dt; step += 1
        if step % 100 == 0 or t >= t_end - 1e-13:
            glh, _ = ghosts(th)
            Nu = np.sum(-(th[0, :] - glh) / h) * h
            liq = f > 0.95; sol = f < 0.05
            uc = 0.5 * (u[1:, :] + u[:-1, :]); vc = 0.5 * (v[:, 1:] + v[:, :-1])
            spd = np.hypot(uc, vc)
            hist['t'].append(t); hist['Nu'].append(Nu)
            hist['fbar'].append(f.mean()); hist['umax'].append(umax)
            hist['Qin'].append(Qin); hist['E'].append(Ht.sum() * h * h)
            hist['usolid'].append(spd[sol].max() if sol.any() else 0.0)
            hist['uliq'].append(spd[liq].max() if liq.any() else 0.0)
        if capture_next and snaps:
            snaps[-1]['next'] = dict(t=t, th=th.copy(), f=f.copy(), Ht=Ht.copy(),
                                     uf=u.copy(), vf=v.copy(), dt=dt)
            capture_next = False
        if out_times and t >= out_times[0] - 1e-13:
            uc = 0.5 * (u[1:, :] + u[:-1, :]); vc = 0.5 * (v[:, 1:] + v[:, :-1])
            dudx = (u[1:, :] - u[:-1, :]) / h; dvdy = (v[:, 1:] - v[:, :-1]) / h
            dudy = np.gradient(uc, h, axis=1); dvdx = np.gradient(vc, h, axis=0)
            Phi = 2 * dudx ** 2 + 2 * dvdy ** 2 + (dudy + dvdx) ** 2      # 2 D:D (nondim)
            Ac = Cm * (1 - f) ** 2 / (f ** 3 + eps_m)
            Tabs = 1.0 + DTH * th
            gx = np.gradient(th, h, axis=0); gy = np.gradient(th, h, axis=1)
            glh, _ = ghosts(th)
            snaps.append(dict(t=t, th=th.copy(), f=f.copy(), u=uc, v=vc, Ht=Ht.copy(),
                              uf=u.copy(), vf=v.copy(),
                              visc_int=float(Pr * Phi.sum() * h * h),
                              darcy_int=float((Ac * (uc ** 2 + vc ** 2)).sum() * h * h),
                              visc_ent_int=float(Pr * (Phi / Tabs).sum() * h * h),
                              darcy_ent_int=float((Ac * (uc ** 2 + vc ** 2) / Tabs).sum() * h * h),
                              cond_ent_int=float(((gx ** 2 + gy ** 2) / Tabs ** 2).sum() * h * h),
                              Nu=float(np.sum(-(th[0, :] - glh) / h) * h)))
            out_times.pop(0)
            capture_next = True

    hist = {k: np.array(vv) for k, vv in hist.items()}
    return dict(xc=xc, h=h, snaps=snaps, E0=E0, steps=step, params=dict(
        nc=nc, Ra=Ra, Pr=Pr, Ste=Ste, ws=ws, Cm=Cm, eps_m=eps_m,
        theta_i=theta_i, theta_c=theta_c, t_end=t_end), **hist)
''')

**`v2/run2d_v2_lib.py`** — Summary helpers for the two-dimensional runs.

In [ ]:
writefile('v2/run2d_v2_lib.py', r'''"""v2 two-dimensional audit: three grids, time-step refinement, MUSCL energy advection,
separate sensitivities to w_s and eps_m, dissipation integrals, two-state corruption."""
import sys, json, pickle, time
import numpy as np
sys.path.insert(0, '/home/claude/pof/v2'); sys.path.insert(0, '/home/claude/pof')
from solver2d_v2 import run, f_eq, DTH
from scipy.ndimage import gaussian_filter

BASE = dict(Ra=1e5, Pr=1.0, Ste=0.2, ws=0.02, Cm=1e6, eps_m=1e-3,
            theta_i=-0.1, theta_c=-0.1, t_end=0.25)
SNAP_T = [0.05, 0.10, 0.15, 0.20, 0.25]
OUT = {}

def front_mid(s, xc):
    j = len(xc) // 2
    prof = 0.5 * (s['f'][:, j - 1] + s['f'][:, j]) if len(xc) % 2 == 0 else s['f'][:, j]
    return float(np.interp(0.5, prof[::-1], xc[::-1]))

def front_top_bottom(s, xc):
    n = len(xc); jt = int(0.95 * n); jb = int(0.05 * n)
    return (float(np.interp(0.5, s['f'][:, jt][::-1], xc[::-1])),
            float(np.interp(0.5, s['f'][:, jb][::-1], xc[::-1])))

def summarize(r, label):
    xc = r['xc']
    d = (r['E'] - r['E0']) - r['Qin']
    rec = dict(label=label, nc=r['params']['nc'], steps=r['steps'],
               energy_defect_rel=float(np.abs(d).max() / abs(r['Qin'][-1])),
               t=[s['t'] for s in r['snaps']],
               front_mid=[front_mid(s, xc) for s in r['snaps']],
               front_top=[front_top_bottom(s, xc)[0] for s in r['snaps']],
               front_bot=[front_top_bottom(s, xc)[1] for s in r['snaps']],
               fbar=[float(s['f'].mean()) for s in r['snaps']],
               Nu=[s['Nu'] for s in r['snaps']],
               umax_liq=[float(np.hypot(s['u'], s['v'])[s['f'] > 0.95].max()) for s in r['snaps']],
               usolid=float(r['usolid'].max()), uliq=float(r['uliq'].max()),
               visc_int=[s['visc_int'] for s in r['snaps']],
               darcy_int=[s['darcy_int'] for s in r['snaps']],
               visc_ent_int=[s['visc_ent_int'] for s in r['snaps']],
               darcy_ent_int=[s['darcy_ent_int'] for s in r['snaps']],
               cond_ent_int=[s['cond_ent_int'] for s in r['snaps']])
    print(label, 'steps', r['steps'], 'fbar_end %.4f' % rec['fbar'][-1], 'Nu_end %.3f' % rec['Nu'][-1],
          'front_mid', ['%.3f' % z for z in rec['front_mid']], flush=True)
    return rec

''')

**`v2/run2d_v2.py`** — Two-dimensional grid, time-step, advection, and parameter study (Table V, Fig. 6).

In [ ]:
writefile('v2/run2d_v2.py', r'''"""v2 two-dimensional audit: three grids, time-step refinement, MUSCL energy advection,
separate sensitivities to w_s and eps_m, dissipation integrals, two-state corruption."""
import sys, json, pickle, time
import numpy as np
sys.path.insert(0, '/home/claude/pof/v2'); sys.path.insert(0, '/home/claude/pof')
from solver2d_v2 import run, f_eq, DTH
from scipy.ndimage import gaussian_filter

BASE = dict(Ra=1e5, Pr=1.0, Ste=0.2, ws=0.02, Cm=1e6, eps_m=1e-3,
            theta_i=-0.1, theta_c=-0.1, t_end=0.25)
SNAP_T = [0.05, 0.10, 0.15, 0.20, 0.25]
OUT = {}

def front_mid(s, xc):
    j = len(xc) // 2
    prof = 0.5 * (s['f'][:, j - 1] + s['f'][:, j]) if len(xc) % 2 == 0 else s['f'][:, j]
    return float(np.interp(0.5, prof[::-1], xc[::-1]))

def front_top_bottom(s, xc):
    n = len(xc); jt = int(0.95 * n); jb = int(0.05 * n)
    return (float(np.interp(0.5, s['f'][:, jt][::-1], xc[::-1])),
            float(np.interp(0.5, s['f'][:, jb][::-1], xc[::-1])))

def summarize(r, label):
    xc = r['xc']
    d = (r['E'] - r['E0']) - r['Qin']
    rec = dict(label=label, nc=r['params']['nc'], steps=r['steps'],
               energy_defect_rel=float(np.abs(d).max() / abs(r['Qin'][-1])),
               t=[s['t'] for s in r['snaps']],
               front_mid=[front_mid(s, xc) for s in r['snaps']],
               front_top=[front_top_bottom(s, xc)[0] for s in r['snaps']],
               front_bot=[front_top_bottom(s, xc)[1] for s in r['snaps']],
               fbar=[float(s['f'].mean()) for s in r['snaps']],
               Nu=[s['Nu'] for s in r['snaps']],
               umax_liq=[float(np.hypot(s['u'], s['v'])[s['f'] > 0.95].max()) for s in r['snaps']],
               usolid=float(r['usolid'].max()), uliq=float(r['uliq'].max()),
               visc_int=[s['visc_int'] for s in r['snaps']],
               darcy_int=[s['darcy_int'] for s in r['snaps']],
               visc_ent_int=[s['visc_ent_int'] for s in r['snaps']],
               darcy_ent_int=[s['darcy_ent_int'] for s in r['snaps']],
               cond_ent_int=[s['cond_ent_int'] for s in r['snaps']])
    print(label, 'steps', r['steps'], 'fbar_end %.4f' % rec['fbar'][-1], 'Nu_end %.3f' % rec['Nu'][-1],
          'front_mid', ['%.3f' % z for z in rec['front_mid']], flush=True)
    return rec

runs = {}
t0 = time.time()
for nc in [48, 96]:
    r = run(nc=nc, out_times=SNAP_T, **BASE)
    runs[f'up{nc}'] = summarize(r, f'upwind {nc}')
    if nc == 96:
        pickle.dump(r['snaps'], open('/home/claude/pof/v2/snaps96.pkl', 'wb'))
r = run(nc=96, out_times=SNAP_T, cfl=0.15, **BASE); runs['up96_cfl015'] = summarize(r, 'upwind 96 cfl0.15')
for nc in [48, 96]:
    r = run(nc=nc, out_times=SNAP_T, adv='muscl', **BASE); runs[f'mu{nc}'] = summarize(r, f'muscl {nc}')
# sensitivity (64^2): w_s and eps_m separately; C_m from v1
for ws in [0.01, 0.02, 0.04]:
    p = dict(BASE); p['ws'] = ws
    r = run(nc=64, out_times=SNAP_T, **p); runs[f'ws{ws}'] = summarize(r, f'64 ws={ws}')
for em in [1e-4, 1e-2]:
    p = dict(BASE); p['eps_m'] = em
    r = run(nc=64, out_times=SNAP_T, **p); runs[f'eps{em}'] = summarize(r, f'64 eps_m={em}')
OUT['runs'] = runs
json.dump(OUT, open('/home/claude/pof/v2/run2d_v2.json', 'w'), indent=1)
print('elapsed', time.time() - t0, flush=True)
# finest grids last (long)
r = run(nc=192, out_times=SNAP_T, **BASE); runs['up192'] = summarize(r, 'upwind 192')
OUT['runs'] = runs
json.dump(OUT, open('/home/claude/pof/v2/run2d_v2.json', 'w'), indent=1)
r = run(nc=192, out_times=SNAP_T, adv='muscl', **BASE); runs['mu192'] = summarize(r, 'muscl 192')
OUT['runs'] = runs
json.dump(OUT, open('/home/claude/pof/v2/run2d_v2.json', 'w'), indent=1)
print('ALL_2D_DONE elapsed', time.time() - t0, flush=True)
''')

**`v2/run2d_192_chunk.py`** — Restartable $192^2$ run.

In [ ]:
writefile('v2/run2d_192_chunk.py', r'''import sys, json, pickle, os
sys.path.insert(0, '/home/claude/pof/v2')
from solver2d_v2 import run
from run2d_v2_lib import summarize, BASE, SNAP_T
ck = '/home/claude/pof/v2/state192.pkl'
st = pickle.load(open(ck, 'rb')) if os.path.exists(ck) else None
r = run(nc=192, out_times=list(SNAP_T), init_state=st, wall_limit=float(sys.argv[1]), **BASE)
if r.get('partial'):
    pickle.dump(r['state'], open(ck, 'wb')); print('PARTIAL t=%.4f step=%d' % (r['state']['t'], r['state']['step']))
else:
    rec = summarize(r, 'upwind 192'); json.dump(rec, open('/home/claude/pof/v2/run2d_192.json', 'w'), indent=1); print('DONE192')
''')

**`v2/resid2d.py`** — Two-dimensional residual pair on solver steps.

In [ ]:
writefile('v2/resid2d.py', r'''import numpy as np
from solver2d_v2 import DTH

def residuals_pair(s, L_eff, Ste, ws, h, theta_c):
    """r_E and r_S over one solver step, absolute variables T=1+DTH*theta.
    Flux-form material derivative matching the solver for r_E; r_S discretized
    independently (upwind entropy advection, arithmetic face T in q/T)."""
    th0, Ht0 = s['th'], s['Ht']
    nx = s['next']
    th1, Ht1, dt = nx['th'], nx['Ht'], nx['dt']
    uf, vf = nx['uf'], nx['vf']
    nc = th0.shape[0]
    T0 = 1.0 + DTH * th0; T1 = 1.0 + DTH * th1
    H0 = DTH * Ht0; H1 = DTH * Ht1                     # absolute enthalpy (c=1, h = DTH*Ht)
    # faces of T1 with Dirichlet ghosts
    gl = 2 * (1 + DTH * 1.0) - T1[0, :]; gr = 2 * (1 + DTH * theta_c) - T1[-1, :]
    qx = np.zeros((nc + 1, nc)); qx[1:-1] = -(T1[1:] - T1[:-1]) / h
    qx[0] = -(T1[0] - gl) / h; qx[-1] = -(gr - T1[-1]) / h
    qy = np.zeros((nc, nc + 1)); qy[:, 1:-1] = -(T1[:, 1:] - T1[:, :-1]) / h
    Fx = np.zeros((nc + 1, nc)); Fx[1:-1] = np.where(uf[1:-1] > 0, H1[:-1], H1[1:]) * uf[1:-1]
    Fy = np.zeros((nc, nc + 1)); Fy[:, 1:-1] = np.where(vf[:, 1:-1] > 0, H1[:, :-1], H1[:, 1:]) * vf[:, 1:-1]
    divE = ((Fx + qx)[1:, :] - (Fx + qx)[:-1, :]) / h + ((Fy + qy)[:, 1:] - (Fy + qy)[:, :-1]) / h
    r_E = (H1 - H0) / dt + divE

    # independent entropy discretization: s = ln T + L_eff * g(T), g = int f'(T)/T dT
    Tg = np.linspace(0.5, 1.6, 400001)
    fg = 0.5 / (ws * DTH) / np.cosh((Tg - 1.0) / (ws * DTH)) ** 2   # df/dT in absolute T
    integ = fg / Tg
    G = np.concatenate([[0.0], np.cumsum(0.5 * (integ[1:] + integ[:-1]) * np.diff(Tg))])
    G -= np.interp(1.0, Tg, G)
    def s_of(T): return np.log(T) + L_eff * np.interp(T, Tg, G)
    s0, s1 = s_of(T0), s_of(T1)
    Tfx = np.zeros((nc + 1, nc)); Tfx[1:-1] = 0.5 * (T1[1:] + T1[:-1])
    Tfx[0] = 1 + DTH * 1.0; Tfx[-1] = 1 + DTH * theta_c
    Tfy = np.ones((nc, nc + 1)); Tfy[:, 1:-1] = 0.5 * (T1[:, 1:] + T1[:, :-1])
    Tfy[:, 0] = T1[:, 0]; Tfy[:, -1] = T1[:, -1]
    Sx = np.zeros((nc + 1, nc)); Sx[1:-1] = np.where(uf[1:-1] > 0, s1[:-1], s1[1:]) * uf[1:-1]
    Sy = np.zeros((nc, nc + 1)); Sy[:, 1:-1] = np.where(vf[:, 1:-1] > 0, s1[:, :-1], s1[:, 1:]) * vf[:, 1:-1]
    GxS = Sx + qx / Tfx; GyS = Sy + qy / Tfy
    divS = (GxS[1:, :] - GxS[:-1, :]) / h + (GyS[:, 1:] - GyS[:, :-1]) / h
    gx2 = 0.5 * ((qx[:-1, :] ** 2) + (qx[1:, :] ** 2))
    gy2 = 0.5 * ((qy[:, :-1] ** 2) + (qy[:, 1:] ** 2))
    sigma = (gx2 + gy2) / T1 ** 2
    r_S = (s1 - s0) / dt + divS - sigma
    term = np.linalg.norm((s1 - s0) / dt)
    return r_E, r_S, T1, term

''')

**`v2/datagen2.py`** — Internal-variable datasets at half macro-step and other mobilities.

In [ ]:
writefile('v2/datagen2.py', r'''"""v2 datasets: kinetic closure at half macro-step and at M0 in {100, 1600}.
Reproduces the original train/val/test split exactly (same RNG call sequence)."""
import sys, numpy as np
sys.path.insert(0, '/home/claude/pof')
from solvers1d import solve_kinetic

rng = np.random.default_rng(0)
# replay the original RNG sequence: equilibrium shuffle first
Th_vals = np.linspace(1.10, 1.40, 9); L_vals = np.linspace(1.0, 3.0, 10)
eq_combos = [(th, l) for th in Th_vals for l in L_vals]; rng.shuffle(eq_combos)
Th_vals = np.linspace(1.15, 1.40, 8); L_vals = np.linspace(1.5, 2.5, 9)
combos = [(th, l) for th in Th_vals for l in L_vals]; rng.shuffle(combos)
split = dict(train=combos[:48], val=combos[48:60], test=combos[60:])
N = 128; T_i = 0.95; t_end = 0.1

def gen(M0, nmacro, store_every, out):
    data = {}
    for k, cl in split.items():
        Ts, fs, ths, Ls, ts = [], [], [], [], []
        for (th, l) in cl:
            sol = solve_kinetic(N=N, T_h=th, L=l, M0=M0, Meps=0.0, t_end=t_end,
                                T_init=T_i, f_init=0.01, store_every=store_every)
            sel = np.linspace(0, len(sol['t']) - 1, nmacro - 1).round().astype(int)
            Ts.append(np.vstack([np.full((1, N), T_i), sol['T'][sel]]))
            fs.append(np.vstack([np.full((1, N), 0.01), sol['f'][sel]]))
            ths.append(th); Ls.append(l)
            ts.append(np.concatenate([[0.0], sol['t'][sel]]))
        data[f'{k}_T'] = np.stack(Ts); data[f'{k}_f'] = np.stack(fs)
        data[f'{k}_th'] = np.array(ths); data[f'{k}_L'] = np.array(Ls); data[f'{k}_t'] = np.stack(ts)
    np.savez_compressed(out, **data, M0=M0, T_i=T_i, N=N)
    print(out, data['train_T'].shape, 'mean macro dt', np.diff(data['train_t'][0])[1:].mean())

if __name__ == '__main__':
    gen(400.0, 49, 25, '/home/claude/pof/v2/data_kin_half.npz')
    gen(100.0, 25, 50, '/home/claude/pof/v2/data_kin_M100.npz')
    gen(1600.0, 25, 50, '/home/claude/pof/v2/data_kin_M1600.npz')
    # sanity: regenerate baseline and compare with original file
    gen(400.0, 25, 50, '/home/claude/pof/v2/data_kin_check.npz')
    a = np.load('/home/claude/pof/data_kin.npz'); b = np.load('/home/claude/pof/v2/data_kin_check.npz')
    print('split reproduced:', np.allclose(a['test_th'], b['test_th']) and np.allclose(a['test_L'], b['test_L']),
          ' max |dT|:', np.abs(a['test_T'] - b['test_T']).max())
''')

**`v2/datagen2d.py`** — Two-dimensional convective training dataset.

In [ ]:
writefile('v2/datagen2d.py', r'''"""2D convection-driven melting dataset for operator learning (equilibrium closure)."""
import sys, numpy as np, time
sys.path.insert(0, '/home/claude/pof/v2')
from solver2d_v2 import run

NC = 32
NMAC = 21
T_END = 0.25
times = list(np.linspace(0, T_END, NMAC)[1:])
Ra_vals = [5e4, 8e4, 1.2e5, 1.6e5, 2e5]
Ste_vals = [0.1, 0.15, 0.2, 0.25, 0.3, 0.35]
combos = [(ra, st) for ra in Ra_vals for st in Ste_vals]
rng = np.random.default_rng(1)
rng.shuffle(combos)
split = dict(train=combos[:18], val=combos[18:24], test=combos[24:])
data = {}
t0 = time.time()
for k, cl in split.items():
    TH, U, V, RA, ST, TT = [], [], [], [], [], []
    for (ra, st) in cl:
        r = run(nc=NC, Ra=ra, Pr=1.0, Ste=st, ws=0.02, Cm=1e6, eps_m=1e-3,
                theta_i=-0.1, theta_c=-0.1, t_end=T_END, out_times=times)
        th0 = np.full((NC, NC), -0.1)
        TH.append(np.stack([th0] + [s['th'] for s in r['snaps']]))
        U.append(np.stack([np.zeros((NC, NC))] + [s['u'] for s in r['snaps']]))
        V.append(np.stack([np.zeros((NC, NC))] + [s['v'] for s in r['snaps']]))
        RA.append(ra); ST.append(st)
        TT.append(np.array([0.0] + [s['t'] for s in r['snaps']]))
    data[f'{k}_th'] = np.stack(TH); data[f'{k}_u'] = np.stack(U); data[f'{k}_v'] = np.stack(V)
    data[f'{k}_Ra'] = np.array(RA); data[f'{k}_Ste'] = np.array(ST); data[f'{k}_t'] = np.stack(TT)
    print(k, data[f'{k}_th'].shape, 'elapsed', time.time() - t0, flush=True)
np.savez_compressed('/home/claude/pof/v2/data_2d.npz', **data, NC=NC, ws=0.02, theta_c=-0.1)
print('saved; split', {k: v for k, v in split.items()})
''')

**`v2/train2.py`** — One-dimensional operator training and evaluation (Sec. V).

In [ ]:
writefile('v2/train2.py', r'''"""Matched-budget operator ablation in JAX.

Variants (equilibrium closure, data_eq.npz): D, E, ES, EET
Variants (kinetic closure,     data_kin.npz): D, E, EPH, EK
Same backbone, data, optimizer, schedule, seeds across variants.
"""
import argparse, json, time
import numpy as np
import jax, jax.numpy as jnp, optax

T_M, T_REF = 1.0, 1.0

# ------------------------------------------------------------------ closures (JAX)
def f_eq(T, w):  return 0.5 * (1.0 + jnp.tanh((T - T_M) / w))
def df_eq(T, w):
    c = jnp.cosh((T - T_M) / w); return 0.5 / (w * c * c)
def h_eq(T, L, w):  return (T - T_REF) + L * (f_eq(T, w) - 0.5)

def make_sf_table(w, Tmin=0.6, Tmax=1.8, n=800001):
    Tg = np.linspace(Tmin, Tmax, n)
    integ = (0.5 / (w * np.cosh((Tg - T_M) / w) ** 2)) / Tg
    S = np.concatenate([[0.0], np.cumsum(0.5 * (integ[1:] + integ[:-1]) * np.diff(Tg))])
    S -= np.interp(T_REF, Tg, S)
    return jnp.array(Tg), jnp.array(S)

def s_eq(T, L, tab):
    Tg, Sf = tab
    return jnp.log(T) + L * jnp.interp(T, Tg, Sf)

# ------------------------------------------------------------------ discrete residuals (JAX, batched)
def face_grad(T, th, dx):
    g_in = (T[:, 1:] - T[:, :-1]) / dx
    g0 = ((T[:, :1] - th[:, None]) / (0.5 * dx))
    gN = jnp.zeros_like(g0)
    return jnp.concatenate([g0, g_in, gN], axis=1)

RESID = 'end'   # 'end' (implicit-Euler endpoint) or 'trap' (trapezoidal time quadrature)

def _eq_spatial(T, th, dx):
    g = face_grad(T, th, dx)
    divq = -(g[:, 1:] - g[:, :-1]) / dx
    Tf = jnp.concatenate([th[:, None], 0.5 * (T[:, 1:] + T[:, :-1]), T[:, -1:]], axis=1)
    qf = -g
    entdiv = ((qf / Tf)[:, 1:] - (qf / Tf)[:, :-1]) / dx
    sigma = 0.5 * (g[:, :-1] ** 2 + g[:, 1:] ** 2) / T ** 2
    return divq, entdiv, sigma

def resid_eq(T0, T1, dt, th, L, dx, w, tab):
    d1, e1, s1 = _eq_spatial(T1, th, dx)
    if RESID == 'trap':
        d0, e0, s0 = _eq_spatial(T0, th, dx)
        d1, e1, s1 = 0.5 * (d0 + d1), 0.5 * (e0 + e1), 0.5 * (s0 + s1)
    rE = (h_eq(T1, L[:, None], w) - h_eq(T0, L[:, None], w)) / dt[:, None] + d1
    rS = (s_eq(T1, L[:, None], tab) - s_eq(T0, L[:, None], tab)) / dt[:, None] + e1 - s1
    return rE, rS

def X_drive(T, L):  return L * (1.0 / T_M - 1.0 / T)

def resid_kin(T0, f0, T1, f1, dt, th, L, dx, M0):
    g1 = face_grad(T1, th, dx)
    divq = -(g1[:, 1:] - g1[:, :-1]) / dx
    rate = M0 * f1 * (1.0 - f1) * X_drive(T1, L[:, None])
    Xb = X_drive(T1, L[:, None])
    if RESID == 'trap':
        g0 = face_grad(T0, th, dx)
        divq = 0.5 * (divq - (g0[:, 1:] - g0[:, :-1]) / dx)
        rate = 0.5 * (rate + M0 * f0 * (1.0 - f0) * X_drive(T0, L[:, None]))
        Xb = 0.5 * (Xb + X_drive(T0, L[:, None]))
    dfdt = (f1 - f0) / dt[:, None]
    rE = (T1 - T0) / dt[:, None] + L[:, None] * dfdt + divq
    rf = dfdt - rate
    sph = Xb * dfdt
    return rE, rf, sph

# ------------------------------------------------------------------ model
def init_params(key, cin, cout, width=64, k=5, nlayers=4):
    keys = jax.random.split(key, nlayers)
    sizes = [cin] + [width] * (nlayers - 1) + [cout]
    params = []
    for i in range(nlayers):
        fan = sizes[i] * k
        Wt = jax.random.normal(keys[i], (sizes[i + 1], sizes[i], k)) * jnp.sqrt(2.0 / fan)
        b = jnp.zeros((sizes[i + 1],))
        params.append((Wt, b))
    return params

def conv1d(x, Wt, b):
    y = jax.lax.conv_general_dilated(x, Wt, (1,), 'SAME',
                                     dimension_numbers=('NCH', 'OIH', 'NCH'))
    return y + b[None, :, None]

def forward(params, x, out_scale):
    h = x
    for (Wt, b) in params[:-1]:
        h = jax.nn.gelu(conv1d(h, Wt, b))
    y = conv1d(h, *params[-1])
    return y * out_scale

# ------------------------------------------------------------------ data plumbing
def make_pairs(T, th, L, t, f=None):
    nT, nM, N = T.shape
    T0 = T[:, :-1].reshape(-1, N); T1 = T[:, 1:].reshape(-1, N)
    dt = np.diff(t, axis=1).reshape(-1)
    TH = np.repeat(th, nM - 1); LL = np.repeat(L, nM - 1)
    out = [T0, T1, dt, TH, LL]
    if f is not None:
        out += [f[:, :-1].reshape(-1, N), f[:, 1:].reshape(-1, N)]
    return [jnp.array(a) for a in out]

def build_inputs(T0, th, L, dt, x, f0=None):
    B, N = T0.shape
    chans = [T0]
    if f0 is not None: chans.append(f0)
    chans += [jnp.broadcast_to(th[:, None], (B, N)),
              jnp.broadcast_to(L[:, None], (B, N)),
              jnp.broadcast_to(dt[:, None] / 0.004, (B, N)),
              jnp.broadcast_to(x[None, :], (B, N))]
    return jnp.stack(chans, axis=1)

# ------------------------------------------------------------------ main
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--closure', choices=['eq', 'kin'], required=True)
    ap.add_argument('--variant', required=True)   # D,E,ES,EET / D,E,EPH,EK
    ap.add_argument('--seed', type=int, default=0)
    ap.add_argument('--lam', type=float, default=3e-3)
    ap.add_argument('--steps', type=int, default=3000)
    ap.add_argument('--ntrain', type=int, default=0)
    ap.add_argument('--out', default=None)
    ap.add_argument('--data', default=None)
    ap.add_argument('--resid', choices=['end', 'trap'], default='end')
    ap.add_argument('--tag', default='')
    args = ap.parse_args()
    global RESID
    RESID = args.resid

    d = np.load(args.data or f'/home/claude/pof/data_{args.closure}.npz')
    N = int(d['N']); dx = 1.0 / N
    x = jnp.array((np.arange(N) + 0.5) * dx)
    kin = args.closure == 'kin'
    w = float(d['w']) if not kin else None
    M0 = float(d['M0']) if kin else None
    tab = make_sf_table(w) if not kin else None

    nt = args.ntrain if args.ntrain > 0 else d['train_T'].shape[0]
    if kin:
        tr = make_pairs(d['train_T'][:nt], d['train_th'][:nt], d['train_L'][:nt], d['train_t'][:nt], d['train_f'][:nt])
        va = make_pairs(d['val_T'], d['val_th'], d['val_L'], d['val_t'], d['val_f'])
    else:
        tr = make_pairs(d['train_T'][:nt], d['train_th'][:nt], d['train_L'][:nt], d['train_t'][:nt])
        va = make_pairs(d['val_T'], d['val_th'], d['val_L'], d['val_t'])

    # normalization scales from training data
    dT_scale = float(jnp.std(tr[1] - tr[0]) + 1e-8)
    if kin:
        df_scale = float(jnp.std(tr[6] - tr[5]) + 1e-8)
    # reference residual floors (implicit-Euler quotient on true pairs)
    if kin:
        rE0, rf0, sph0 = resid_kin(tr[0], tr[5], tr[1], tr[6], tr[2], tr[3], tr[4], dx, M0)
        E0 = float(jnp.sqrt(jnp.mean(rE0 ** 2))); F0 = float(jnp.sqrt(jnp.mean(rf0 ** 2)))
        SIG0 = float(jnp.sqrt(jnp.mean(sph0 ** 2)) + 1e-8)   # two-sided scale: RMS of |sigma_ph| on reference pairs
    else:
        rE0, rS0 = resid_eq(tr[0], tr[1], tr[2], tr[3], tr[4], dx, w, tab)
        E0 = float(jnp.sqrt(jnp.mean(rE0 ** 2)))
        S0 = float(jnp.sqrt(jnp.mean((rE0 / tr[1]) ** 2)))

    cin = 6 if kin else 5
    cout = 2 if kin else 1
    key = jax.random.PRNGKey(args.seed)
    params = init_params(key, cin, cout)
    sched = optax.cosine_decay_schedule(3e-3, args.steps, 1e-2)
    opt = optax.adam(sched)
    ostate = opt.init(params)

    lam = args.lam
    variant = args.variant

    def predict(params, T0, th, L, dt, f0=None):
        xin = build_inputs(T0, th, L, dt, x, f0)
        y = forward(params, xin, out_scale=1.0)
        T1 = T0 + y[:, 0] * dT_scale
        if kin:
            f1 = f0 + y[:, 1] * df_scale
            return T1, f1
        return T1

    def loss_fn(params, batch, return_terms=False):
        if kin:
            T0, T1t, dt, th, L, f0, f1t = batch
            T1, f1 = predict(params, T0, th, L, dt, f0)
            Ld = jnp.mean(((T1 - T1t) / dT_scale) ** 2) + jnp.mean(((f1 - f1t) / df_scale) ** 2)
            rE, rf, sph = resid_kin(T0, f0, T1, f1, dt, th, L, dx, M0)
            terms = dict(E=jnp.mean((rE / E0) ** 2), K=jnp.mean((rf / F0) ** 2),
                         PH=jnp.mean((jnp.maximum(0., -sph) / SIG0) ** 2))
        else:
            T0, T1t, dt, th, L = batch
            T1 = predict(params, T0, th, L, dt)
            Ld = jnp.mean(((T1 - T1t) / dT_scale) ** 2)
            rE, rS = resid_eq(T0, T1, dt, th, L, dx, w, tab)
            terms = dict(E=jnp.mean((rE / E0) ** 2), S=jnp.mean((rS / S0) ** 2),
                         EET=jnp.mean(((rE / T1) / S0) ** 2))
        if return_terms:
            return Ld, terms
        L_tot = Ld
        if variant in ('E', 'ES', 'EET', 'EPH', 'EK'):
            L_tot = L_tot + lam * terms['E']
        if variant == 'ES':   L_tot = L_tot + lam * terms['S']
        if variant == 'EET':  L_tot = L_tot + lam * terms['EET']
        if variant == 'EPH':  L_tot = L_tot + lam * terms['PH']
        if variant == 'EK':   L_tot = L_tot + lam * terms['K']
        return L_tot, Ld

    @jax.jit
    def step(params, ostate, batch):
        (Lt, Ld), g = jax.value_and_grad(loss_fn, has_aux=True)(params, batch)
        upd, ostate = opt.update(g, ostate)
        return optax.apply_updates(params, upd), ostate, Lt, Ld

    @jax.jit
    def val_loss(params):
        _, Ld = loss_fn(params, tuple(va))
        return Ld

    # ---- loss-component diagnostics: value and gradient norm of each (lam-weighted) term
    diag_batch = tuple(a[:min(256, a.shape[0])] for a in tr)
    term_names = ['K', 'PH'] if kin else ['S', 'EET']
    term_names = ['E'] + term_names
    def _gnorm(g):
        return jnp.sqrt(sum(jnp.sum(x ** 2) for x in jax.tree_util.tree_leaves(g)))
    def component_diag(p):
        out = {}
        Ld, terms = loss_fn(p, diag_batch, return_terms=True)
        out['data'] = dict(value=float(Ld), gradnorm=float(_gnorm(jax.grad(
            lambda q: loss_fn(q, diag_batch, return_terms=True)[0])(p))))
        for nm in term_names:
            gv = jax.grad(lambda q, nm=nm: lam * loss_fn(q, diag_batch, return_terms=True)[1][nm])(p)
            out[nm] = dict(value=float(terms[nm]), weighted=float(lam * terms[nm]), gradnorm=float(_gnorm(gv)))
        return out
    diag_init = component_diag(params)

    nb = tr[0].shape[0]
    bs = min(256, nb)
    rngnp = np.random.default_rng(args.seed)
    best = (1e18, params)
    t0 = time.time()
    for it in range(args.steps):
        idx = rngnp.choice(nb, size=bs, replace=False)
        batch = tuple(a[idx] for a in tr)
        params, ostate, Lt, Ld = step(params, ostate, batch)
        if (it + 1) % 200 == 0:
            vl = float(val_loss(params))
            if vl < best[0]:
                best = (vl, jax.tree_util.tree_map(lambda a: a, params))
    params = best[1]
    train_time = time.time() - t0
    diag_final = component_diag(params)
    import os
    os.makedirs('/home/claude/pof/v2/ckpt', exist_ok=True)
    flat = {f'W{i}': np.array(W) for i, (W, b) in enumerate(params)}
    flat.update({f'b{i}': np.array(b) for i, (W, b) in enumerate(params)})
    ck = f'/home/claude/pof/v2/ckpt/{args.closure}_{variant}_s{args.seed}{args.tag}.npz'
    np.savez(ck, **flat, dT_scale=dT_scale, df_scale=(df_scale if kin else 0.0))

    # ------------------------------------------------------- rollout evaluation on test set
    dts = d['test_t']
    Ttest = d['test_T']; th_t = d['test_th']; L_t = d['test_L']
    ftest = d['test_f'] if kin else None
    nTr, nM, _ = Ttest.shape

    @jax.jit
    def roll_step(params, T0, th, L, dt, f0=None):
        return predict(params, T0, th, L, dt, f0)

    def front_pos(fprof):
        prof = fprof - 0.5
        s = np.sign(prof)
        idx = np.where(np.diff(s) < 0)[0]
        if len(idx) == 0:
            return 0.0 if prof[0] < 0 else 1.0
        i = idx[0]
        x0, x1, p0, p1 = (i + .5) * dx, (i + 1.5) * dx, prof[i], prof[i + 1]
        return x0 + (x1 - x0) * p0 / (p0 - p1)

    mets = dict(relT=[], front=[], dE=[], rE=[], rS_minus=[],
                rf=[], sphneg=[], fout=[], relf=[],
                Pm_0=[], Pm_001=[], Pm_01=[], Pm_1=[], Dminus=[],
                neg_near=[], neg_mid=[], neg_far=[], frontT_end=[])
    ETAS = {'Pm_0': 0.0, 'Pm_001': 0.01, 'Pm_01': 0.1, 'Pm_1': 1.0}
    def negdiag(T1s, sph, dtv):
        o = {k: float(jnp.mean(sph < -eta * SIG0 - 1e-12)) for k, eta in ETAS.items()}
        o['Dminus'] = float(jnp.sum(jnp.maximum(0., -sph) * dx * dtv[:, None]))
        dev = jnp.abs(T1s - T_M); neg = sph < -1e-12
        tot = jnp.maximum(jnp.sum(neg), 1)
        o['neg_near'] = float(jnp.sum(neg & (dev < 0.01)) / tot)
        o['neg_mid'] = float(jnp.sum(neg & (dev >= 0.01) & (dev < 0.05)) / tot)
        o['neg_far'] = float(jnp.sum(neg & (dev >= 0.05)) / tot)
        return o
    for j in range(nTr):
        th_j = jnp.full((1,), th_t[j]); L_j = jnp.full((1,), L_t[j])
        Tp = [Ttest[j, 0][None, :]]
        fp = [ftest[j, 0][None, :]] if kin else None
        for k in range(nM - 1):
            dtk = jnp.full((1,), dts[j, k + 1] - dts[j, k])
            if kin:
                Tn, fn = roll_step(params, jnp.array(Tp[-1]), th_j, L_j, dtk, jnp.array(fp[-1]))
                fp.append(np.array(fn))
            else:
                Tn = roll_step(params, jnp.array(Tp[-1]), th_j, L_j, dtk)
            Tp.append(np.array(Tn))
        Tp = np.concatenate(Tp, 0)
        relT = np.linalg.norm(Tp[1:] - Ttest[j, 1:]) / np.linalg.norm(Ttest[j, 1:])
        mets['relT'].append(float(relT))
        if kin:
            fp = np.concatenate(fp, 0)
            fr_p = np.array([front_pos(fk) for fk in fp[1:]])
            fr_r = np.array([front_pos(fk) for fk in ftest[j, 1:]])
            mets['relf'].append(float(np.linalg.norm(fp[1:] - ftest[j, 1:]) /
                                      (np.linalg.norm(ftest[j, 1:]) + 1e-12)))
            mets['fout'].append(float(np.mean((fp < -1e-6) | (fp > 1 + 1e-6))))
        else:
            fprof_p = 0.5 * (1 + np.tanh((Tp - T_M) / w))
            fprof_r = 0.5 * (1 + np.tanh((Ttest[j] - T_M) / w))
            fr_p = np.array([front_pos(fk) for fk in fprof_p[1:]])
            fr_r = np.array([front_pos(fk) for fk in fprof_r[1:]])
        mets['front'].append(float(np.mean(np.abs(fr_p - fr_r))))
        # global energy defect at final time (macro-trapezoid Qin), minus reference same-way value
        def defect(Tarr, farr=None):
            if kin:
                E = (Tarr + L_t[j] * farr).sum(axis=1) * dx
            else:
                E = np.array([np.sum(np.array(h_eq(jnp.array(Tk), L_t[j], w))) * dx for Tk in Tarr])
            q0 = (th_t[j] - Tarr[:, 0]) / (0.5 * dx)
            Qin = np.concatenate([[0.0], np.cumsum(0.5 * (q0[1:] + q0[:-1]) * np.diff(dts[j]))])
            return (E - E[0]) - Qin
        dE_p = defect(Tp, fp if kin else None)
        dE_r = defect(Ttest[j], ftest[j] if kin else None)
        mets['dE'].append(float(abs(dE_p[-1] - dE_r[-1]) / (abs(dE_r[-1]) + np.abs(dE_r).max() + 1e-12)))
        # residual diagnostics on predicted pairs
        T0s = jnp.array(Tp[:-1]); T1s = jnp.array(Tp[1:])
        dtv = jnp.array(np.diff(dts[j]))
        thv = jnp.full((nM - 1,), th_t[j]); Lv = jnp.full((nM - 1,), L_t[j])
        if kin:
            rE, rf, sph = resid_kin(T0s, jnp.array(fp[:-1]), T1s, jnp.array(fp[1:]), dtv, thv, Lv, dx, M0)
            mets['rE'].append(float(jnp.sqrt(jnp.mean(rE ** 2)) / E0))
            mets['rf'].append(float(jnp.sqrt(jnp.mean(rf ** 2)) / F0))
            mets['sphneg'].append(float(jnp.mean(sph < -1e-9)))
            for k_, v_ in negdiag(T1s, sph, dtv).items():
                mets[k_].append(v_)
            if j == 0 or True:
                # reference-trajectory floor for the same diagnostics
                rT0 = jnp.array(Ttest[j, :-1]); rT1 = jnp.array(Ttest[j, 1:])
                _, _, sphr = resid_kin(rT0, jnp.array(ftest[j, :-1]), rT1, jnp.array(ftest[j, 1:]),
                                       dtv, thv, Lv, dx, M0)
                for k_, v_ in negdiag(rT1, sphr, dtv).items():
                    mets.setdefault('ref_' + k_, []).append(v_)
        else:
            rE, rS = resid_eq(T0s, T1s, dtv, thv, Lv, dx, w, tab)
            mets['rE'].append(float(jnp.sqrt(jnp.mean(rE ** 2)) / E0))
            mets['rS_minus'].append(float(jnp.sqrt(jnp.mean((rS - rE / T1s) ** 2)) /
                                          (float(jnp.sqrt(jnp.mean((rE / T1s) ** 2))) + 1e-12)))

    res = dict(closure=args.closure, variant=variant, seed=args.seed, lam=lam, ntrain=nt,
               resid=args.resid, data=args.data or f'data_{args.closure}.npz',
               scales=dict(E0=E0, S0=(S0 if not kin else None), F0=(F0 if kin else None),
                           SIG0=(SIG0 if kin else None), dT_scale=dT_scale),
               diag_init=diag_init, diag_final=diag_final,
               steps=args.steps, val_loss=float(best[0]), train_time_s=train_time,
               n_params=int(sum(np.prod(W.shape) + b.shape[0] for W, b in params)),
               metrics={k: (float(np.mean(v)) if len(v) else None) for k, v in mets.items()},
               metrics_std={k: (float(np.std(v)) if len(v) else None) for k, v in mets.items()},
               per_traj={k: [float(z) for z in v] for k, v in mets.items()})
    out = args.out or f'/home/claude/pof/v2/runs/{args.closure}_{variant}_s{args.seed}{args.tag}.json'
    import os; os.makedirs(os.path.dirname(out), exist_ok=True)
    json.dump(res, open(out, 'w'), indent=1)
    print(json.dumps(dict(variant=variant, seed=args.seed, val=best[0],
                          **{k: res['metrics'][k] for k in ('relT', 'front', 'dE', 'rE')}),
                     indent=None))

if __name__ == '__main__':
    main()
''')

**`v2/train2d.py`** — Two-dimensional operator training ($\lambda=10^{-4}$, $10^{-2}$, $10^{-1}$).

In [ ]:
writefile('v2/train2d.py', r'''"""2D operator ablation on convection-driven melting (equilibrium closure).
Variants: D, E, ES, EET. Same backbone/data/optimizer/seeds; only loss coefficients differ."""
import argparse, json, time, os
import numpy as np
import jax, jax.numpy as jnp, optax

DTH = 0.2

def f_eq(th, ws): return 0.5 * (1.0 + jnp.tanh(th / ws))

def make_G(ws, Tmin=0.6, Tmax=1.6, n=400001):
    Tg = np.linspace(Tmin, Tmax, n); wa = ws * DTH
    integ = (0.5 / (wa * np.cosh((Tg - 1.0) / wa) ** 2)) / Tg
    G = np.concatenate([[0.0], np.cumsum(0.5 * (integ[1:] + integ[:-1]) * np.diff(Tg))])
    G -= np.interp(1.0, Tg, G)
    return jnp.array(Tg), jnp.array(G)

def faces_from_centers(u, v):
    """centered (B,N,N) -> MAC faces with zero normal velocity at walls."""
    B, N, _ = u.shape
    uf = jnp.concatenate([jnp.zeros((B, 1, N)), 0.5 * (u[:, 1:] + u[:, :-1]), jnp.zeros((B, 1, N))], axis=1)
    vf = jnp.concatenate([jnp.zeros((B, N, 1)), 0.5 * (v[:, :, 1:] + v[:, :, :-1]), jnp.zeros((B, N, 1))], axis=2)
    return uf, vf

def cond_fluxes(T, h, Th_abs, Tc_abs):
    B, N, _ = T.shape
    gl = 2 * Th_abs - T[:, :1]; gr = 2 * Tc_abs - T[:, -1:]
    qx = jnp.concatenate([-(T[:, :1] - gl) / h, -(T[:, 1:] - T[:, :-1]) / h, -(gr - T[:, -1:]) / h], axis=1)
    qy = jnp.concatenate([jnp.zeros((B, N, 1)), -(T[:, :, 1:] - T[:, :, :-1]) / h, jnp.zeros((B, N, 1))], axis=2)
    return qx, qy

def upwind(q, uf, vf):
    B, N, _ = q.shape
    Fx_in = jnp.where(uf[:, 1:-1] > 0, q[:, :-1], q[:, 1:]) * uf[:, 1:-1]
    Fy_in = jnp.where(vf[:, :, 1:-1] > 0, q[:, :, :-1], q[:, :, 1:]) * vf[:, :, 1:-1]
    Fx = jnp.concatenate([jnp.zeros((B, 1, N)), Fx_in, jnp.zeros((B, 1, N))], axis=1)
    Fy = jnp.concatenate([jnp.zeros((B, N, 1)), Fy_in, jnp.zeros((B, N, 1))], axis=2)
    return Fx, Fy

def div(Fx, Fy, h):
    return (Fx[:, 1:] - Fx[:, :-1]) / h + (Fy[:, :, 1:] - Fy[:, :, :-1]) / h

def residuals(th0, th1, u1, v1, dt, Ste, ws, h, thc, Gtab):
    """Absolute-variable energy and independently discretized entropy residuals."""
    f0, f1 = f_eq(th0, ws), f_eq(th1, ws)
    H0 = DTH * (th0 + f0 / Ste[:, None, None]); H1 = DTH * (th1 + f1 / Ste[:, None, None])
    T0 = 1.0 + DTH * th0; T1 = 1.0 + DTH * th1
    uf, vf = faces_from_centers(u1, v1)
    qx, qy = cond_fluxes(T1, h, 1.0 + DTH, 1.0 + DTH * thc)
    Ax, Ay = upwind(H1, uf, vf)
    rE = (H1 - H0) / dt[:, None, None] + div(Ax + qx, Ay + qy, h)
    Tg, G = Gtab
    Labs = DTH / Ste[:, None, None]
    s0 = jnp.log(T0) + Labs * jnp.interp(T0, Tg, G); s1 = jnp.log(T1) + Labs * jnp.interp(T1, Tg, G)
    B, N, _ = T1.shape
    Tfx = jnp.concatenate([jnp.full((B, 1, N), 1.0 + DTH), 0.5 * (T1[:, 1:] + T1[:, :-1]),
                           jnp.full((B, 1, N), 1.0 + DTH * thc)], axis=1)
    Tfy = jnp.concatenate([T1[:, :, :1], 0.5 * (T1[:, :, 1:] + T1[:, :, :-1]), T1[:, :, -1:]], axis=2)
    Sx, Sy = upwind(s1, uf, vf)
    divS = div(Sx + qx / Tfx, Sy + qy / Tfy, h)
    sig = (0.5 * (qx[:, :-1] ** 2 + qx[:, 1:] ** 2) + 0.5 * (qy[:, :, :-1] ** 2 + qy[:, :, 1:] ** 2)) / T1 ** 2
    rS = (s1 - s0) / dt[:, None, None] + divS - sig
    return rE, rS, T1

def init_params(key, cin, cout, width=24, k=3, nlayers=4):
    keys = jax.random.split(key, nlayers); sizes = [cin] + [width] * (nlayers - 1) + [cout]; ps = []
    for i in range(nlayers):
        W = jax.random.normal(keys[i], (sizes[i + 1], sizes[i], k, k)) * jnp.sqrt(2.0 / (sizes[i] * k * k))
        ps.append((W, jnp.zeros((sizes[i + 1],))))
    return ps

def conv2d(x, W, b):
    return jax.lax.conv_general_dilated(x, W, (1, 1), 'SAME', dimension_numbers=('NCHW', 'OIHW', 'NCHW')) + b[None, :, None, None]

def forward(ps, x):
    h = x
    for W, b in ps[:-1]:
        h = jax.nn.gelu(conv2d(h, W, b))
    return conv2d(h, *ps[-1])

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--variant', required=True)
    ap.add_argument('--seed', type=int, default=0)
    ap.add_argument('--lam', type=float, default=1e-3)
    ap.add_argument('--steps', type=int, default=1500)
    ap.add_argument('--out', default=None)
    a = ap.parse_args()
    d = np.load('/home/claude/pof/v2/data_2d.npz')
    N = int(d['NC']); h = 1.0 / N; ws = float(d['ws']); thc = float(d['theta_c'])
    Gtab = make_G(ws)
    xc = (np.arange(N) + 0.5) * h
    X, Y = np.meshgrid(xc, xc, indexing='ij')

    def pairs(k):
        th, u, v, t = d[f'{k}_th'], d[f'{k}_u'], d[f'{k}_v'], d[f'{k}_t']
        n, m = th.shape[:2]
        sl0 = lambda z: z[:, :-1].reshape(-1, N, N); sl1 = lambda z: z[:, 1:].reshape(-1, N, N)
        dt = np.diff(t, axis=1).reshape(-1)
        Ra = np.repeat(d[f'{k}_Ra'], m - 1); St = np.repeat(d[f'{k}_Ste'], m - 1)
        return [jnp.array(z) for z in (sl0(th), sl0(u), sl0(v), sl1(th), sl1(u), sl1(v), dt, Ra, St)]
    tr, va = pairs('train'), pairs('val')
    s_th = float(jnp.std(tr[3] - tr[0])); s_u = float(jnp.std(tr[4] - tr[1])); s_v = float(jnp.std(tr[5] - tr[2]))
    U0 = float(jnp.std(tr[4])) + 1e-6
    rE0, rS0, T1r = residuals(tr[0], tr[3], tr[4], tr[5], tr[6], tr[8], ws, h, thc, Gtab)
    E0 = float(jnp.sqrt(jnp.mean(rE0 ** 2))); S0 = float(jnp.sqrt(jnp.mean((rE0 / T1r) ** 2)))
    ident_ref = float(jnp.sqrt(jnp.mean((rS0 - rE0 / T1r) ** 2)) / S0)

    Xj, Yj = jnp.array(X), jnp.array(Y)
    def inputs(th, u, v, Ra, St):
        B = th.shape[0]
        c = lambda s: jnp.broadcast_to(s[:, None, None], (B, N, N))
        return jnp.stack([th, u / U0, v / U0, c(Ra / 1e5), c(St / 0.2),
                          jnp.broadcast_to(Xj, (B, N, N)), jnp.broadcast_to(Yj, (B, N, N))], axis=1)
    def predict(ps, th, u, v, Ra, St):
        y = forward(ps, inputs(th, u, v, Ra, St))
        return th + s_th * y[:, 0], u + s_u * y[:, 1], v + s_v * y[:, 2]
    lam, var = a.lam, a.variant
    def loss_fn(ps, b, terms_only=False):
        th0, u0, v0, th1t, u1t, v1t, dt, Ra, St = b
        th1, u1, v1 = predict(ps, th0, u0, v0, Ra, St)
        Ld = (jnp.mean(((th1 - th1t) / s_th) ** 2) + jnp.mean(((u1 - u1t) / s_u) ** 2)
              + jnp.mean(((v1 - v1t) / s_v) ** 2))
        rE, rS, T1 = residuals(th0, th1, u1, v1, dt, St, ws, h, thc, Gtab)
        terms = dict(E=jnp.mean((rE / E0) ** 2), S=jnp.mean((rS / S0) ** 2), EET=jnp.mean((rE / T1 / S0) ** 2))
        if terms_only: return Ld, terms
        Lt = Ld
        if var in ('E', 'ES', 'EET'): Lt = Lt + lam * terms['E']
        if var == 'ES': Lt = Lt + lam * terms['S']
        if var == 'EET': Lt = Lt + lam * terms['EET']
        return Lt, Ld
    ps = init_params(jax.random.PRNGKey(a.seed), 7, 3)
    opt = optax.adam(optax.cosine_decay_schedule(3e-3, a.steps, 1e-2)); os_ = opt.init(ps)
    @jax.jit
    def step(ps, os_, b):
        (Lt, Ld), g = jax.value_and_grad(loss_fn, has_aux=True)(ps, b)
        up, os_ = opt.update(g, os_); return optax.apply_updates(ps, up), os_, Lt
    @jax.jit
    def vloss(ps): return loss_fn(ps, tuple(va))[1]
    gn = lambda g: float(jnp.sqrt(sum(jnp.sum(x ** 2) for x in jax.tree_util.tree_leaves(g))))
    db = tuple(z[:32] for z in tr)
    def diag(p):
        o = {'data': gn(jax.grad(lambda q: loss_fn(q, db, True)[0])(p))}
        for nm in ('E', 'S', 'EET'):
            o[nm] = gn(jax.grad(lambda q, nm=nm: lam * loss_fn(q, db, True)[1][nm])(p))
        return o
    dinit = diag(ps)
    grad_hist = []
    nb = tr[0].shape[0]; bs = 16; rng = np.random.default_rng(a.seed); best = (1e18, ps); t0 = time.time()
    for it in range(a.steps):
        idx = rng.choice(nb, bs, replace=False)
        ps, os_, Lt = step(ps, os_, tuple(z[idx] for z in tr))
        if it == 0 or (it + 1) % 300 == 0:
            dd = diag(ps); grad_hist.append(dict(step=it + 1, **dd))
        if (it + 1) % 100 == 0:
            vl = float(vloss(ps))
            if vl < best[0]: best = (vl, ps)
    ps = best[1]; ttime = time.time() - t0; dfin = diag(ps)

    # ---------------- rollout evaluation
    @jax.jit
    def rstep(ps, th, u, v, Ra, St): return predict(ps, th, u, v, Ra, St)
    def front_rows(th):
        f = 1.0 / (1.0 + np.exp(-2 * th / ws))   # = f_eq
        out = []
        for j in range(N):
            prof = f[:, j] - 0.5; s = np.sign(prof); idx = np.where(np.diff(s) < 0)[0]
            if len(idx) == 0: out.append(0.0 if prof[0] < 0 else 1.0); continue
            i = idx[0]; out.append(xc[i] + h * prof[i] / (prof[i] - prof[i + 1]))
        return np.array(out)
    def nu(th): return float(np.sum(-(th[0] - (2 * 1.0 - th[0])) / h) * h)
    M = dict(relT=[], relU=[], front=[], nu=[], fbar=[], rE=[], ident=[])
    th_t, u_t, v_t, t_t = d['test_th'], d['test_u'], d['test_v'], d['test_t']
    for j in range(th_t.shape[0]):
        Ra = jnp.array([d['test_Ra'][j]]); St = jnp.array([d['test_Ste'][j]])
        th, u, v = [jnp.array(th_t[j, :1])], [jnp.array(u_t[j, :1])], [jnp.array(v_t[j, :1])]
        for k in range(th_t.shape[1] - 1):
            a1, b1, c1 = rstep(ps, th[-1], u[-1], v[-1], Ra, St); th.append(a1); u.append(b1); v.append(c1)
        TH = np.concatenate([np.array(z) for z in th]); UU = np.concatenate([np.array(z) for z in u]); VV = np.concatenate([np.array(z) for z in v])
        if j == 0:
            os.makedirs('/home/claude/pof/v2/ckpt2d', exist_ok=True)
            np.savez_compressed(f'/home/claude/pof/v2/ckpt2d/{var}_s{a.seed}_rollout0.npz', th=TH, u=UU, v=VV,
                                **{f'W{i}': np.array(W) for i, (W, b) in enumerate(ps)},
                                **{f'b{i}': np.array(b) for i, (W, b) in enumerate(ps)})
        R = slice(1, None)
        M['relT'].append(float(np.linalg.norm(TH[R] - th_t[j, R]) / np.linalg.norm(th_t[j, R])))
        M['relU'].append(float(np.sqrt(np.sum((UU[R] - u_t[j, R]) ** 2 + (VV[R] - v_t[j, R]) ** 2) /
                                       np.sum(u_t[j, R] ** 2 + v_t[j, R] ** 2))))
        M['front'].append(float(np.mean([np.mean(np.abs(front_rows(TH[k]) - front_rows(th_t[j, k]))) for k in range(1, TH.shape[0])])))
        M['nu'].append(float(np.mean([abs(nu(TH[k]) - nu(th_t[j, k])) / abs(nu(th_t[j, k])) for k in range(1, TH.shape[0])])))
        fp = 0.5 * (1 + np.tanh(TH / ws)); fr = 0.5 * (1 + np.tanh(th_t[j] / ws))
        M['fbar'].append(float(np.mean(np.abs(fp[R].mean(axis=(1, 2)) - fr[R].mean(axis=(1, 2))))))
        dtv = jnp.array(np.diff(t_t[j])); Stv = jnp.full((TH.shape[0] - 1,), d['test_Ste'][j])
        rE, rS, T1 = residuals(jnp.array(TH[:-1]), jnp.array(TH[1:]), jnp.array(UU[1:]), jnp.array(VV[1:]), dtv, Stv, ws, h, thc, Gtab)
        M['rE'].append(float(jnp.sqrt(jnp.mean(rE ** 2)) / E0))
        M['ident'].append(float(jnp.sqrt(jnp.mean((rS - rE / T1) ** 2)) / jnp.sqrt(jnp.mean((rE / T1) ** 2))))
    res = dict(variant=var, seed=a.seed, lam=lam, val_loss=float(best[0]), train_time_s=ttime,
               scales=dict(E0=E0, S0=S0, ident_ref=ident_ref), diag_init=dinit, diag_final=dfin, grad_hist=grad_hist,
               metrics={k: float(np.mean(v)) for k, v in M.items()}, per_traj=M)
    out = a.out or f'/home/claude/pof/v2/runs2d/{var}_s{a.seed}.json'
    os.makedirs(os.path.dirname(out), exist_ok=True); json.dump(res, open(out, 'w'), indent=1)
    print(json.dumps(dict(variant=var, seed=a.seed, lam=lam, val=best[0], t=round(ttime), **res['metrics'])))

if __name__ == '__main__':
    main()
''')

**`v2/train2d_r4.py`** — Two-dimensional training with continuity correction (E+S$_c$) and common projection.

In [ ]:
writefile('v2/train2d_r4.py', r'''"""Round-4 variant of train2d.py: adds the continuity-corrected entropy variant ESC and an optional
common discrete projection (--proj) of the reconstructed face velocities used by every residual.
2D operator ablation on convection-driven melting (equilibrium closure).
Variants: D, E, ES, EET. Same backbone/data/optimizer/seeds; only loss coefficients differ."""
import argparse, json, time, os
import numpy as np
import jax, jax.numpy as jnp, optax

DTH = 0.2

def f_eq(th, ws): return 0.5 * (1.0 + jnp.tanh(th / ws))

def make_G(ws, Tmin=0.6, Tmax=1.6, n=400001):
    Tg = np.linspace(Tmin, Tmax, n); wa = ws * DTH
    integ = (0.5 / (wa * np.cosh((Tg - 1.0) / wa) ** 2)) / Tg
    G = np.concatenate([[0.0], np.cumsum(0.5 * (integ[1:] + integ[:-1]) * np.diff(Tg))])
    G -= np.interp(1.0, Tg, G)
    return jnp.array(Tg), jnp.array(G)

def faces_from_centers(u, v):
    """centered (B,N,N) -> MAC faces with zero normal velocity at walls."""
    B, N, _ = u.shape
    uf = jnp.concatenate([jnp.zeros((B, 1, N)), 0.5 * (u[:, 1:] + u[:, :-1]), jnp.zeros((B, 1, N))], axis=1)
    vf = jnp.concatenate([jnp.zeros((B, N, 1)), 0.5 * (v[:, :, 1:] + v[:, :, :-1]), jnp.zeros((B, N, 1))], axis=2)
    return uf, vf

def cond_fluxes(T, h, Th_abs, Tc_abs):
    B, N, _ = T.shape
    gl = 2 * Th_abs - T[:, :1]; gr = 2 * Tc_abs - T[:, -1:]
    qx = jnp.concatenate([-(T[:, :1] - gl) / h, -(T[:, 1:] - T[:, :-1]) / h, -(gr - T[:, -1:]) / h], axis=1)
    qy = jnp.concatenate([jnp.zeros((B, N, 1)), -(T[:, :, 1:] - T[:, :, :-1]) / h, jnp.zeros((B, N, 1))], axis=2)
    return qx, qy

def upwind(q, uf, vf):
    B, N, _ = q.shape
    Fx_in = jnp.where(uf[:, 1:-1] > 0, q[:, :-1], q[:, 1:]) * uf[:, 1:-1]
    Fy_in = jnp.where(vf[:, :, 1:-1] > 0, q[:, :, :-1], q[:, :, 1:]) * vf[:, :, 1:-1]
    Fx = jnp.concatenate([jnp.zeros((B, 1, N)), Fx_in, jnp.zeros((B, 1, N))], axis=1)
    Fy = jnp.concatenate([jnp.zeros((B, N, 1)), Fy_in, jnp.zeros((B, N, 1))], axis=2)
    return Fx, Fy

def div(Fx, Fy, h):
    return (Fx[:, 1:] - Fx[:, :-1]) / h + (Fy[:, :, 1:] - Fy[:, :, :-1]) / h

_PROJ = {}
def make_proj(N, h):
    """Dense pseudo-inverse of the cell-centred Neumann Laplacian (MAC faces, zero wall flux)."""
    n = N * N; A = np.zeros((n, n)); idx = lambda i, j: i * N + j
    for i in range(N):
        for j in range(N):
            for di, dj in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                ii, jj = i + di, j + dj
                if 0 <= ii < N and 0 <= jj < N:
                    A[idx(i, j), idx(i, j)] -= 1 / h ** 2; A[idx(i, j), idx(ii, jj)] += 1 / h ** 2
    return jnp.array(np.linalg.pinv(A, rcond=1e-8, hermitian=True))

def project(uf, vf, h):
    B = uf.shape[0]; N = uf.shape[2]
    dv = ((uf[:, 1:] - uf[:, :-1]) + (vf[:, :, 1:] - vf[:, :, :-1])) / h
    phi = (dv.reshape(B, -1) @ _PROJ['Ainv'].T).reshape(B, N, N)
    uf = uf.at[:, 1:-1].add(-(phi[:, 1:] - phi[:, :-1]) / h)
    vf = vf.at[:, :, 1:-1].add(-(phi[:, :, 1:] - phi[:, :, :-1]) / h)
    return uf, vf

def residuals(th0, th1, u1, v1, dt, Ste, ws, h, thc, Gtab, proj=False, full=False):
    """Absolute-variable energy and independently discretized entropy residuals."""
    f0, f1 = f_eq(th0, ws), f_eq(th1, ws)
    H0 = DTH * (th0 + f0 / Ste[:, None, None]); H1 = DTH * (th1 + f1 / Ste[:, None, None])
    T0 = 1.0 + DTH * th0; T1 = 1.0 + DTH * th1
    uf, vf = faces_from_centers(u1, v1)
    if proj: uf, vf = project(uf, vf, h)
    rm = div(uf, vf, h)
    qx, qy = cond_fluxes(T1, h, 1.0 + DTH, 1.0 + DTH * thc)
    Ax, Ay = upwind(H1, uf, vf)
    rE = (H1 - H0) / dt[:, None, None] + div(Ax + qx, Ay + qy, h)
    Tg, G = Gtab
    Labs = DTH / Ste[:, None, None]
    s0 = jnp.log(T0) + Labs * jnp.interp(T0, Tg, G); s1 = jnp.log(T1) + Labs * jnp.interp(T1, Tg, G)
    B, N, _ = T1.shape
    Tfx = jnp.concatenate([jnp.full((B, 1, N), 1.0 + DTH), 0.5 * (T1[:, 1:] + T1[:, :-1]),
                           jnp.full((B, 1, N), 1.0 + DTH * thc)], axis=1)
    Tfy = jnp.concatenate([T1[:, :, :1], 0.5 * (T1[:, :, 1:] + T1[:, :, :-1]), T1[:, :, -1:]], axis=2)
    Sx, Sy = upwind(s1, uf, vf)
    divS = div(Sx + qx / Tfx, Sy + qy / Tfy, h)
    sig = (0.5 * (qx[:, :-1] ** 2 + qx[:, 1:] ** 2) + 0.5 * (qy[:, :, :-1] ** 2 + qy[:, :, 1:] ** 2)) / T1 ** 2
    rS = (s1 - s0) / dt[:, None, None] + divS - sig
    if full: return rE, rS, T1, s1, H1, rm
    return rE, rS, T1

def init_params(key, cin, cout, width=24, k=3, nlayers=4):
    keys = jax.random.split(key, nlayers); sizes = [cin] + [width] * (nlayers - 1) + [cout]; ps = []
    for i in range(nlayers):
        W = jax.random.normal(keys[i], (sizes[i + 1], sizes[i], k, k)) * jnp.sqrt(2.0 / (sizes[i] * k * k))
        ps.append((W, jnp.zeros((sizes[i + 1],))))
    return ps

def conv2d(x, W, b):
    return jax.lax.conv_general_dilated(x, W, (1, 1), 'SAME', dimension_numbers=('NCHW', 'OIHW', 'NCHW')) + b[None, :, None, None]

def forward(ps, x):
    h = x
    for W, b in ps[:-1]:
        h = jax.nn.gelu(conv2d(h, W, b))
    return conv2d(h, *ps[-1])

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--variant', required=True)
    ap.add_argument('--seed', type=int, default=0)
    ap.add_argument('--lam', type=float, default=1e-3)
    ap.add_argument('--steps', type=int, default=1500)
    ap.add_argument('--out', default=None)
    ap.add_argument('--proj', type=int, default=0)
    a = ap.parse_args()
    d = np.load('/home/claude/pof/v2/data_2d.npz')
    N = int(d['NC']); h = 1.0 / N; ws = float(d['ws']); thc = float(d['theta_c'])
    Gtab = make_G(ws)
    if a.proj: _PROJ['Ainv'] = make_proj(N, h)
    PR = bool(a.proj)
    xc = (np.arange(N) + 0.5) * h
    X, Y = np.meshgrid(xc, xc, indexing='ij')

    def pairs(k):
        th, u, v, t = d[f'{k}_th'], d[f'{k}_u'], d[f'{k}_v'], d[f'{k}_t']
        n, m = th.shape[:2]
        sl0 = lambda z: z[:, :-1].reshape(-1, N, N); sl1 = lambda z: z[:, 1:].reshape(-1, N, N)
        dt = np.diff(t, axis=1).reshape(-1)
        Ra = np.repeat(d[f'{k}_Ra'], m - 1); St = np.repeat(d[f'{k}_Ste'], m - 1)
        return [jnp.array(z) for z in (sl0(th), sl0(u), sl0(v), sl1(th), sl1(u), sl1(v), dt, Ra, St)]
    tr, va = pairs('train'), pairs('val')
    s_th = float(jnp.std(tr[3] - tr[0])); s_u = float(jnp.std(tr[4] - tr[1])); s_v = float(jnp.std(tr[5] - tr[2]))
    U0 = float(jnp.std(tr[4])) + 1e-6
    rE0, rS0, T1r = residuals(tr[0], tr[3], tr[4], tr[5], tr[6], tr[8], ws, h, thc, Gtab)
    E0std = float(jnp.sqrt(jnp.mean(rE0 ** 2)))
    if PR:
        rE0, rS0, T1r = residuals(tr[0], tr[3], tr[4], tr[5], tr[6], tr[8], ws, h, thc, Gtab, proj=True)
    E0 = float(jnp.sqrt(jnp.mean(rE0 ** 2))); S0 = float(jnp.sqrt(jnp.mean((rE0 / T1r) ** 2)))
    ident_ref = float(jnp.sqrt(jnp.mean((rS0 - rE0 / T1r) ** 2)) / S0)

    Xj, Yj = jnp.array(X), jnp.array(Y)
    def inputs(th, u, v, Ra, St):
        B = th.shape[0]
        c = lambda s: jnp.broadcast_to(s[:, None, None], (B, N, N))
        return jnp.stack([th, u / U0, v / U0, c(Ra / 1e5), c(St / 0.2),
                          jnp.broadcast_to(Xj, (B, N, N)), jnp.broadcast_to(Yj, (B, N, N))], axis=1)
    def predict(ps, th, u, v, Ra, St):
        y = forward(ps, inputs(th, u, v, Ra, St))
        return th + s_th * y[:, 0], u + s_u * y[:, 1], v + s_v * y[:, 2]
    lam, var = a.lam, a.variant
    def loss_fn(ps, b, terms_only=False):
        th0, u0, v0, th1t, u1t, v1t, dt, Ra, St = b
        th1, u1, v1 = predict(ps, th0, u0, v0, Ra, St)
        Ld = (jnp.mean(((th1 - th1t) / s_th) ** 2) + jnp.mean(((u1 - u1t) / s_u) ** 2)
              + jnp.mean(((v1 - v1t) / s_v) ** 2))
        rE, rS, T1, s1, H1, rm = residuals(th0, th1, u1, v1, dt, St, ws, h, thc, Gtab, proj=PR, full=True)
        rSc = rS - (s1 - H1 / T1) * rm
        terms = dict(E=jnp.mean((rE / E0) ** 2), S=jnp.mean((rS / S0) ** 2), EET=jnp.mean((rE / T1 / S0) ** 2),
                     SC=jnp.mean((rSc / S0) ** 2))
        if terms_only: return Ld, terms
        Lt = Ld
        if var in ('E', 'ES', 'EET', 'ESC'): Lt = Lt + lam * terms['E']
        if var == 'ESC': Lt = Lt + lam * terms['SC']
        if var == 'ES': Lt = Lt + lam * terms['S']
        if var == 'EET': Lt = Lt + lam * terms['EET']
        return Lt, Ld
    ps = init_params(jax.random.PRNGKey(a.seed), 7, 3)
    opt = optax.adam(optax.cosine_decay_schedule(3e-3, a.steps, 1e-2)); os_ = opt.init(ps)
    @jax.jit
    def step(ps, os_, b):
        (Lt, Ld), g = jax.value_and_grad(loss_fn, has_aux=True)(ps, b)
        up, os_ = opt.update(g, os_); return optax.apply_updates(ps, up), os_, Lt
    @jax.jit
    def vloss(ps): return loss_fn(ps, tuple(va))[1]
    gn = lambda g: float(jnp.sqrt(sum(jnp.sum(x ** 2) for x in jax.tree_util.tree_leaves(g))))
    db = tuple(z[:32] for z in tr)
    def diag(p):
        o = {'data': gn(jax.grad(lambda q: loss_fn(q, db, True)[0])(p))}
        for nm in ('E', 'S', 'EET', 'SC'):
            o[nm] = gn(jax.grad(lambda q, nm=nm: lam * loss_fn(q, db, True)[1][nm])(p))
        return o
    dinit = diag(ps)
    grad_hist = []
    nb = tr[0].shape[0]; bs = 16; rng = np.random.default_rng(a.seed); best = (1e18, ps); t0 = time.time()
    for it in range(a.steps):
        idx = rng.choice(nb, bs, replace=False)
        ps, os_, Lt = step(ps, os_, tuple(z[idx] for z in tr))
        if it == 0 or (it + 1) % 300 == 0:
            dd = diag(ps); grad_hist.append(dict(step=it + 1, **dd))
        if (it + 1) % 100 == 0:
            vl = float(vloss(ps))
            if vl < best[0]: best = (vl, ps)
    ps = best[1]; ttime = time.time() - t0; dfin = diag(ps)

    # ---------------- rollout evaluation
    @jax.jit
    def rstep(ps, th, u, v, Ra, St): return predict(ps, th, u, v, Ra, St)
    def front_rows(th):
        f = 1.0 / (1.0 + np.exp(-2 * th / ws))   # = f_eq
        out = []
        for j in range(N):
            prof = f[:, j] - 0.5; s = np.sign(prof); idx = np.where(np.diff(s) < 0)[0]
            if len(idx) == 0: out.append(0.0 if prof[0] < 0 else 1.0); continue
            i = idx[0]; out.append(xc[i] + h * prof[i] / (prof[i] - prof[i + 1]))
        return np.array(out)
    def nu(th): return float(np.sum(-(th[0] - (2 * 1.0 - th[0])) / h) * h)
    M = dict(relT=[], relU=[], front=[], nu=[], fbar=[], rE=[], ident=[])
    th_t, u_t, v_t, t_t = d['test_th'], d['test_u'], d['test_v'], d['test_t']
    for j in range(th_t.shape[0]):
        Ra = jnp.array([d['test_Ra'][j]]); St = jnp.array([d['test_Ste'][j]])
        th, u, v = [jnp.array(th_t[j, :1])], [jnp.array(u_t[j, :1])], [jnp.array(v_t[j, :1])]
        for k in range(th_t.shape[1] - 1):
            a1, b1, c1 = rstep(ps, th[-1], u[-1], v[-1], Ra, St); th.append(a1); u.append(b1); v.append(c1)
        TH = np.concatenate([np.array(z) for z in th]); UU = np.concatenate([np.array(z) for z in u]); VV = np.concatenate([np.array(z) for z in v])
        if j == 0:
            os.makedirs('/home/claude/pof/v2/ckpt2d_r4', exist_ok=True)
            np.savez_compressed(f'/home/claude/pof/v2/ckpt2d_r4/{var}_p{a.proj}_lam{lam:g}_s{a.seed}_rollout0.npz', th=TH, u=UU, v=VV,
                                **{f'W{i}': np.array(W) for i, (W, b) in enumerate(ps)},
                                **{f'b{i}': np.array(b) for i, (W, b) in enumerate(ps)})
        R = slice(1, None)
        M['relT'].append(float(np.linalg.norm(TH[R] - th_t[j, R]) / np.linalg.norm(th_t[j, R])))
        M['relU'].append(float(np.sqrt(np.sum((UU[R] - u_t[j, R]) ** 2 + (VV[R] - v_t[j, R]) ** 2) /
                                       np.sum(u_t[j, R] ** 2 + v_t[j, R] ** 2))))
        M['front'].append(float(np.mean([np.mean(np.abs(front_rows(TH[k]) - front_rows(th_t[j, k]))) for k in range(1, TH.shape[0])])))
        M['nu'].append(float(np.mean([abs(nu(TH[k]) - nu(th_t[j, k])) / abs(nu(th_t[j, k])) for k in range(1, TH.shape[0])])))
        fp = 0.5 * (1 + np.tanh(TH / ws)); fr = 0.5 * (1 + np.tanh(th_t[j] / ws))
        M['fbar'].append(float(np.mean(np.abs(fp[R].mean(axis=(1, 2)) - fr[R].mean(axis=(1, 2))))))
        dtv = jnp.array(np.diff(t_t[j])); Stv = jnp.full((TH.shape[0] - 1,), d['test_Ste'][j])
        rE, rS, T1 = residuals(jnp.array(TH[:-1]), jnp.array(TH[1:]), jnp.array(UU[1:]), jnp.array(VV[1:]), dtv, Stv, ws, h, thc, Gtab)
        M['rE'].append(float(jnp.sqrt(jnp.mean(rE ** 2)) / E0std))
        M['ident'].append(float(jnp.sqrt(jnp.mean((rS - rE / T1) ** 2)) / jnp.sqrt(jnp.mean((rE / T1) ** 2))))
    res = dict(variant=var, seed=a.seed, lam=lam, proj=a.proj, val_loss=float(best[0]), train_time_s=ttime,
               scales=dict(E0=E0, S0=S0, E0std=E0std, ident_ref=ident_ref), diag_init=dinit, diag_final=dfin, grad_hist=grad_hist,
               metrics={k: float(np.mean(v)) for k, v in M.items()}, per_traj=M)
    out = a.out or f'/home/claude/pof/v2/runs2d/{var}_s{a.seed}.json'
    os.makedirs(os.path.dirname(out), exist_ok=True); json.dump(res, open(out, 'w'), indent=1)
    print(json.dumps(dict(variant=var, seed=a.seed, lam=lam, val=best[0], t=round(ttime), **res['metrics'])))

if __name__ == '__main__':
    main()
''')

**`v2/mms_kinetic.py`** — Manufactured solution and nucleation-seed sweep (Fig. 1d,f; Appendix B).

In [ ]:
writefile('v2/mms_kinetic.py', r'''"""Manufactured-solution verification of the 1D internal-variable solver, plus
f0 sensitivity and limiter activity of the production solver."""
import json, sys
import numpy as np
sys.path.insert(0, '/home/claude/pof')
from solvers1d import X_drive, front_from_T

M0, L = 400.0, 2.0
ws, x0, vf = 0.08, 0.2, 0.5

def Tstar(x, t):   return 1.05 + 0.1 * np.cos(np.pi * x) * np.exp(-t)
def Tstar_t(x, t): return -0.1 * np.cos(np.pi * x) * np.exp(-t)
def Tstar_xx(x, t): return -0.1 * np.pi ** 2 * np.cos(np.pi * x) * np.exp(-t)
def fstar(x, t):   return 0.5 * (1 + np.tanh((x - (x0 + vf * t)) / ws))
def fstar_t(x, t):
    c = np.cosh((x - (x0 + vf * t)) / ws)
    return -0.5 * vf / (ws * c * c)

def solve_mms(N, t_end=0.05, cfl=0.25):
    """Same scheme as solvers1d.solve_kinetic (explicit Euler, half-cell Dirichlet ghost,
    adiabatic right wall, conservative increment limiter) with manufactured sources."""
    dx = 1.0 / N; x = (np.arange(N) + 0.5) * dx
    dt = cfl * dx * dx; n = int(np.ceil(t_end / dt)); dt = t_end / n
    T = Tstar(x, 0.0); f = fstar(x, 0.0); t = 0.0; nlim = 0
    for _ in range(n):
        Th = Tstar(0.0, t)
        q = np.empty(N + 1)
        q[1:-1] = -(T[1:] - T[:-1]) / dx
        q[0] = -(T[0] - Th) / (0.5 * dx); q[-1] = 0.0
        lap = -(q[1:] - q[:-1]) / dx
        Sf = fstar_t(x, t) - M0 * fstar(x, t) * (1 - fstar(x, t)) * X_drive(Tstar(x, t), L)
        ST = Tstar_t(x, t) - Tstar_xx(x, t) + L * fstar_t(x, t)
        df = dt * (M0 * f * (1 - f) * X_drive(T, L) + Sf)
        dfl = np.clip(df, -f, 1.0 - f); nlim += int(np.sum(dfl != df))
        T = T + dt * (lap + ST) - L * dfl
        f = f + dfl; t += dt
    eT = np.sqrt(np.mean((T - Tstar(x, t)) ** 2)); ef = np.sqrt(np.mean((f - fstar(x, t)) ** 2))
    return eT, ef, nlim

out = {'mms': []}
for N in [32, 64, 128, 256]:
    eT, ef, nl = solve_mms(N)
    out['mms'].append(dict(N=N, eT=eT, ef=ef, limiter_activations=nl))
    print(f"N={N}: eT={eT:.3e} ef={ef:.3e} limiter={nl}")
for k in range(1, 4):
    a, b = out['mms'][k - 1], out['mms'][k]
    b['order_T'] = float(np.log2(a['eT'] / b['eT'])); b['order_f'] = float(np.log2(a['ef'] / b['ef']))
    print(f"order N={b['N']}: T {b['order_T']:.2f}  f {b['order_f']:.2f}")

# ---------------------------------------------------------------- f0 sensitivity + limiter activity
def solve_kinetic_count(N, T_h, L, M0, t_end, f_init, T_init=0.95, cfl=0.25):
    dx = 1.0 / N; x = (np.arange(N) + 0.5) * dx
    dt = cfl * dx * dx; n = int(np.ceil(t_end / dt)); dt = t_end / n
    T = np.full(N, T_init); f = np.full(N, f_init); nlim = 0; tot = 0
    for _ in range(n):
        q = np.empty(N + 1); q[1:-1] = -(T[1:] - T[:-1]) / dx
        q[0] = -(T[0] - T_h) / (0.5 * dx); q[-1] = 0.0
        lap = -(q[1:] - q[:-1]) / dx
        df = dt * M0 * f * (1 - f) * X_drive(T, L)
        dfl = np.clip(df, -f, 1.0 - f); nlim += int(np.sum(dfl != df)); tot += N
        T = T + dt * lap - L * dfl; f = f + dfl
    return x, T, f, nlim, tot

sens = []
for f0 in [0.001, 0.003, 0.01, 0.03]:
    x, T, f, nl, tot = solve_kinetic_count(128, 1.25, 2.0, 400.0, 0.1, f0)
    fr = front_from_T(x, T, from_f=f)
    sens.append(dict(f0=f0, front=float(fr), limiter_frac=nl / tot, fmax=float(f.max())))
    print(f"f0={f0}: front={fr:.4f} limiter activations {nl}/{tot}")
out['f0_sensitivity'] = sens
# limiter activity on the full production dataset grid (M0=400, f0=0.01)
acts = []
for th in np.linspace(1.15, 1.40, 8):
    for l in np.linspace(1.5, 2.5, 9):
        _, _, _, nl, tot = solve_kinetic_count(128, th, l, 400.0, 0.1, 0.01)
        acts.append(nl / tot)
out['limiter_production'] = dict(max_frac=float(max(acts)), mean_frac=float(np.mean(acts)))
print("production limiter activation fraction: max", max(acts), "mean", np.mean(acts))
json.dump(out, open('/home/claude/pof/v2/mms_kinetic.json', 'w'), indent=1)
''')

**`v2/make_sharp_limit.py`** — Mobility sweep (Fig. 1e).

In [ ]:
writefile('v2/make_sharp_limit.py', r'''"""Mobility sweep (Fig. 1e): kinetic front and interface temperature at t=0.1 vs M0 (T_h=1.25, L=2, f0=0.01)."""
import sys, json, numpy as np
sys.path.insert(0, '/home/claude/pof'); sys.path.insert(0, '/home/claude/pof/v2')
from solvers1d import front_from_T, neumann_front
from solvers1d import X_drive
def solve_kinetic_count(N, T_h, L, M0, t_end, f_init, T_init=0.95, cfl=0.25):
    dx = 1.0 / N; x = (np.arange(N) + 0.5) * dx
    dt = cfl * dx * dx; n = int(np.ceil(t_end / dt)); dt = t_end / n
    T = np.full(N, T_init); f = np.full(N, f_init); nlim = 0; tot = 0
    for _ in range(n):
        q = np.empty(N + 1); q[1:-1] = -(T[1:] - T[:-1]) / dx
        q[0] = -(T[0] - T_h) / (0.5 * dx); q[-1] = 0.0
        lap = -(q[1:] - q[:-1]) / dx
        df = dt * M0 * f * (1 - f) * X_drive(T, L)
        dfl = np.clip(df, -f, 1.0 - f); nlim += int(np.sum(dfl != df)); tot += N
        T = T + dt * lap - L * dfl; f = f + dfl
    return x, T, f, nlim, tot
import os
OUT = os.environ.get("RECON_OUT", "/home/claude/pof/v2/sharp_limit.json")
rows = []
nf = float(neumann_front(0.1, 1.25, 0.95, 2.0))
for M0 in [100, 400, 1600, 6400]:
    x, T, f, nl, tot = solve_kinetic_count(128, 1.25, 2.0, float(M0), 0.1, 0.01)
    fr = float(front_from_T(x, T, from_f=f)) if f.max() >= 0.5 else 0.0
    Ti = float(np.interp(fr, x, T)) if fr > 0 else float(T[0])
    rows.append(dict(M0=M0, front=fr, rel_to_neumann=fr / nf, T_interface=Ti))
    print(rows[-1])
json.dump(dict(neumann=nf, rows=rows), open(OUT, 'w'), indent=1)
''')

**`v2/make_vdd.py`** — de Vahl Davis single-phase check (Sec. VI A).

In [ ]:
writefile('v2/make_vdd.py', r'''"""Single-phase check (Sec. VI A): de Vahl Davis cavity, Ra = 1e5, Pr = 0.71, no drag (C_m = 0), latent heat switched off."""
import sys, os, json, numpy as np
sys.path.insert(0, '/home/claude/pof/v2')
from solver2d_v2 import run
OUT = os.environ.get('RECON_OUT', '/home/claude/pof/vdd.json')
res = {}
for nc in [48, 96]:
    r = run(nc=nc, Ra=1e5, Pr=0.71, Ste=1e12, Cm=0.0, theta_i=0.5, theta_c=0.0, t_end=0.6, out_times=[0.5, 0.6])
    res[str(nc)] = dict(Nu=float(r['snaps'][-1]['Nu']), Nu_prev=float(r['snaps'][-2]['Nu'])); print(nc, res[str(nc)], flush=True)
json.dump(res, open(OUT, 'w'), indent=1)
''')

**`v2/baseline_floors.py`** — Interpolation baselines and residual floors (Table III).

In [ ]:
writefile('v2/baseline_floors.py', r'''"""(1) Parameter-space interpolation baselines; (2) residual floors (Table IV) for
endpoint vs trapezoidal macro-step quadrature at two macro-step sizes."""
import json, sys
import numpy as np
from scipy.interpolate import LinearNDInterpolator, NearestNDInterpolator
sys.path.insert(0, '/home/claude/pof/v2')
import jax.numpy as jnp
import train2 as T2

out = {}
def front_pos(prof, dx):
    prof = prof - 0.5; s = np.sign(prof); idx = np.where(np.diff(s) < 0)[0]
    if len(idx) == 0: return 0.0 if prof[0] < 0 else 1.0
    i = idx[0]; x0 = (i + .5) * dx
    return x0 + dx * prof[i] / (prof[i] - prof[i + 1])

def interp_baseline(d, ntr, kin, w=None):
    P = np.stack([d['train_th'][:ntr], d['train_L'][:ntr]], 1)
    Q = np.stack([d['test_th'], d['test_L']], 1)
    def predict(field):
        Y = field[:ntr].reshape(ntr, -1)
        lin = LinearNDInterpolator(P, Y)(Q); near = NearestNDInterpolator(P, Y)(Q)
        bad = np.isnan(lin).any(1); lin[bad] = near[bad]
        # inverse-distance (p=2) on normalized parameters
        sc = P.std(0); Dm = np.linalg.norm((Q[:, None] - P[None]) / sc, axis=2) + 1e-12
        W = 1 / Dm ** 2; W /= W.sum(1, keepdims=True); idw = W @ Y
        return lin.reshape((-1,) + field.shape[1:]), idw.reshape((-1,) + field.shape[1:]), int(bad.sum())
    N = int(d['N']); dx = 1.0 / N
    res = {}
    TL, TI, nout = predict(d['train_T'])
    if kin: FL, FI, _ = predict(d['train_f'])
    for name, Tp, Fp in [('linear', TL, FL if kin else None), ('idw', TI, FI if kin else None)]:
        m = dict(relT=[], front=[], relf=[])
        for j in range(Q.shape[0]):
            Tr = d['test_T'][j]
            m['relT'].append(float(np.linalg.norm(Tp[j, 1:] - Tr[1:]) / np.linalg.norm(Tr[1:])))
            if kin:
                Fr = d['test_f'][j]
                m['relf'].append(float(np.linalg.norm(Fp[j, 1:] - Fr[1:]) / np.linalg.norm(Fr[1:])))
                fp, fr = Fp[j], Fr
            else:
                fp = 0.5 * (1 + np.tanh((Tp[j] - 1) / w)); fr = 0.5 * (1 + np.tanh((Tr - 1) / w))
            m['front'].append(float(np.mean([abs(front_pos(fp[k], dx) - front_pos(fr[k], dx)) for k in range(1, Tr.shape[0])])))
        res[name] = dict(per_traj=m, mean={k: (float(np.mean(v)) if v else None) for k, v in m.items()})
    res['n_test_outside_hull'] = nout
    return res

deq = np.load('/home/claude/pof/data_eq.npz'); dkin = np.load('/home/claude/pof/data_kin.npz')
for ntr in [12, 60]:
    out[f'eq_interp_ntr{ntr}'] = interp_baseline(deq, ntr, False, w=float(deq['w']))
for ntr in [12, 48]:
    out[f'kin_interp_ntr{ntr}'] = interp_baseline(dkin, ntr, True)
for k, v in out.items():
    print(k, {m: v[m]['mean'] for m in ('linear', 'idw')}, 'outside hull:', v['n_test_outside_hull'])

# ---------------------------------------------------------------- floors
floors = {}
def pairs(d, kin, ntr=12):
    return T2.make_pairs(d['train_T'][:ntr], d['train_th'][:ntr], d['train_L'][:ntr], d['train_t'][:ntr],
                         d['train_f'][:ntr] if kin else None)
tab = T2.make_sf_table(float(deq['w']))
dhalf = np.load('/home/claude/pof/v2/data_kin_half.npz')
for resid in ['end', 'trap']:
    T2.RESID = resid
    tr = pairs(deq, False)
    rE, rS = T2.resid_eq(tr[0], tr[1], tr[2], tr[3], tr[4], 1 / 128, float(deq['w']), tab)
    floors[f'eq_{resid}'] = dict(E0=float(jnp.sqrt(jnp.mean(rE ** 2))), S0=float(jnp.sqrt(jnp.mean((rE / tr[1]) ** 2))),
                                 rS_rms=float(jnp.sqrt(jnp.mean(rS ** 2))),
                                 ident=float(jnp.sqrt(jnp.mean((rS - rE / tr[1]) ** 2)) / jnp.sqrt(jnp.mean((rE / tr[1]) ** 2))),
                                 rate_scale=float(jnp.sqrt(jnp.mean(((T2.h_eq(tr[1], tr[4][:, None], float(deq['w'])) - T2.h_eq(tr[0], tr[4][:, None], float(deq['w']))) / tr[2][:, None]) ** 2))))
    for nm, dd in [('kin', dkin), ('kin_half', dhalf)]:
        tr = pairs(dd, True)
        rE, rf, sph = T2.resid_kin(tr[0], tr[5], tr[1], tr[6], tr[2], tr[3], tr[4], 1 / 128, float(dd['M0']))
        rate = jnp.sqrt(jnp.mean(((tr[6] - tr[5]) / tr[2][:, None]) ** 2))
        floors[f'{nm}_{resid}'] = dict(E0=float(jnp.sqrt(jnp.mean(rE ** 2))), F0=float(jnp.sqrt(jnp.mean(rf ** 2))),
                                       SIG0=float(jnp.sqrt(jnp.mean(sph ** 2))),
                                       F0_rel=float(jnp.sqrt(jnp.mean(rf ** 2)) / rate),
                                       neg_frac=float(jnp.mean(sph < -1e-12)))
T2.RESID = 'end'
for k, v in floors.items(): print(k, {a: round(b, 5) for a, b in v.items()})
out['floors'] = floors
json.dump(out, open('/home/claude/pof/v2/baseline_floors.json', 'w'), indent=1)
''')

**`v2/make_floors_skip.py`** — Floors excluding the initial layer (Table III, Fig. 2c).

In [ ]:
writefile('v2/make_floors_skip.py', r'''"""Residual floors on exact training pairs (12 trajectories) with the first k macro-steps excluded (Table III, Fig. 2c)."""
import sys, os, json, numpy as np
sys.path.insert(0, '/home/claude/pof/v2')
import jax.numpy as jnp
import train2 as T2
OUT = os.environ.get('RECON_OUT', '/home/claude/pof/v2/floors_skip.json')
deq = np.load('/home/claude/pof/data_eq.npz'); dkin = np.load('/home/claude/pof/data_kin.npz')
dhalf = np.load('/home/claude/pof/v2/data_kin_half.npz')
w = float(deq['w']); tab = T2.make_sf_table(w); ntr = 12
def sl(a, k): return a[:ntr, k:]
res = {}
for resid in ['end', 'trap']:
    T2.RESID = resid
    for skip in [0, 1, 2]:
        d = deq; T = d['train_T'][:ntr]; t = d['train_t'][:ntr]; th = d['train_th'][:ntr]; L = d['train_L'][:ntr]
        n = T.shape[1] - 1
        T0 = T[:, skip:n].reshape(-1, T.shape[2]); T1 = T[:, skip + 1:].reshape(-1, T.shape[2])
        dt = np.diff(t, axis=1)[:, skip:].reshape(-1); TH = np.repeat(th, n - skip); LL = np.repeat(L, n - skip)
        T0, T1, dt, TH, LL = map(jnp.array, (T0, T1, dt, TH, LL))
        rE, rS = T2.resid_eq(T0, T1, dt, TH, LL, 1 / 128, w, tab)
        rate = (T2.h_eq(T1, LL[:, None], w) - T2.h_eq(T0, LL[:, None], w)) / dt[:, None]
        res[f'eq_{resid}_skip{skip}'] = dict(E0_rel=float(jnp.sqrt(jnp.mean(rE ** 2)) / jnp.sqrt(jnp.mean(rate ** 2))),
                                            ident=float(jnp.sqrt(jnp.mean((rS - rE / T1) ** 2)) / jnp.sqrt(jnp.mean((rE / T1) ** 2))))
        for nm, d in [('kin', dkin), ('kin_half', dhalf)]:
            T = d['train_T'][:ntr]; F = d['train_f'][:ntr]; t = d['train_t'][:ntr]; th = d['train_th'][:ntr]; L = d['train_L'][:ntr]
            n = T.shape[1] - 1; N = T.shape[2]
            T0 = T[:, skip:n].reshape(-1, N); T1 = T[:, skip + 1:].reshape(-1, N)
            F0 = F[:, skip:n].reshape(-1, N); F1 = F[:, skip + 1:].reshape(-1, N)
            dt = np.diff(t, axis=1)[:, skip:].reshape(-1); TH = np.repeat(th, n - skip); LL = np.repeat(L, n - skip)
            T0, T1, F0, F1, dt, TH, LL = map(jnp.array, (T0, T1, F0, F1, dt, TH, LL))
            rE, rf, sph = T2.resid_kin(T0, F0, T1, F1, dt, TH, LL, 1 / 128, float(d['M0']))
            srate = (T1 - T0) / dt[:, None] + LL[:, None] * (F1 - F0) / dt[:, None]
            frate = (F1 - F0) / dt[:, None]
            res[f'{nm}_{resid}_skip{skip}'] = dict(E0_rel=float(jnp.sqrt(jnp.mean(rE ** 2)) / jnp.sqrt(jnp.mean(srate ** 2))),
                                                  F0_rel=float(jnp.sqrt(jnp.mean(rf ** 2)) / jnp.sqrt(jnp.mean(frate ** 2))),
                                                  neg=float(jnp.mean(sph < -1e-12)))
T2.RESID = 'end'
json.dump(res, open(OUT, 'w'), indent=1)
ref = json.load(open('/home/claude/pof/v2/floors_skip.json'))
for k in ref:
    print(k, {a: (round(res[k][a], 4), round(ref[k][a], 4)) for a in ref[k]} if k in res else 'MISSING')
''')

**`v2/corruption_v2.py`** — Corrupted-field diagnostics (Sec. VI B).

In [ ]:
writefile('v2/corruption_v2.py', r'''"""Corruption audit: single-state vs consistent two-state Gaussian smoothing of the
t=0.25 solver pair (96^2). Reports r_E, r_S, identity defect, apparent global production."""
import sys, json, pickle, numpy as np
sys.path.insert(0, '/home/claude/pof'); sys.path.insert(0, '/home/claude/pof/v2')
from resid2d import residuals_pair
BASE = dict(ws=0.02, Ste=0.2, theta_c=-0.1)
from solver2d_v2 import f_eq, DTH
from scipy.ndimage import gaussian_filter
snaps = pickle.load(open('/home/claude/pof/v2/snaps96.pkl', 'rb'))
s = [sn for sn in snaps if 'next' in sn][-1]
h = 1.0 / 96; ws, Ste, thc = BASE['ws'], BASE['Ste'], BASE['theta_c']; L_eff = DTH / Ste
Tg = np.linspace(0.5, 1.6, 400001)
fg = 0.5 / (ws * DTH) / np.cosh((Tg - 1.0) / (ws * DTH)) ** 2
integ = fg / Tg
G = np.concatenate([[0.0], np.cumsum(0.5 * (integ[1:] + integ[:-1]) * np.diff(Tg))]); G -= np.interp(1.0, Tg, G)
s_of = lambda T: np.log(T) + L_eff * np.interp(T, Tg, G)
def global_prod(th0, th1, dt):
    T0, T1 = 1 + DTH * th0, 1 + DTH * th1
    dS = (s_of(T1).sum() - s_of(T0).sum()) * h * h / dt
    gl = 2 * (1 + DTH) - T1[0]; gr = 2 * (1 + DTH * thc) - T1[-1]
    ql = -(T1[0] - gl) / h; qr = -(gr - T1[-1]) / h
    bflux = (-ql / (1 + DTH)).sum() * h + (qr / (1 + DTH * thc)).sum() * h
    return dS + bflux
rows = []
_, _, _, term0 = residuals_pair(s, L_eff, Ste, ws, h, thc)   # uncorrupted storage-term scale
for mode in ['single', 'both']:
    for sig in [0.0, 1.0, 2.0, 4.0]:
        sm = lambda a: gaussian_filter(a, sigma=sig) if sig > 0 else a
        s2 = dict(s); nx = dict(s['next'])
        th1 = sm(nx['th']); nx['th'] = th1; nx['Ht'] = th1 + f_eq(th1, ws) / Ste
        if mode == 'both':
            th0 = sm(s['th']); s2['th'] = th0; s2['Ht'] = th0 + f_eq(th0, ws) / Ste
        s2['next'] = nx
        rE, rS, T1, term = residuals_pair(s2, L_eff, Ste, ws, h, thc)
        P = global_prod(s2['th'], nx['th'], nx['dt'])
        rows.append(dict(mode=mode, sigma=sig, rE_over_term=float(np.linalg.norm(rE) / term0),
                         ratio=float(np.linalg.norm(rS) / np.linalg.norm(rE / T1)),
                         ident=float(np.linalg.norm(rS - rE / T1) / np.linalg.norm(rE / T1)),
                         global_prod=float(P)))
        print(rows[-1])
json.dump(rows, open('/home/claude/pof/v2/corruption_v2.json', 'w'), indent=1)
''')

**`v2/make_dissipation.py`** — Mechanical heating ratios (Appendix F).

In [ ]:
writefile('v2/make_dissipation.py', r'''"""Mechanical heating ratios (Appendix F) from the integrals stored in run2d_v2.json (t = 0.25)."""
import os, json
OUT = os.environ.get('RECON_OUT', '/home/claude/pof/v2/dissipation.json')
R = json.load(open('/home/claude/pof/v2/run2d_v2.json'))['runs']
Ec, T0dT = 5e-17, 30.0          # (alpha/H)^2/(c_p dT) with alpha=1e-7, H=0.1, c_p=2e3, dT=10 K; T0/dT = 300/10
rows = {}
for k in ['up48', 'up96', 'mu96', 'ws0.01', 'eps0.0001', 'eps0.01']:
    r = R[k]
    rows[k] = dict(visc_int=r['visc_int'][-1], darcy_int=r['darcy_int'][-1], Nu=r['Nu'][-1],
                   ratio_visc_heat=Ec * r['visc_int'][-1] / r['Nu'][-1], ratio_darcy_heat=Ec * r['darcy_int'][-1] / r['Nu'][-1],
                   ratio_visc_ent=Ec * T0dT * r['visc_ent_int'][-1] / r['cond_ent_int'][-1],
                   ratio_darcy_ent=Ec * T0dT * r['darcy_ent_int'][-1] / r['cond_ent_int'][-1],
                   usolid=r['usolid'], uliq=r['uliq'])
json.dump(dict(Ec_alpha=Ec, T0_over_dT=T0dT, rows=rows), open(OUT, 'w'), indent=1)
ref = json.load(open('/home/claude/pof/v2/dissipation.json'))['rows']
print('max rel diff vs stored:', max(abs(rows[k][a] - ref[k][a]) / (abs(ref[k][a]) + 1e-300) for k in ref for a in ref[k]))
''')

**`v2/discrete_gradient_check.py`** — Discrete-gradient pair on manufactured pairs.

In [ ]:
writefile('v2/discrete_gradient_check.py', r'''"""Independent check of a discrete-gradient (secant) energy-entropy residual pair for conduction:
r_S,i = a_i r_E,i exactly, with non-negative face production (cf. Gonzalez 1996)."""
import json, numpy as np
from numpy.polynomial.legendre import leggauss
w = 0.03; L = 2.0
h = lambda T: T + L * 0.5 * (1 + np.tanh((T - 1) / w))
dh = lambda T: 1 + L * 0.5 / (w * np.cosh((T - 1) / w) ** 2)
def ds(Ta, Tb, n=128):            # s(Tb)-s(Ta) = int dh/T
    x, wt = leggauss(n); m = 0.5 * (Ta + Tb); r = 0.5 * (Tb - Ta)
    T = m[..., None] + r[..., None] * x
    return r * np.sum(wt * dh(T) / T, axis=-1)
rows = []
for N in [32, 64, 128, 256]:
    dx = 1 / N; x = (np.arange(N) + 0.5) * dx; dt = 0.01; G = 1 / dx
    Tm = 1.02 + 0.12 * np.sin(2 * np.pi * x); Tp = Tm + 0.015 * np.cos(2 * np.pi * x + 0.2)
    dH = h(Tp) - h(Tm); dS = ds(Tm, Tp); a = dS / dH; Td = 1 / a
    j = np.roll(np.arange(N), -1)                  # periodic right neighbour
    Q = G * (Td - Td[j]); J = 0.5 * (a + a[j]) * Q; prod = Q * (a[j] - a)   # face i -> i+1
    divQ = (Q - np.roll(Q, 1)) / dx; divJ = (J - np.roll(J, 1)) / dx
    Pi = 0.5 * (prod + np.roll(prod, 1)) / dx
    rE = dH / dt + divQ; rS = dS / dt + divJ - Pi
    rem = np.abs(rS - a * rE).max() / np.abs(rS).max()
    rows.append(dict(N=N, remainder=float(rem), min_face_production=float(prod.min()),
                     secant_between=bool(np.all((Td - np.minimum(Tm, Tp)) >= -1e-12) and np.all((np.maximum(Tm, Tp) - Td) >= -1e-12))))
    print(rows[-1])
json.dump(rows, open('discrete_gradient_check.json', 'w'), indent=1)
''')

**`v2/stats_v2.py`** — Seed-level inference for the primary and secondary contrasts (Table IV).

In [ ]:
writefile('v2/stats_v2.py', r'''"""Seed-level statistical reanalysis per PREREGISTRATION.md."""
import json, glob, itertools
import numpy as np
from scipy import stats

rng = np.random.default_rng(12345)

def load(pattern):
    runs = {}
    for f in glob.glob(pattern):
        d = json.load(open(f)); runs[d['seed']] = d
    return [runs[s] for s in sorted(runs)]

def matrix(runs, metric):
    return np.array([r['per_traj'][metric] for r in runs])        # seeds x trajectories

def analyse(A, B, metric, margin=None, nboot=10000):
    a, b = matrix(A, metric), matrix(B, metric)
    assert a.shape == b.shape, (a.shape, b.shape)
    D = a - b; S, J = D.shape
    ms = D.mean(1); mean = ms.mean(); sd = ms.std(ddof=1); se = sd / np.sqrt(S)
    tq = stats.t.ppf(0.975, S - 1); tq90 = stats.t.ppf(0.95, S - 1)
    ci95 = (mean - tq * se, mean + tq * se); ci90 = (mean - tq90 * se, mean + tq90 * se)
    # crossed bootstrap
    bs = np.empty(nboot)
    for k in range(nboot):
        si = rng.integers(0, S, S); ti = rng.integers(0, J, J)
        bs[k] = D[np.ix_(si, ti)].mean()
    cib = (np.percentile(bs, 2.5), np.percentile(bs, 97.5))
    # exact sign-flip permutation over seed means
    flips = np.array(list(itertools.product([-1, 1], repeat=S)))
    null = (flips * ms).mean(1)
    p_perm = float(np.mean(np.abs(null) >= abs(mean) - 1e-15))
    # two-way random-effects variance components (no replication)
    rm, cm = D.mean(1), D.mean(0)
    MSs = J * np.sum((rm - D.mean()) ** 2) / (S - 1)
    MSt = S * np.sum((cm - D.mean()) ** 2) / (J - 1)
    res = D - rm[:, None] - cm[None, :] + D.mean()
    MSe = np.sum(res ** 2) / ((S - 1) * (J - 1))
    vc = dict(seed=max((MSs - MSe) / J, 0.0), traj=max((MSt - MSe) / S, 0.0), resid=MSe)
    mde = (stats.t.ppf(0.975, S - 1) + stats.t.ppf(0.80, S - 1)) * sd / np.sqrt(S)
    base = b.mean()
    out = dict(metric=metric, n_seeds=S, n_traj=J, mean=mean, rel=mean / base, base_mean=base,
               seed_sd=sd, ci95=ci95, ci90=ci90, ci95_boot=cib, p_perm=p_perm, mde=mde,
               mde_rel=mde / base, var_components=vc, seed_means=ms.tolist())
    if margin is not None:
        lo, hi = -margin, margin
        t1 = (mean - lo) / se; t2 = (hi - mean) / se
        p_tost = max(1 - stats.t.cdf(t1, S - 1), 1 - stats.t.cdf(t2, S - 1))
        out.update(margin=margin, p_tost=float(p_tost),
                   equivalent=bool(ci90[0] > lo and ci90[1] < hi),
                   equivalent_boot=bool(np.percentile(bs, 5) > lo and np.percentile(bs, 95) < hi))
    return out

def fmt(o):
    s = (f"{o['metric']:>8s}: diff={o['mean']:+.3e} ({100*o['rel']:+.1f}%) CI95=[{o['ci95'][0]:+.2e},{o['ci95'][1]:+.2e}] "
         f"boot=[{o['ci95_boot'][0]:+.2e},{o['ci95_boot'][1]:+.2e}] p_perm={o['p_perm']:.4f} MDE={o['mde']:.2e} ({100*o['mde_rel']:.1f}%)")
    if 'margin' in o: s += f" | margin={o['margin']:.2e} p_TOST={o['p_tost']:.4f} equiv={o['equivalent']}"
    return s

R = {}
V1 = '/home/claude/pof/runs'; V2 = '/home/claude/pof/v2/runs'
eq = {v: load(f'{V1}/eq_{v}_s[0-3].json') + load(f'{V2}/eq_{v}_s[4-7].json') for v in ['D', 'E', 'ES', 'EET']}
kn = {v: load(f'{V2}/kin_{v}_s[0-7].json') for v in ['D', 'E', 'EPH', 'EK']}
print('seeds:', {k: len(v) for k, v in eq.items()}, {k: len(v) for k, v in kn.items()})

Emean_relT_eq = matrix(eq['E'], 'relT').mean()
prim = {}
prim['P1'] = analyse(eq['ES'], eq['EET'], 'relT', margin=0.10 * Emean_relT_eq)
prim['P2'] = analyse(eq['ES'], eq['EET'], 'front', margin=0.1 / 128)
prim['P3'] = analyse(kn['EK'], kn['E'], 'relf')
prim['P4'] = analyse(kn['EK'], kn['E'], 'relT')
# Holm over the four primary p-values (TOST p for P1,P2; permutation p for P3,P4)
pv = {'P1': prim['P1']['p_tost'], 'P2': prim['P2']['p_tost'], 'P3': prim['P3']['p_perm'], 'P4': prim['P4']['p_perm']}
order = sorted(pv, key=pv.get); m = len(order); adj = {}; running = 0
for i, k in enumerate(order):
    running = max(running, min(1.0, (m - i) * pv[k])); adj[k] = running
for k in prim: prim[k]['p_holm'] = adj[k]
R['primary'] = prim
print('PRIMARY'); [print(k, fmt(v), f"p_holm={v['p_holm']:.4f}") for k, v in prim.items()]

sec = {}
for (A, B) in [('E', 'D'), ('ES', 'E'), ('EET', 'E'), ('ES', 'EET')]:
    for mt in ['relT', 'front', 'rE', 'dE']:
        sec[f'eq_{A}-{B}_{mt}'] = analyse(eq[A], eq[B], mt)
for (A, B) in [('E', 'D'), ('EK', 'E'), ('EPH', 'E'), ('EK', 'EPH'), ('EK', 'D')]:
    for mt in ['relT', 'relf', 'front', 'rE', 'rf', 'sphneg', 'Pm_001', 'Pm_01', 'Dminus']:
        sec[f'kin_{A}-{B}_{mt}'] = analyse(kn[A], kn[B], mt)
R['secondary'] = sec
print('SECONDARY'); [print(k, fmt(v)) for k, v in sec.items()]

# descriptive means
R['means_eq'] = {v: {mt: float(matrix(eq[v], mt).mean()) for mt in ['relT', 'front', 'rE', 'dE', 'rS_minus']} for v in eq}
R['means_kin'] = {v: {mt: float(matrix(kn[v], mt).mean()) for mt in
                      ['relT', 'relf', 'front', 'rE', 'rf', 'sphneg', 'Pm_001', 'Pm_01', 'Pm_1', 'Dminus',
                       'neg_near', 'neg_mid', 'neg_far', 'fout', 'ref_Pm_0', 'ref_Pm_001', 'ref_Dminus', 'ref_neg_near']} for v in kn}
R['seed_sd_eq'] = {v: {mt: float(matrix(eq[v], mt).mean(1).std(ddof=1)) for mt in ['relT', 'front']} for v in eq}

# loss-component gradient diagnostics (v2 runs only)
def gd(runs):
    o = {}
    for r in runs:
        for when in ['diag_init', 'diag_final']:
            if when not in r: continue
            for k, v in r[when].items():
                o.setdefault(f'{when}:{k}', []).append(v['gradnorm'])
    return {k: (float(np.mean(v)), float(np.std(v))) for k, v in o.items()}
R['grad_diag_eq'] = {v: gd([r for r in eq[v] if 'diag_init' in r]) for v in eq}
R['grad_diag_kin'] = {v: gd(kn[v]) for v in kn}
json.dump(R, open('/home/claude/pof/v2/stats_v2.json', 'w'), indent=1, default=float)
print('saved stats_v2.json')
''')

**`v2/stats_round3.py`** — Holm-level TOST and bootstrap intervals; heating mappings.

In [ ]:
writefile('v2/stats_round3.py', r'''"""Round-3 statistics: Holm-adjusted TOST intervals for P1/P2 and heating-ratio mappings.
Reproduces stats_round3.json (bootstrap intervals vary in the last digit with the RNG stream)."""
import json, numpy as np
from scipy import stats
import importlib.util, sys, io, contextlib
spec = importlib.util.spec_from_file_location('s2', '/home/claude/pof/v2/stats_v2.py')
s2 = importlib.util.module_from_spec(spec)
with contextlib.redirect_stdout(io.StringIO()): spec.loader.exec_module(s2)   # loads runs, primary p-values
prim = s2.R['primary']
pv = {'P1': prim['P1']['p_tost'], 'P2': prim['P2']['p_tost'], 'P3': prim['P3']['p_perm'], 'P4': prim['P4']['p_perm']}
order = sorted(pv, key=pv.get); alphas = {k: 0.05 / (len(order) - i) for i, k in enumerate(order)}
rng = np.random.default_rng(2026)
out = {'equivalence_adjusted': {}, 'holm_alphas': alphas}
for k, metric, margin in [('P1', 'relT', prim['P1']['margin']), ('P2', 'front', prim['P2']['margin'])]:
    D = s2.matrix(s2.eq['ES'], metric) - s2.matrix(s2.eq['EET'], metric); S, J = D.shape
    ms = D.mean(1); m = ms.mean(); se = ms.std(ddof=1) / np.sqrt(S); a = alphas[k]
    q = stats.t.ppf(1 - a, S - 1); ci = [m - q * se, m + q * se]
    bs = np.array([D[np.ix_(rng.integers(0, S, S), rng.integers(0, J, J))].mean() for _ in range(10000)])
    cib = [np.percentile(bs, 100 * a), np.percentile(bs, 100 * (1 - a))]
    out['equivalence_adjusted'][k] = dict(alpha_holm=a, ci_adjusted=ci, ci_adjusted_boot=cib, margin=margin,
        inside=bool(-margin < ci[0] and ci[1] < margin), inside_boot=bool(-margin < cib[0] and cib[1] < margin))
d = json.load(open('/home/claude/pof/v2/dissipation.json'))['rows']['up96']
alpha, H, cp, Tm = 1e-7, 0.1, 2e3, 300.0
def mapping(dT, T0):
    Ec = (alpha / H) ** 2 / (cp * dT); f = Ec / 5e-17
    return dict(Ec=Ec, visc_heat=d['ratio_visc_heat'] * f, darcy_heat=d['ratio_darcy_heat'] * f,
                visc_ent=d['ratio_visc_ent'] * f * (T0 / dT) / 30.0, darcy_ent=d['ratio_darcy_ent'] * f * (T0 / dT) / 30.0)
out['model_mapping_delta0.2'] = mapping(0.2 * Tm, Tm)
out['illustrative_10K_300K'] = mapping(10.0, 300.0)
json.dump(out, open('/home/claude/pof/v2/stats_round3.json', 'w'), indent=1)
print(json.dumps(out, indent=1))
''')

**`v2/stats_round4.py`** — Plan-time margin, post-plan seeds, exported paired arrays.

In [ ]:
writefile('v2/stats_round4.py', r'''"""Round-4 statistics: (i) primary equivalence with the plan-time margin (primary) and the final margin
(sensitivity); (ii) confirmatory re-analysis restricted to seeds generated after the analysis plan
(equilibrium seeds 4-7, internal-variable seeds 3-7); (iii) export of per-run and paired per-trajectory arrays."""
import json, io, contextlib, importlib.util, itertools, csv, os
import numpy as np
from scipy import stats
spec = importlib.util.spec_from_file_location('s2', '/home/claude/pof/v2/stats_v2.py'); s2 = importlib.util.module_from_spec(spec)
with contextlib.redirect_stdout(io.StringIO()): spec.loader.exec_module(s2)
eq, kn, matrix = s2.eq, s2.kn, s2.matrix
PLAN_MARGIN_T = 3.21e-4; FINAL_MARGIN_T = 0.10 * matrix(eq['E'], 'relT').mean(); MARGIN_F = 0.1 / 128
HOLM = {'P1': 0.0125, 'P2': 0.05 / 3, 'P4': 0.025, 'P3': 0.05}

def tost(D, margin, alpha):
    ms = D.mean(1); S = len(ms); m = ms.mean(); se = ms.std(ddof=1) / np.sqrt(S)
    q = stats.t.ppf(1 - alpha, S - 1)
    p = max(1 - stats.t.cdf((m + margin) / se, S - 1), 1 - stats.t.cdf((margin - m) / se, S - 1))
    return dict(n_seeds=S, mean=m, ci=[m - q * se, m + q * se], margin=margin, p_tost=float(p),
                equivalent=bool(m - q * se > -margin and m + q * se < margin))
def signflip(D):
    ms = D.mean(1); fl = np.array(list(itertools.product([-1, 1], repeat=len(ms))))
    return float(np.mean(np.abs((fl * ms).mean(1)) >= abs(ms.mean()) - 1e-15))
out = {}
Dt = matrix(eq['ES'], 'relT') - matrix(eq['EET'], 'relT'); Df = matrix(eq['ES'], 'front') - matrix(eq['EET'], 'front')
out['P1_plan_margin_primary'] = tost(Dt, PLAN_MARGIN_T, HOLM['P1'])
out['P1_final_margin_sensitivity'] = tost(Dt, FINAL_MARGIN_T, HOLM['P1'])
out['P2'] = tost(Df, MARGIN_F, HOLM['P2'])
# confirmatory subset: seeds generated after the plan
idx_eq = [i for i, r in enumerate(eq['ES']) if r['seed'] >= 4]
out['P1_new_seeds_only'] = tost(Dt[idx_eq], PLAN_MARGIN_T, HOLM['P1'])
out['P2_new_seeds_only'] = tost(Df[idx_eq], MARGIN_F, HOLM['P2'])
idx_kn = [i for i, r in enumerate(kn['EK']) if r['seed'] >= 3]
for nm, met in (('P3', 'relf'), ('P4', 'relT')):
    D = (matrix(kn['EK'], met) - matrix(kn['E'], met))[idx_kn]; ms = D.mean(1); base = matrix(kn['E'], met)[idx_kn].mean()
    se = ms.std(ddof=1) / np.sqrt(len(ms)); q = stats.t.ppf(0.975, len(ms) - 1)
    out[f'{nm}_new_seeds_only'] = dict(n_seeds=len(ms), rel=float(ms.mean() / base),
                                       ci95_rel=[float((ms.mean() - q * se) / base), float((ms.mean() + q * se) / base)],
                                       p_signflip=signflip(D))
# exports
os.makedirs('/home/claude/pof/v2/stats_export', exist_ok=True)
for clos, R in (('eq', eq), ('kin', kn)):
    with open(f'/home/claude/pof/v2/stats_export/{clos}_per_run_means.csv', 'w', newline='') as fh:
        w = csv.writer(fh); mets = list(R['E'][0]['per_traj'].keys())
        w.writerow(['variant', 'seed'] + mets)
        for v, runs in R.items():
            for r in runs: w.writerow([v, r['seed']] + [np.mean(r['per_traj'][m]) if len(r['per_traj'][m]) else '' for m in mets])
    np.savez_compressed(f'/home/claude/pof/v2/stats_export/{clos}_per_trajectory.npz',
                        **{f'{v}__{m}': matrix(runs, m) for v, runs in R.items() for m in runs[0]['per_traj'] if len(runs[0]['per_traj'][m])},
                        seeds=np.array([r['seed'] for r in R['E']]))
json.dump(out, open('/home/claude/pof/v2/stats_round4.json', 'w'), indent=1, default=float)
print(json.dumps(out, indent=1, default=float))
''')

**`v2/stats_round5.py`** — Holm family for the post-plan seeds (Appendix D).

In [ ]:
writefile('v2/stats_round5.py', r'''"""Round-5: Holm family for the post-plan seeds only (eq seeds 4-7, kin seeds 3-7), with Holm-level TOST intervals."""
import json, io, contextlib, importlib.util, itertools, numpy as np
from scipy import stats
spec = importlib.util.spec_from_file_location('s2', '/home/claude/pof/v2/stats_v2.py'); s2 = importlib.util.module_from_spec(spec)
with contextlib.redirect_stdout(io.StringIO()): spec.loader.exec_module(s2)
eq, kn, M = s2.eq, s2.kn, s2.matrix
ie = [i for i, r in enumerate(eq['ES']) if r['seed'] >= 4]; ik = [i for i, r in enumerate(kn['EK']) if r['seed'] >= 3]
D = {'P1': (M(eq['ES'], 'relT') - M(eq['EET'], 'relT'))[ie], 'P2': (M(eq['ES'], 'front') - M(eq['EET'], 'front'))[ie],
     'P3': (M(kn['EK'], 'relf') - M(kn['E'], 'relf'))[ik], 'P4': (M(kn['EK'], 'relT') - M(kn['E'], 'relT'))[ik]}
margin = {'P1': 3.21e-4, 'P2': 0.1 / 128}
p = {}
for k, X in D.items():
    ms = X.mean(1); S = len(ms); m = ms.mean(); se = ms.std(ddof=1) / np.sqrt(S)
    if k in margin:
        mg = margin[k]; p[k] = max(1 - stats.t.cdf((m + mg) / se, S - 1), 1 - stats.t.cdf((mg - m) / se, S - 1))
    else:
        fl = np.array(list(itertools.product([-1, 1], repeat=S))); p[k] = float(np.mean(np.abs((fl * ms).mean(1)) >= abs(m) - 1e-15))
order = sorted(p, key=p.get); adj = {}; run = 0; alpha = {}
for i, k in enumerate(order):
    run = max(run, min(1.0, (len(order) - i) * p[k])); adj[k] = run; alpha[k] = 0.05 / (len(order) - i)
out = {}
for k in D:
    X = D[k]; ms = X.mean(1); S = len(ms); m = ms.mean(); se = ms.std(ddof=1) / np.sqrt(S)
    o = dict(n_seeds=S, p_raw=float(p[k]), p_holm=float(adj[k]), holm_alpha=alpha[k], order=order.index(k) + 1)
    if k in margin:
        q = stats.t.ppf(1 - alpha[k], S - 1); o.update(ci=[m - q * se, m + q * se], margin=margin[k],
                                                      equivalent=bool(m - q * se > -margin[k] and m + q * se < margin[k]))
    out[k] = o
json.dump(out, open('/home/claude/pof/v2/stats_round5.json', 'w'), indent=1, default=float); print(json.dumps(out, indent=1, default=float))
''')

**`v2/make_analysis2.py`** — Tuning sweeps, kinetic robustness, $\lambda=10^{-4}$ two-dimensional summaries (Fig. 5).

In [ ]:
writefile('v2/make_analysis2.py', r'''"""Tuning sweeps (Fig. 5b,c), kinetic robustness audit (Fig. 5a), and lambda=1e-4 2D summaries -> analysis2.json."""
import os, json, numpy as np
from scipy import stats
V = '/home/claude/pof/v2/'
OUT = os.environ.get('RECON_OUT', V + 'analysis2.json')
J = lambda p: json.load(open(V + p))
out = {}
# equal-search-budget tuning (seed 0); lambda = 3e-4 is the main seed-0 run
tun = {}
for clos, vs, extra in [('eq', ['E', 'ES', 'EET'], []), ('kin', ['E', 'EPH', 'EK'], ['relf', 'rf'])]:
    for v in vs:
        rows = []
        for lam, f in [(1e-4, f'runs/{clos}_{v}_s0_lam1e-4.json'), (3e-4, f'runs/{clos}_{v}_s0.json'), (1e-3, f'runs/{clos}_{v}_s0_lam1e-3.json')]:
            d = json.load(open(V + f)) if os.path.exists(V + f) else json.load(open('/home/claude/pof/runs/' + f.split('/')[-1]))
            m = d['metrics']
            rows.append(dict(lam=lam, val=d['val_loss'], relT=m['relT'], front=m['front'], rE=m['rE'], **{k: m[k] for k in extra}))
        tun[f'{clos}_{v}'] = dict(rows=rows, selected=min(rows, key=lambda r: r['val'])['lam'])
out['tuning'] = tun
# robustness of the kinetic penalty: E+K minus E, two seeds, per setting
aud = {}
for tag, suf in [('_trap', '_trap'), ('_half', '_half'), ('_M1600', '_M1600'), ('_end_s01', '')]:
    a = {}
    for m in ['relT', 'relf', 'front', 'rf', 'rE']:
        e = [J(f'runs/kin_E_s{s}{suf}.json')['metrics'][m] for s in (0, 1)]
        k = [J(f'runs/kin_EK_s{s}{suf}.json')['metrics'][m] for s in (0, 1)]
        diffs = [b - c for b, c in zip(k, e)]; base = float(np.mean(e))
        a[m] = dict(seed_diffs=diffs, mean=float(np.mean(diffs)), rel=float(np.mean(diffs) / base), base=base)
    aud[tag] = a
out['audit'] = aud
# two-dimensional ablation at lambda = 1e-4
R = {v: [J(f'runs2d/{v}_s{s}.json') for s in range(3)] for v in ['D', 'E', 'ES', 'EET']}
mets = ['relT', 'relU', 'front', 'nu', 'fbar', 'rE', 'ident']
out['2d_means'] = {v: {m: float(np.mean([r['metrics'][m] for r in R[v]])) for m in mets} for v in R}
out['2d_seedsd'] = {v: {m: float(np.std([r['metrics'][m] for r in R[v]], ddof=1)) for m in mets} for v in R}
C = {}
for A, B in [('ES', 'EET'), ('E', 'D'), ('ES', 'E'), ('EET', 'E')]:
    for m in mets:
        D = np.array([np.mean(np.array(a['per_traj'][m]) - np.array(b['per_traj'][m])) for a, b in zip(R[A], R[B])])
        q = stats.t.ppf(0.975, 2); se = D.std(ddof=1) / np.sqrt(3)
        C[f'{A}-{B}_{m}'] = dict(mean=float(D.mean()), rel=float(D.mean() / np.mean([b['metrics'][m] for b in R[B]])),
                                 ci=[float(D.mean() - q * se), float(D.mean() + q * se)], seed=D.tolist())
out['2d_contrasts'] = C
out['2d_scales'] = R['E'][0]['scales']
out['2d_diag'] = {v: [r['diag_init'] for r in R[v]] for v in R}
json.dump(out, open(OUT, 'w'), indent=1)
''')

**`v2/make_active2d.py`** — Active-weight two-dimensional summaries (Fig. 8e–h).

In [ ]:
writefile('v2/make_active2d.py', r'''"""Active-weight two-dimensional runs (lambda = 1e-2, 1e-1): table, gradient ratios, contrasts -> active2d.json."""
import os, json, numpy as np
from scipy import stats
V = '/home/claude/pof/v2/'
OUT = os.environ.get('RECON_OUT', V + 'active2d.json')
J = lambda p: json.load(open(V + p))
mets = ['relT', 'relU', 'front', 'nu', 'fbar', 'rE', 'ident']
R = {('D', '0'): [J(f'runs2d/D_s{s}.json') for s in range(3)]}
for v in ['E', 'ES', 'EET']:
    for l in ['1e-1', '1e-2']: R[(v, l)] = [J(f'runs2d_active/{v}_s{s}_lam{l}.json') for s in range(3)]
    R[(v, '1e-4')] = [J(f'runs2d/{v}_s{s}.json') for s in range(3)]
order = [('E', '1e-1'), ('ES', '1e-1'), ('EET', '1e-1'), ('E', '1e-2'), ('ES', '1e-2'), ('EET', '1e-2'), ('D', '0'),
         ('E', '1e-4'), ('ES', '1e-4'), ('EET', '1e-4')]
table = {f'{v}@{l}': dict(val=float(np.mean([r['val_loss'] for r in R[(v, l)]])),
                          **{m: float(np.mean([r['metrics'][m] for r in R[(v, l)]])) for m in mets}) for v, l in order}
terms = {'E': ['E'], 'ES': ['E', 'S'], 'EET': ['E', 'EET']}
gr = {}
for v in ['E', 'ES', 'EET']:
    for l in ['1e-1', '1e-2']:
        H = [r['grad_hist'] for r in R[(v, l)]]
        gr[f'{v}@{l}'] = dict(steps=[h['step'] for h in H[0]],
                              ratio=[float(np.mean([sum(H[s][i][t] for t in terms[v]) / H[s][i]['data'] for s in range(3)])) for i in range(len(H[0]))])
C = {}
for l in ['1e-1', '1e-2']:
    for A, B in [(('ES', l), ('EET', l)), (('E', l), ('D', '0')), (('ES', l), ('E', l)), (('EET', l), ('E', l))]:
        for m in ['relT', 'front', 'nu', 'rE']:
            D = np.array([np.mean(np.array(a['per_traj'][m]) - np.array(b['per_traj'][m])) for a, b in zip(R[A], R[B])])
            base = float(np.mean([np.mean(b['per_traj'][m]) for b in R[B]])); q = stats.t.ppf(0.975, 2); se = D.std(ddof=1) / np.sqrt(3)
            C[f'{A[0]}-{B[0]}@{l}_{m}'] = dict(mean=float(D.mean()), rel=float(D.mean() / base),
                                               ci_rel=[float((D.mean() - q * se) / base), float((D.mean() + q * se) / base)],
                                               seed=(D / base).tolist())
json.dump(dict(table=table, grad_ratio=gr, contrasts=C), open(OUT, 'w'), indent=1)
''')

**`v2/agg_r4.py`** — Corrected and projected two-dimensional runs, reference-shift audit (Fig. 7c,d).

In [ ]:
writefile('v2/agg_r4.py', r'''"""Round-4 aggregation: active-weight 2D table incl. continuity-corrected (ESC) and commonly projected (p1)
variants, seed-level contrasts, gauge (reference-state) sensitivity, 2D multiple-crossing frequency."""
import json, glob, sys, itertools, numpy as np
from scipy import stats
sys.path.insert(0, '/home/claude/pof/v2')
import jax.numpy as jnp
import train2d_r4 as T
V = '/home/claude/pof/v2/'
def load(f): return json.load(open(f))
R = {}
for s in range(3):
    R[('D', 0, '0', s)] = load(V + f'runs2d_active/D_s{s}_logged.json')
    for v in ['E', 'ES', 'EET']:
        R[(v, 0, '1e-4', s)] = load(V + f'runs2d/{v}_s{s}.json')
        for l in ['1e-2', '1e-1']:
            R[(v, 0, l, s)] = load(V + f'runs2d_active/{v}_s{s}_lam{l}.json')
for f in glob.glob(V + 'runs2d_r4/*.json'):
    d = load(f); nm = f.split('/')[-1][:-5]; v, p, sd, lm = nm.split('_')
    R[(v, int(p[1]), lm.replace('lam', ''), int(sd[1]))] = d
mets = ['relT', 'relU', 'front', 'nu', 'fbar', 'rE']
table = {}
for key in sorted({k[:3] for k in R}):
    runs = [R[key + (s,)] for s in range(3) if key + (s,) in R]
    table['%s|p%d|%s' % key] = dict(n=len(runs), val=float(np.mean([r['val_loss'] for r in runs])),
        **{m: float(np.mean([r['metrics'][m] for r in runs])) for m in mets},
        seeds={m: [r['metrics'][m] for r in runs] for m in mets})
def contrast(A, B, m):
    a = [R[A + (s,)] for s in range(3)]; b = [R[B + (s,)] for s in range(3)]
    D = np.array([np.mean(np.array(x['per_traj'][m]) - np.array(y['per_traj'][m])) for x, y in zip(a, b)])
    base = np.mean([np.mean(y['per_traj'][m]) for y in b]); se = D.std(ddof=1) / np.sqrt(3); q = stats.t.ppf(0.975, 2)
    fl = np.array(list(itertools.product([-1, 1], repeat=3)))
    p_sf = float(np.mean(np.abs((fl * D).mean(1)) >= abs(D.mean()) - 1e-15))
    return dict(rel=float(D.mean() / base), ci_rel=[float((D.mean() - q * se) / base), float((D.mean() + q * se) / base)],
                seed_rel=(D / base).tolist(), p_signflip_min_possible=0.25, p_signflip=p_sf)
C = {}
for l in ['1e-2', '1e-1']:
    pairs = {'ES-EET': (('ES', 0, l), ('EET', 0, l)), 'ESC-EET': (('ESC', 0, l), ('EET', 0, l)),
             'ES-ESC': (('ES', 0, l), ('ESC', 0, l)), 'ESC-E': (('ESC', 0, l), ('E', 0, l)),
             'ESp-EETp': (('ES', 1, l), ('EET', 1, l)), 'Ep-D': (('E', 1, l), ('D', 0, '0')),
             'ESp-Ep': (('ES', 1, l), ('E', 1, l)), 'Ep-E': (('E', 1, l), ('E', 0, l)), 'E-D': (('E', 0, l), ('D', 0, '0'))}
    for nm, (A, B) in pairs.items():
        if all(A + (s,) in R and B + (s,) in R for s in range(3)):
            for m in ['relT', 'nu', 'rE']: C[f'{nm}@{l}_{m}'] = contrast(A, B, m)
# gauge sensitivity on fixed fields (reference pairs with reconstructed faces; ESC predictions)
d = np.load(V + 'data_2d.npz'); N = 32; h = 1 / N; ws = 0.02; thc = -0.1; Gtab = T.make_G(ws)
def gauge(th, u, v, t, Ste, E0, S0):
    th0, th1 = jnp.array(th[:-1]), jnp.array(th[1:]); n = th.shape[0] - 1
    rE, rS, T1, s1, H1, rm = T.residuals(th0, th1, jnp.array(u[1:]), jnp.array(v[1:]), jnp.array(np.diff(t)),
                                         jnp.full((n,), Ste), ws, h, thc, Gtab, full=True)
    C1 = T.DTH / Ste                     # one nondimensional latent-heat unit
    LE = lambda c: float(jnp.mean(((rE + c * rm) / E0) ** 2)); LS = lambda c: float(jnp.mean(((rS + c * rm) / S0) ** 2))
    LET = lambda c: float(jnp.mean(((rE + c * rm) / T1 / S0) ** 2))
    return dict(C_unit=C1, E=[LE(-C1) / LE(0), LE(C1) / LE(0)], S=[LS(-C1) / LS(0), LS(C1) / LS(0)],
                EET=[LET(-C1) / LET(0), LET(C1) / LET(0)], rm_over_rE=float(jnp.sqrt(jnp.mean(rm ** 2)) / jnp.sqrt(jnp.mean(rE ** 2))))
sc = R[('ESC', 0, '1e-2', 0)]['scales']; G = {}
for j in range(6):
    G[f'ref_test{j}'] = gauge(d['test_th'][j], d['test_u'][j], d['test_v'][j], d['test_t'][j], float(d['test_Ste'][j]), sc['E0'], sc['S0'])
for f in sorted(glob.glob(V + 'ckpt2d_r4/ESC_p0_lam0.01_s*_rollout0.npz')):
    z = np.load(f); G[f.split('/')[-1]] = gauge(z['th'], z['u'], z['v'], d['test_t'][0], float(d['test_Ste'][0]), sc['E0'], sc['S0'])
# multiple crossings in 2D (rows), reference and saved rollouts
def nrow_multi(th):
    f = 0.5 * (1 + np.tanh(th / ws)) - 0.5; c = np.sum((f[:-1, :] > 0) & (f[1:, :] <= 0), axis=0); return np.mean(c > 1), np.mean(c == 0)
mc_ref = [nrow_multi(th) for th in d['test_th'][:, 1:].reshape(-1, N, N)]
mc_pred = [nrow_multi(th) for f in glob.glob(V + 'ckpt2d_r4/*rollout0.npz') for th in np.load(f)['th'][1:]]
MC = dict(ref_multi=float(np.mean([a for a, b in mc_ref])), ref_none=float(np.mean([b for a, b in mc_ref])),
          pred_multi=float(np.mean([a for a, b in mc_pred])), pred_none=float(np.mean([b for a, b in mc_pred])), n_pred_fields=len(mc_pred))
out = dict(table=table, contrasts=C, gauge=G, multi_crossing_2d=MC)
json.dump(out, open(V + 'active2d_r4.json', 'w'), indent=1)
for k, v in table.items(): print(k, v['n'], {m: round(v[m], 4) for m in ['relT', 'nu', 'rE']}, round(v['val'], 4))
for k, v in C.items(): print(k, round(100 * v['rel'], 1), [round(100 * x, 1) for x in v['ci_rel']], [round(100 * x, 1) for x in v['seed_rel']], v['p_signflip'])
print(json.dumps(G, indent=0)[:1500]); print(MC)
''')

**`v2/audit3.py`** — Admissibility stratification and cellwise macro-step ledger (Sec. IV B, Appendix E).

In [ ]:
writefile('v2/audit3.py', r'''"""Third-round audit computations.
A: continuity contribution to the 2D identity defect (reference pairs and network rollouts),
   with and without a discrete projection of the compared velocities.
B: kinetic predictions stratified into admissible (0<=f<=1) and inadmissible states;
   predicted temperature extrema (1D, both closures).
D: time-integrated macro balance of the 1D kinetic reference solver, and the origin of the
   initial-interval trapezoidal floor.
"""
import json, sys, glob
import numpy as np
sys.path.insert(0, '/home/claude/pof/v2'); sys.path.insert(0, '/home/claude/pof')
import jax.numpy as jnp
from scipy.fft import dctn, idctn
OUT = {}

# =================================================================== A
import train2d as T2D
d2 = np.load('/home/claude/pof/v2/data_2d.npz'); N = 32; h = 1.0 / N; ws = 0.02; thc = -0.1; DTH = 0.2
G = T2D.make_G(ws)

def residuals_faces(th0, th1, uf, vf, dt, Ste):
    """Same as T2D.residuals but with given face velocities; returns rE, rS, T1, s1, H1."""
    th0, th1 = jnp.array(th0)[None], jnp.array(th1)[None]
    f0, f1 = T2D.f_eq(th0, ws), T2D.f_eq(th1, ws)
    H0 = DTH * (th0 + f0 / Ste); H1 = DTH * (th1 + f1 / Ste)
    T0 = 1 + DTH * th0; T1 = 1 + DTH * th1
    uf, vf = jnp.array(uf)[None], jnp.array(vf)[None]
    qx, qy = T2D.cond_fluxes(T1, h, 1 + DTH, 1 + DTH * thc)
    Ax, Ay = T2D.upwind(H1, uf, vf)
    rE = (H1 - H0) / dt + T2D.div(Ax + qx, Ay + qy, h)
    Tg, Gt = G; Labs = DTH / Ste
    s0 = jnp.log(T0) + Labs * jnp.interp(T0, Tg, Gt); s1 = jnp.log(T1) + Labs * jnp.interp(T1, Tg, Gt)
    Tfx = jnp.concatenate([jnp.full((1, 1, N), 1 + DTH), 0.5 * (T1[:, 1:] + T1[:, :-1]), jnp.full((1, 1, N), 1 + DTH * thc)], 1)
    Tfy = jnp.concatenate([T1[:, :, :1], 0.5 * (T1[:, :, 1:] + T1[:, :, :-1]), T1[:, :, -1:]], 2)
    Sx, Sy = T2D.upwind(s1, uf, vf)
    sig = (0.5 * (qx[:, :-1] ** 2 + qx[:, 1:] ** 2) + 0.5 * (qy[:, :, :-1] ** 2 + qy[:, :, 1:] ** 2)) / T1 ** 2
    rS = (s1 - s0) / dt + T2D.div(Sx + qx / Tfx, Sy + qy / Tfy, h) - sig
    return [np.array(z[0]) for z in (rE, rS, T1, s1, H1)]

def faces(u, v):
    uf = np.zeros((N + 1, N)); uf[1:-1] = 0.5 * (u[1:] + u[:-1])
    vf = np.zeros((N, N + 1)); vf[:, 1:-1] = 0.5 * (v[:, 1:] + v[:, :-1])
    return uf, vf

lam_ = (2 * (np.cos(np.pi * np.arange(N) / N) - 1) / h ** 2)
DEN = lam_[:, None] + lam_[None, :]; DEN[0, 0] = 1.0
def project(uf, vf):
    dv = (uf[1:] - uf[:-1]) / h + (vf[:, 1:] - vf[:, :-1]) / h
    w = dctn(dv, type=2, norm='ortho') / DEN; w[0, 0] = 0
    phi = idctn(w, type=2, norm='ortho')
    uf = uf.copy(); vf = vf.copy()
    uf[1:-1] -= (phi[1:] - phi[:-1]) / h; vf[:, 1:-1] -= (phi[:, 1:] - phi[:, :-1]) / h
    return uf, vf

def audit_traj(TH, UU, VV, tt, Ste):
    rows = []
    for k in range(1, TH.shape[0] - 1):        # skip the first (impulsive start) pair
        dt = tt[k + 1] - tt[k]
        uf, vf = faces(UU[k + 1], VV[k + 1])
        dv = (uf[1:] - uf[:-1]) / h + (vf[:, 1:] - vf[:, :-1]) / h
        rE, rS, T1, s1, H1 = residuals_faces(TH[k], TH[k + 1], uf, vf, dt, Ste)
        D = rS - rE / T1; c = (s1 - H1 / T1) * dv
        ufp, vfp = project(uf, vf)
        dvp = (ufp[1:] - ufp[:-1]) / h + (vfp[:, 1:] - vfp[:, :-1]) / h
        rEp, rSp, T1p, _, _ = residuals_faces(TH[k], TH[k + 1], ufp, vfp, dt, Ste)
        Dp = rSp - rEp / T1p
        S0 = np.sqrt(np.mean((rE / T1) ** 2))
        rows.append(dict(div_rms=float(np.sqrt(np.mean(dv ** 2))), speed_over_h=float(np.sqrt(np.mean(UU[k+1]**2+VV[k+1]**2)) / h),
                         div_proj_rms=float(np.sqrt(np.mean(dvp ** 2))),
                         ident=float(np.linalg.norm(D) / np.linalg.norm(rE / T1)),
                         ident_minus_cont=float(np.linalg.norm(D - c) / np.linalg.norm(rE / T1)),
                         ident_projected=float(np.linalg.norm(Dp) / np.linalg.norm(rEp / T1p)),
                         cont_share=float(np.linalg.norm(c) / np.linalg.norm(D))))
    return {k: float(np.median([r[k] for r in rows])) for k in rows[0]}

A = {}
for j in range(6):
    A[f'ref_test{j}'] = audit_traj(d2['test_th'][j], d2['test_u'][j], d2['test_v'][j], d2['test_t'][j], float(d2['test_Ste'][j]))
for v in ['D', 'E', 'ES', 'EET']:
    for s in [1, 2]:
        z = np.load(f'/home/claude/pof/v2/ckpt2d/{v}_s{s}_rollout0.npz')
        A[f'{v}_s{s}'] = audit_traj(z['th'], z['u'], z['v'], d2['test_t'][0], float(d2['test_Ste'][0]))
OUT['continuity_2d'] = A
for k, v in A.items(): print('A', k, {a: round(b, 4) for a, b in v.items()})

# =================================================================== B
import train2 as TR
def load_params(f):
    z = np.load(f); n = len([k for k in z.files if k.startswith('W')])
    return [(jnp.array(z[f'W{i}']), jnp.array(z[f'b{i}'])) for i in range(n)], float(z['dT_scale']), float(z['df_scale'])

def rollout(params, dTs, dfs, d, j, kin):
    x = jnp.array((np.arange(128) + 0.5) / 128)
    T = [jnp.array(d['test_T'][j, :1])]; f = [jnp.array(d['test_f'][j, :1])] if kin else None
    th = jnp.array([d['test_th'][j]]); L = jnp.array([d['test_L'][j]]); t = d['test_t'][j]
    for k in range(t.shape[0] - 1):
        dt = jnp.array([t[k + 1] - t[k]])
        xin = TR.build_inputs(T[-1], th, L, dt, x, f[-1] if kin else None)
        y = TR.forward(params, xin, 1.0)
        T.append(T[-1] + y[:, 0] * dTs)
        if kin: f.append(f[-1] + y[:, 1] * dfs)
    return np.concatenate([np.array(a) for a in T]), (np.concatenate([np.array(a) for a in f]) if kin else None)

dk = np.load('/home/claude/pof/data_kin.npz'); de = np.load('/home/claude/pof/data_eq.npz')
B = {}
for v in ['D', 'E', 'EPH', 'EK']:
    agg = dict(out_frac=[], neg_total=[], neg_in_inadm=[], P0_adm=[], P0_inadm=[], Rf_adm=[], Rf_inadm=[], Tmin=[], Tmax=[], fmin=[], fmax=[])
    for s in range(8):
        params, dTs, dfs = load_params(f'/home/claude/pof/v2/ckpt/kin_{v}_s{s}.npz')
        F0 = json.load(open(f'/home/claude/pof/v2/runs/kin_{v}_s{s}.json'))['scales']['F0']
        for j in range(dk['test_T'].shape[0]):
            T, f = rollout(params, dTs, dfs, dk, j, True)
            t = dk['test_t'][j]; dtv = jnp.array(np.diff(t))
            n = T.shape[0] - 1
            rE, rf, sph = TR.resid_kin(jnp.array(T[:-1]), jnp.array(f[:-1]), jnp.array(T[1:]), jnp.array(f[1:]), dtv,
                                       jnp.full((n,), dk['test_th'][j]), jnp.full((n,), dk['test_L'][j]), 1 / 128, 400.0)
            rf, sph = np.array(rf), np.array(sph)
            adm = (f[1:] >= 0) & (f[1:] <= 1) & (f[:-1] >= 0) & (f[:-1] <= 1)
            neg = sph < -1e-12
            agg['out_frac'].append(1 - adm.mean()); agg['neg_total'].append(neg.mean())
            agg['neg_in_inadm'].append((neg & ~adm).sum() / max(neg.sum(), 1))
            agg['P0_adm'].append((neg & adm).sum() / max(adm.sum(), 1)); agg['P0_inadm'].append((neg & ~adm).sum() / max((~adm).sum(), 1))
            agg['Rf_adm'].append(np.sqrt(np.mean(rf[adm] ** 2)) / F0 if adm.any() else np.nan)
            agg['Rf_inadm'].append(np.sqrt(np.mean(rf[~adm] ** 2)) / F0 if (~adm).any() else np.nan)
            agg['Tmin'].append(T.min()); agg['Tmax'].append(T.max()); agg['fmin'].append(f.min()); agg['fmax'].append(f.max())
    B[v] = {k: (float(np.nanmean(x)) if k not in ('Tmin', 'fmin') else float(np.min(x))) for k, x in agg.items()}
    B[v]['Tmax'] = float(np.max(agg['Tmax'])); B[v]['fmax'] = float(np.max(agg['fmax']))
    print('B kin', v, {a: round(b, 4) for a, b in B[v].items()})
for v in ['D', 'E', 'ES', 'EET']:
    tmin, tmax = [], []
    for s in range(4, 8):
        params, dTs, _ = load_params(f'/home/claude/pof/v2/ckpt/eq_{v}_s{s}.npz')
        for j in range(de['test_T'].shape[0]):
            T, _ = rollout(params, dTs, 0, de, j, False); tmin.append(T.min()); tmax.append(T.max())
    B[f'eq_{v}'] = dict(Tmin=float(min(tmin)), Tmax=float(max(tmax)))
    print('B eq', v, B[f'eq_{v}'])
OUT['kinetic_admissibility'] = B

# =================================================================== D
from solvers1d import X_drive
def kinetic_ledger(Th=1.25, L=2.0, M0=400.0, N=128, t_end=0.1, nmacro=25):
    dx = 1 / N; dt0 = 0.25 * dx * dx; n = int(np.ceil(t_end / dt0)); dt = t_end / n
    T = np.full(N, 0.95); f = np.full(N, 0.01)
    edges = np.linspace(0, n, nmacro).round().astype(int)
    states = [(T.copy(), f.copy())]; intdiv = np.zeros(N); intrate = np.zeros(N); ledgers = []
    for k in range(n):
        q = np.empty(N + 1); q[1:-1] = -(T[1:] - T[:-1]) / dx; q[0] = -(T[0] - Th) / (0.5 * dx); q[-1] = 0
        lap = -(q[1:] - q[:-1]) / dx; rate = M0 * f * (1 - f) * X_drive(T, L)
        df = np.clip(dt * rate, -f, 1 - f)
        intdiv += dt * (-lap); intrate += df
        T = T + dt * lap - L * df; f = f + df
        if k + 1 in edges[1:]:
            T0, f0 = states[-1]; Dt = (k + 1 - edges[len(states) - 1]) * dt
            H0, H1 = T0 + L * f0, T + L * f
            exact = (H1 - H0) / Dt + intdiv / Dt                      # time-integrated balance
            # endpoint and trapezoid approximations of the flux integral
            def dq(TT):
                qq = np.empty(N + 1); qq[1:-1] = -(TT[1:] - TT[:-1]) / dx; qq[0] = -(TT[0] - Th) / (0.5 * dx); qq[-1] = 0
                return (qq[1:] - qq[:-1]) / dx
            endp = (H1 - H0) / Dt + dq(T); trap = (H1 - H0) / Dt + 0.5 * (dq(T) + dq(T0))
            rate_scale = np.sqrt(np.mean(((H1 - H0) / Dt) ** 2))
            ledgers.append(dict(interval=len(states), exact_rel=float(np.sqrt(np.mean(exact ** 2)) / rate_scale),
                                end_rel=float(np.sqrt(np.mean(endp ** 2)) / rate_scale),
                                trap_rel=float(np.sqrt(np.mean(trap ** 2)) / rate_scale),
                                kin_exact=float(np.abs((f - f0) - intrate).max()),
                                wall_flux_t0=float(-(T0[0] - Th) / (0.5 * dx)), mean_wall_flux=float(intdiv[0] * 0 + 0)))
            states.append((T.copy(), f.copy())); intdiv[:] = 0; intrate[:] = 0
    return ledgers
Dl = kinetic_ledger()
OUT['macro_ledger'] = Dl
for r in Dl[:4] + Dl[-2:]: print('D', {a: (round(b, 6) if isinstance(b, float) else b) for a, b in r.items()})
json.dump(OUT, open('/home/claude/pof/v2/audit3.json', 'w'), indent=1)
print('saved audit3.json')
''')

**`v2/audit4_velocity.py`** — Native, reconstructed, and projected velocities (Sec. VI C, Fig. 7a,b).

In [ ]:
writefile('v2/audit4_velocity.py', r'''"""Round-4 audit (all ratios use the native-face rms of r_E/T as common denominator): native MAC vs reconstructed (centre->face) vs projected-reconstructed velocities.
Re-runs the six 32^2 test trajectories with the dataset solver settings, keeps native MAC faces, and
evaluates the macro-step energy/entropy residuals with each face field (common denominator S0)."""
import sys, json, numpy as np
sys.path.insert(0, '/home/claude/pof/v2')
import jax; jax.config.update('jax_platform_name', 'cpu')
import jax.numpy as jnp
from solver2d_v2 import run
import train2d_r4 as T

NC = 32; h = 1.0 / NC; ws = 0.02; thc = -0.1
times = list(np.linspace(0, 0.25, 21)[1:])
Ra_vals = [5e4, 8e4, 1.2e5, 1.6e5, 2e5]; Ste_vals = [0.1, 0.15, 0.2, 0.25, 0.3, 0.35]
combos = [(ra, st) for ra in Ra_vals for st in Ste_vals]
rng = np.random.default_rng(1); rng.shuffle(combos); test = combos[24:]
d = np.load('/home/claude/pof/v2/data_2d.npz')
Gtab = T.make_G(ws); T._PROJ['Ainv'] = T.make_proj(NC, h)

def resid_with_faces(th0, th1, uf, vf, dt, St):
    """same discretization as train2d_r4.residuals but with given faces."""
    f0, f1 = T.f_eq(th0, ws), T.f_eq(th1, ws)
    H0 = T.DTH * (th0 + f0 / St[:, None, None]); H1 = T.DTH * (th1 + f1 / St[:, None, None])
    T0 = 1.0 + T.DTH * th0; T1 = 1.0 + T.DTH * th1
    rm = T.div(uf, vf, h)
    qx, qy = T.cond_fluxes(T1, h, 1.0 + T.DTH, 1.0 + T.DTH * thc)
    Ax, Ay = T.upwind(H1, uf, vf)
    rE = (H1 - H0) / dt[:, None, None] + T.div(Ax + qx, Ay + qy, h)
    Tg, G = Gtab; Labs = T.DTH / St[:, None, None]
    s0 = jnp.log(T0) + Labs * jnp.interp(T0, Tg, G); s1 = jnp.log(T1) + Labs * jnp.interp(T1, Tg, G)
    B, N, _ = T1.shape
    Tfx = jnp.concatenate([jnp.full((B, 1, N), 1.0 + T.DTH), 0.5 * (T1[:, 1:] + T1[:, :-1]), jnp.full((B, 1, N), 1.0 + T.DTH * thc)], axis=1)
    Tfy = jnp.concatenate([T1[:, :, :1], 0.5 * (T1[:, :, 1:] + T1[:, :, :-1]), T1[:, :, -1:]], axis=2)
    Sx, Sy = T.upwind(s1, uf, vf)
    divS = T.div(Sx + qx / Tfx, Sy + qy / Tfy, h)
    sig = (0.5 * (qx[:, :-1] ** 2 + qx[:, 1:] ** 2) + 0.5 * (qy[:, :, :-1] ** 2 + qy[:, :, 1:] ** 2)) / T1 ** 2
    rS = (s1 - s0) / dt[:, None, None] + divS - sig
    return rE, rS, T1, s1, H1, rm

rows = {}
for j, (ra, st) in enumerate(test):
    assert abs(d['test_Ra'][j] - ra) < 1e-9 and abs(d['test_Ste'][j] - st) < 1e-12
    r = run(nc=NC, Ra=ra, Pr=1.0, Ste=st, ws=0.02, Cm=1e6, eps_m=1e-3, theta_i=-0.1, theta_c=-0.1, t_end=0.25, out_times=times)
    sn = r['snaps']
    th = np.stack([np.full((NC, NC), -0.1)] + [s['th'] for s in sn])
    assert np.abs(th - d['test_th'][j]).max() < 1e-9, np.abs(th - d['test_th'][j]).max()
    ufn = np.stack([s['uf'] for s in sn]); vfn = np.stack([s['vf'] for s in sn])
    uc = np.stack([s['u'] for s in sn]); vc = np.stack([s['v'] for s in sn])
    tt = np.array([0.0] + [s['t'] for s in sn]); dt = jnp.array(np.diff(tt)); Stv = jnp.full((len(sn),), st)
    th0, th1 = jnp.array(th[:-1]), jnp.array(th[1:])
    ufr, vfr = T.faces_from_centers(jnp.array(uc), jnp.array(vc))
    ufp, vfp = T.project(ufr, vfr, h)
    ufnp, vfnp = T.project(jnp.array(ufn), jnp.array(vfn), h)
    out = {}
    Urms = float(np.sqrt(np.mean(uc[1:] ** 2 + vc[1:] ** 2)))
    base = None
    for name, (uf, vf) in dict(native=(jnp.array(ufn), jnp.array(vfn)), reconstructed=(ufr, vfr), recon_projected=(ufp, vfp)).items():
        rE, rS, T1, s1, H1, rm = resid_with_faces(th0, th1, uf, vf, dt, Stv)
        if base is None: base = float(jnp.sqrt(jnp.mean((rE / T1) ** 2)))     # S0 from native faces: common denominator
        if name == 'native': base1 = float(jnp.sqrt(jnp.mean((rE[1:] / T1[1:]) ** 2)))
        rSc = rS - (s1 - H1 / T1) * rm
        # reference-state (gauge) sensitivity: shift h by C_h, s by C_s
        dE = float(jnp.sqrt(jnp.mean(rm ** 2)))
        out[name] = dict(div_rms_dimless=float(jnp.sqrt(jnp.mean(rm[1:] ** 2))) / Urms,    # H=1 domain scale
                         div_rms=float(jnp.sqrt(jnp.mean(rm[1:] ** 2))),
                         rE_rms=float(jnp.sqrt(jnp.mean(rE ** 2))),
                         ident=float(jnp.sqrt(jnp.mean((rS - rE / T1) ** 2))) / base,
                         ident_corrected=float(jnp.sqrt(jnp.mean((rSc - rE / T1) ** 2))) / base,
                         ident_excl1=float(jnp.sqrt(jnp.mean((rS[1:] - rE[1:] / T1[1:]) ** 2))) / base1,
                         ident_corrected_excl1=float(jnp.sqrt(jnp.mean((rSc[1:] - rE[1:] / T1[1:]) ** 2))) / base1,
                         rE_shift_per_unit_Ch=dE)
    out['diff_native_vs_recon_faces_rms'] = float(np.sqrt(np.mean((ufn - np.array(ufr)) ** 2) + np.mean((vfn - np.array(vfr)) ** 2))) / Urms
    out['Urms'] = Urms; out['S0_common'] = base
    rows[f'test{j}'] = out
    print(j, ra, st, json.dumps({k: (v if not isinstance(v, dict) else {kk: round(vv, 5) for kk, vv in v.items()}) for k, v in out.items()}), flush=True)
json.dump(rows, open('/home/claude/pof/v2/audit4_velocity.json', 'w'), indent=1)
''')

**`v2/audit4_dg.py`** — Discrete-gradient pair on the training pairs (Appendix G).

In [ ]:
writefile('v2/audit4_dg.py', r'''"""Round-4: fully specified discrete-gradient (secant) energy-entropy residual pair for the 1D
equilibrium conduction problem, evaluated on the exact reference pairs used for training (12 trajectories).
Construction (cell i, macro-step n->n+1, k = rho c = 1, dx = 1/N):
  a_i = (s_i^{n+1}-s_i^n)/(h_i^{n+1}-h_i^n)  (secant), a_i = 1/T(h_i^{n+1/2}) if |dh| < 1e-9;  T^d_i = 1/a_i
  interior heat flux  Q_{i+1/2} = (T^d_i - T^d_{i+1})/dx ;  wall Q_w = (T_h - T^d_1)/(dx/2) ;  adiabatic Q_N = 0
  r_E,i = (h^{n+1}-h^n)/dt + (Q_{i+1/2}-Q_{i-1/2})/dx
  entropy flux J_{i+1/2} = (a_i+a_{i+1})/2 Q_{i+1/2} ;  J_w = Q_w/T_h
  production  Pi_i = [ Q_{i+1/2}(a_{i+1}-a_i)/2 + Q_{i-1/2}(a_i-a_{i-1})/2 ]/dx  (+ Q_w(a_1-1/T_h)/dx in cell 1)
  r_S,i = (s^{n+1}-s^n)/dt + (J_{i+1/2}-J_{i-1/2})/dx - Pi_i
Then r_S = a r_E identically (algebra), and every face production is >= 0 when a > 0.
The numerical check below is an implementation check of that algebra, not an independent verification."""
import json, sys, numpy as np
sys.path.insert(0, '/home/claude/pof/v2')
import train2 as T2
d = np.load('/home/claude/pof/data_eq.npz'); w = float(d['w']); N = int(d['N']); dx = 1.0 / N
tab = T2.make_sf_table(w); Tg, Sf = [np.array(z) for z in tab]
h = lambda T, L: (T - 1.0) + L * (0.5 * (1 + np.tanh((T - 1.0) / w)) - 0.5)
s = lambda T, L: np.log(T) + L * np.interp(T, Tg, Sf)      # table entropy (as in the training losses)
from numpy.polynomial.legendre import leggauss
_x, _wq = leggauss(64)
def ds_quad(T0, T1, L):
    """s(T1)-s(T0) = int_{T0}^{T1} h'(T)/T dT by 64-point Gauss-Legendre (consistent with the closure)."""
    m = 0.5 * (T0 + T1); r = 0.5 * (T1 - T0); TT = m[:, None] + r[:, None] * _x
    hp = 1.0 + L * 0.5 / (w * np.cosh((TT - 1.0) / w) ** 2)
    return r * np.sum(_wq * hp / TT, axis=1)
ntr = 12; out = {'rows': []}
QUAD = '--table' not in sys.argv
END_all, TRAP_all = [], []
E_all, S_all, rate_all, rem_all, prodmin, eqlim, a_ok = [], [], [], [], [], 0, True
for j in range(ntr):
    TT = d['train_T'][j].astype(float); th = float(d['train_th'][j]); L = float(d['train_L'][j]); t = d['train_t'][j]
    for k in range(TT.shape[0] - 1):
        T0, T1 = TT[k], TT[k + 1]; dt = t[k + 1] - t[k]
        dh = h(T1, L) - h(T0, L); ds = ds_quad(T0.astype(float), T1.astype(float), L) if QUAD else s(T1, L) - s(T0, L)
        small = np.abs(dh) < 1e-9; eqlim += int(small.sum())
        a = np.where(small, 1.0 / (0.5 * (T0 + T1)), ds / np.where(small, 1.0, dh))
        a_ok &= bool(np.all(a > 0))
        Td = 1.0 / a
        Q = np.zeros(N + 1); Q[1:-1] = (Td[:-1] - Td[1:]) / dx; Q[0] = (th - Td[0]) / (0.5 * dx)
        rE = dh / dt + (Q[1:] - Q[:-1]) / dx
        J = np.zeros(N + 1); J[1:-1] = 0.5 * (a[:-1] + a[1:]) * Q[1:-1]; J[0] = Q[0] / th
        pf = np.zeros(N + 1); pf[1:-1] = Q[1:-1] * (a[1:] - a[:-1]); pw = Q[0] * (a[0] - 1.0 / th)
        Pi = 0.5 * (pf[1:] + pf[:-1]) / dx; Pi[0] = 0.5 * pf[1] / dx + pw / dx
        rS = ds / dt + (J[1:] - J[:-1]) / dx - Pi
        rem_all.append(np.abs(rS - a * rE).max() / max(np.abs(rS).max(), 1e-30))
        prodmin.append(min(pf[1:-1].min(), pw))
        E_all.append((k, rE)); rate_all.append((k, dh / dt))
        def dq(TT_):
            q = np.zeros(N + 1); q[1:-1] = (TT_[:-1] - TT_[1:]) / dx; q[0] = (th - TT_[0]) / (0.5 * dx)
            return (q[1:] - q[:-1]) / dx
        END_all.append((k, dh / dt + dq(T1))); TRAP_all.append((k, dh / dt + 0.5 * (dq(T0) + dq(T1))))
def floor(skip, A=None):
    A = E_all if A is None else A
    num = np.concatenate([e for k, e in A if k >= skip]); den = np.concatenate([r for k, r in rate_all if k >= skip])
    return float(np.sqrt(np.mean(num ** 2)) / np.sqrt(np.mean(den ** 2)))
out['floors_same_script'] = {nm: {f'skip{sk}': floor(sk, A) for sk in (0, 2, 5)} for nm, A in
                             (('discrete_gradient', E_all), ('endpoint', END_all), ('trapezoid', TRAP_all))}
ratio = [np.sqrt(np.mean(e ** 2)) / np.sqrt(np.mean(f ** 2)) for (k, e), (_, f) in zip(E_all, END_all) if k >= 5]
out['median_interval_ratio_dg_over_endpoint_k>=5'] = float(np.median(ratio))
out.update(dg_floor_all=floor(0), dg_floor_skip1=floor(1), dg_floor_skip2=floor(2),
           remainder_max=float(max(rem_all)), min_face_production=float(min(prodmin)),
           equal_state_cells=eqlim, secant_positive=a_ok,
           note='floors are rms(r_E)/rms(storage rate) over the 12 training trajectories; compare floors_skip.json eq_end/eq_trap')
fs = json.load(open('/home/claude/pof/v2/floors_skip.json'))
out['endpoint'] = {k: fs[f'eq_end_skip{k}']['E0_rel'] for k in (0, 1, 2)}
out['trapezoid'] = {k: fs[f'eq_trap_skip{k}']['E0_rel'] for k in (0, 1, 2)}
out['entropy_increment'] = 'gauss' if QUAD else 'table'
json.dump(out, open('/home/claude/pof/v2/audit4_dg%s.json' % ('' if QUAD else '_table'), 'w'), indent=1)
print(json.dumps({k: v for k, v in out.items() if k != 'rows'}, indent=1))
''')

**`v2/audit4_kin.py`** — Mobility in inadmissible cells; front crossings (Appendices C, E).

In [ ]:
writefile('v2/audit4_kin.py', r'''"""Round-4: (a) logistic-mobility values and kinetic-loss share in inadmissible cells of 1D kinetic rollouts;
(b) frequency of multiple f=1/2 crossings in 1D (eq, kin) rollouts and in reference profiles."""
import json, sys, numpy as np
sys.path.insert(0, '/home/claude/pof/v2')
import jax.numpy as jnp
import train2 as TR
def load_params(f):
    z = np.load(f); n = len([k for k in z.files if k.startswith('W')])
    return [(jnp.array(z[f'W{i}']), jnp.array(z[f'b{i}'])) for i in range(n)], float(z['dT_scale']), float(z['df_scale'])
def rollout(params, dTs, dfs, d, j, kin):
    x = jnp.array((np.arange(128) + 0.5) / 128)
    T = [jnp.array(d['test_T'][j, :1])]; f = [jnp.array(d['test_f'][j, :1])] if kin else None
    th = jnp.array([d['test_th'][j]]); L = jnp.array([d['test_L'][j]]); t = d['test_t'][j]
    for k in range(t.shape[0] - 1):
        dt = jnp.array([t[k + 1] - t[k]])
        y = TR.forward(params, TR.build_inputs(T[-1], th, L, dt, x, f[-1] if kin else None), 1.0)
        T.append(T[-1] + y[:, 0] * dTs)
        if kin: f.append(f[-1] + y[:, 1] * dfs)
    return np.concatenate([np.array(a) for a in T]), (np.concatenate([np.array(a) for a in f]) if kin else None)
def ncross(prof):
    p = prof - 0.5; return int(np.sum((p[:-1] > 0) & (p[1:] <= 0)))
dk = np.load('/home/claude/pof/data_kin.npz'); de = np.load('/home/claude/pof/data_eq.npz'); w = float(de['w'])
M0 = 400.0; out = {}
for v in ['D', 'E', 'EPH', 'EK']:
    Mi, share, mult, nprof = [], [], 0, 0
    for s in range(8):
        params, dTs, dfs = load_params(f'/home/claude/pof/v2/ckpt/kin_{v}_s{s}.npz')
        for j in range(dk['test_T'].shape[0]):
            T, f = rollout(params, dTs, dfs, dk, j, True); t = dk['test_t'][j]; n = T.shape[0] - 1
            rE, rf, sph = TR.resid_kin(jnp.array(T[:-1]), jnp.array(f[:-1]), jnp.array(T[1:]), jnp.array(f[1:]), jnp.array(np.diff(t)),
                                       jnp.full((n,), dk['test_th'][j]), jnp.full((n,), dk['test_L'][j]), 1 / 128, M0)
            rf = np.array(rf); f1 = f[1:]; adm = (f1 >= 0) & (f1 <= 1)
            Mi.append(M0 * f1[~adm] * (1 - f1[~adm])); share.append(np.sum(rf[~adm] ** 2) / np.sum(rf ** 2))
            for k in range(1, T.shape[0]): mult += ncross(f[k]) > 1; nprof += 1
    Mi = np.concatenate(Mi)
    out[f'kin_{v}'] = dict(M_inadm_q=[float(np.quantile(Mi, q)) for q in (0.01, 0.1, 0.5, 0.9)] if Mi.size else None,
                          M_inadm_min=float(Mi.min()) if Mi.size else None, M_admissible_max=M0 / 4,
                          kinetic_loss_share_inadm=float(np.mean(share)), multi_crossing_frac=mult / nprof)
    print(v, out[f'kin_{v}'], flush=True)
for v in ['D', 'E', 'ES', 'EET']:
    mult = nprof = 0
    for s in range(4, 8):
        params, dTs, _ = load_params(f'/home/claude/pof/v2/ckpt/eq_{v}_s{s}.npz')
        for j in range(de['test_T'].shape[0]):
            T, _ = rollout(params, dTs, 0, de, j, False)
            for k in range(1, T.shape[0]): mult += ncross(0.5 * (1 + np.tanh((T[k] - 1) / w))) > 1; nprof += 1
    out[f'eq_{v}_s4-7'] = dict(multi_crossing_frac=mult / nprof); print(v, out[f'eq_{v}_s4-7'], flush=True)
ref = [ncross(fk) > 1 for fk in dk['test_f'].reshape(-1, 128)]
out['ref_kin_multi_crossing_frac'] = float(np.mean(ref))
json.dump(out, open('/home/claude/pof/v2/audit4_kin.json', 'w'), indent=1)
''')

**`v2/audit5_dg.py`** — Discrete-gradient branch precision and threshold sensitivity (Appendix G).

In [ ]:
writefile('v2/audit5_dg.py', r'''"""Round-5: precision of the discrete-gradient pair (Appendix C).
(1) thresholded branch (a = 2/(T^n+T^{n+1}) if |dh| < tau): compatibility defect |ds - a dh|/dt, threshold sensitivity;
(2) stable evaluation: dh and ds both by the same 64-point Gauss rule, a = int h'/T / int h' (equal-state limit only
    when T^{n+1} == T^n exactly); then r_S - a r_E = (ds - a dh)/dt vanishes to floating point."""
import json, numpy as np
from numpy.polynomial.legendre import leggauss
d = np.load('/home/claude/pof/data_eq.npz'); w = float(d['w']); N = int(d['N']); dx = 1.0 / N
xg, wg = leggauss(64)
hp = lambda T, L: 1.0 + L * 0.5 / (w * np.cosh((T - 1.0) / w) ** 2)
h = lambda T, L: (T - 1.0) + L * (0.5 * (1 + np.tanh((T - 1.0) / w)) - 0.5)
def quad(T0, T1, L):
    m = 0.5 * (T0 + T1); r = 0.5 * (T1 - T0); TT = m[:, None] + r[:, None] * xg
    H = hp(TT, L); return r * np.sum(wg * H, 1), r * np.sum(wg * H / TT, 1), np.sum(wg * H / TT, 1) / np.sum(wg * H, 1)
def run(mode, tau=1e-9):
    res = dict(defect_abs_max=0.0, n_branch=0, rem=0.0, prodmin=np.inf, dh_quad_vs_exact_rel=0.0)
    E, END, R = [], [], []; s_rate = []
    for j in range(12):
        TT = d['train_T'][j].astype(float); th = float(d['train_th'][j]); L = float(d['train_L'][j]); t = d['train_t'][j]
        for k in range(TT.shape[0] - 1):
            T0, T1 = TT[k], TT[k + 1]; dt = t[k + 1] - t[k]
            dhq, ds, aq = quad(T0, T1, L); dhe = h(T1, L) - h(T0, L)
            if mode == 'threshold':
                dh = dhe; small = np.abs(dh) < tau
                a = np.where(small, 2.0 / (T0 + T1), ds / np.where(small, 1.0, dh)); res['n_branch'] += int(small.sum())
            else:
                dh = dhq; eq = (T1 == T0); a = np.where(eq, 1.0 / T0, aq); res['n_branch'] += int(eq.sum())
                big = np.abs(dhe) > 1e-6
                if big.any(): res['dh_quad_vs_exact_rel'] = max(res['dh_quad_vs_exact_rel'], float(np.max(np.abs(dhq[big] - dhe[big]) / np.abs(dhe[big]))))
            res['defect_abs_max'] = max(res['defect_abs_max'], float(np.max(np.abs(ds - a * dh)) / dt))
            s_rate.append(ds / dt)
            Td = 1 / a
            Q = np.zeros(N + 1); Q[1:-1] = (Td[:-1] - Td[1:]) / dx; Q[0] = (th - Td[0]) / (0.5 * dx)   # Q[N] = 0 (adiabatic)
            rE = dh / dt + (Q[1:] - Q[:-1]) / dx
            J = np.zeros(N + 1); J[1:-1] = 0.5 * (a[:-1] + a[1:]) * Q[1:-1]; J[0] = Q[0] / th
            pf = np.zeros(N + 1); pf[1:-1] = Q[1:-1] * (a[1:] - a[:-1]); pf[0] = Q[0] * (a[0] - 1 / th)   # pf[N] = 0
            Pi = (0.5 * pf[1:] + 0.5 * pf[:-1]) / dx; Pi[0] = (0.5 * pf[1] + pf[0]) / dx
            rS = ds / dt + (J[1:] - J[:-1]) / dx - Pi
            res['rem'] = max(res['rem'], float(np.abs(rS - a * rE).max() / np.abs(rS).max()))
            res['prodmin'] = min(res['prodmin'], float(pf.min()))
            q = np.zeros(N + 1); q[1:-1] = (T1[:-1] - T1[1:]) / dx; q[0] = (th - T1[0]) / (0.5 * dx)
            E.append((k, rE)); END.append((k, dhe / dt + (q[1:] - q[:-1]) / dx)); R.append((k, dhe / dt))
    fl = lambda A, sk: float(np.sqrt(np.mean(np.concatenate([e for kk, e in A if kk >= sk]) ** 2)) /
                             np.sqrt(np.mean(np.concatenate([e for kk, e in R if kk >= sk]) ** 2)))
    res['defect_rel_to_rms_entropy_rate'] = res['defect_abs_max'] / float(np.sqrt(np.mean(np.concatenate(s_rate) ** 2)))
    res['floors'] = {f'skip{sk}': fl(E, sk) for sk in (0, 2, 5)}
    return res
out = {f'threshold_{tau:g}': run('threshold', tau) for tau in (1e-12, 1e-9, 1e-6)}
out['stable'] = run('stable')
json.dump(out, open('/home/claude/pof/v2/audit5_dg.json', 'w'), indent=1)
for k, v in out.items(): print(k, {a: (round(b, 16) if isinstance(b, float) else b) for a, b in v.items()})
''')

**`v2/audit5_front.py`** — Spatial front-transition width (Appendix D).

In [ ]:
writefile('v2/audit5_front.py', r'''"""Round-5: spatial 10-90% liquid-fraction width of the reference fronts (1D equilibrium data), in cells."""
import json, numpy as np
d = np.load('/home/claude/pof/data_eq.npz'); w = float(d['w']); N = int(d['N']); dx = 1 / N
x = (np.arange(N) + 0.5) * dx
def cross(f, lev):
    p = f - lev; i = np.where((p[:-1] > 0) & (p[1:] <= 0))[0]
    if len(i) == 0: return None
    i = i[0]; return x[i] + dx * p[i] / (p[i] - p[i + 1])
W, Wg = [], []; n_front = 0; n_excl = 0; n_prof = 0
for split in ('train', 'val', 'test'):
    for T in d[f'{split}_T']:
        for Tk in T[1:]:
            f = 0.5 * (1 + np.tanh((Tk - 1) / w)); a, b = cross(f, 0.9), cross(f, 0.1); n_prof += 1
            if cross(f, 0.5) is not None: n_front += 1
            if a is None or b is None:
                n_excl += 1; continue
            W.append((b - a) / dx)
            xf = cross(f, 0.5); g = abs(np.interp(xf, x[1:-1], (Tk[2:] - Tk[:-2]) / (2 * dx)))
            Wg.append(2 * np.arctanh(0.8) * w / g / dx)
W, Wg = np.array(W), np.array(Wg)
out = dict(n=len(W), n_profiles=n_prof, n_with_half_crossing=n_front, n_excluded_missing_0p1_or_0p9=n_excl,
           gradient_location='central difference of T, linearly interpolated to the f=1/2 crossing; estimate divided by dx', width_cells_q=[float(np.quantile(W, q)) for q in (0.05, 0.25, 0.5, 0.75, 0.95)],
           width_cells_min=float(W.min()), gradient_estimate_median=float(np.median(Wg)))
json.dump(out, open('/home/claude/pof/v2/audit5_front.json', 'w'), indent=1); print(out)
''')

**`v2/audit5_precision.py`** — Double- versus single-precision divergence (Appendix C).

In [ ]:
writefile('v2/audit5_precision.py', r'''"""Round-5: dtypes and divergence of native MAC faces in float64 vs float32."""
import json, sys, numpy as np
sys.path.insert(0, '/home/claude/pof/v2')
from solver2d_v2 import run
d = np.load('/home/claude/pof/v2/data_2d.npz'); N = 32; h = 1 / N
r = run(nc=N, Ra=float(d['test_Ra'][0]), Pr=1.0, Ste=float(d['test_Ste'][0]), ws=0.02, Cm=1e6, eps_m=1e-3,
        theta_i=-0.1, theta_c=-0.1, t_end=0.25, out_times=list(np.linspace(0, 0.25, 21)[1:]))
out = dict(solver_dtype=str(r['snaps'][0]['uf'].dtype), dataset_dtype=str(d['test_u'].dtype))
def div(uf, vf): return (uf[1:] - uf[:-1]) / h + (vf[:, 1:] - vf[:, :-1]) / h
rel64, rel32 = [], []
for s in r['snaps']:
    U = np.sqrt(np.mean(s['u'] ** 2 + s['v'] ** 2))
    rel64.append(np.sqrt(np.mean(div(s['uf'], s['vf']) ** 2)) / U)
    rel32.append(np.sqrt(np.mean(div(s['uf'].astype(np.float32), s['vf'].astype(np.float32)).astype(np.float64) ** 2)) / U)
out.update(native_div_float64_max=float(max(rel64)), native_div_float32_max=float(max(rel32)))
json.dump(out, open('/home/claude/pof/v2/audit5_precision.json', 'w'), indent=1); print(out)
''')

**`v2/audit6_muscl.py`** — Reconstruction defect with upwind and MUSCL transport (Section 6.3).

In [ ]:
writefile('v2/audit6_muscl.py', r'''"""Review response: does a second-order (MUSCL) discretization change the velocity-reconstruction defect
relative to the thermal (native-face) mismatch?  Double precision throughout.
Solver advection in {upwind, MUSCL} x residual transport in {upwind, MUSCL}; faces native / reconstructed /
reconstructed with continuity term removed / reconstructed and projected.  Same six 32^2 test trajectories,
same macro-steps, defects = RMS(r_S - r_E/T) / RMS(r_E^nat / T) per trajectory."""
import sys, json, numpy as np
sys.path.insert(0, '/home/claude/pof/v2')
from solver2d_v2 import run, muscl_faces, NeumannPoisson
NC = 32; h = 1.0 / NC; ws = 0.02; thc = -0.1; DTH = 0.2
times = list(np.linspace(0, 0.25, 21)[1:])
Ra_vals = [5e4, 8e4, 1.2e5, 1.6e5, 2e5]; Ste_vals = [0.1, 0.15, 0.2, 0.25, 0.3, 0.35]
combos = [(ra, st) for ra in Ra_vals for st in Ste_vals]
rng = np.random.default_rng(1); rng.shuffle(combos); test = combos[24:]
Tg = np.linspace(0.6, 1.6, 400001); wa = ws * DTH
integ = (0.5 / (wa * np.cosh((Tg - 1.0) / wa) ** 2)) / Tg
G = np.concatenate([[0.0], np.cumsum(0.5 * (integ[1:] + integ[:-1]) * np.diff(Tg))]); G -= np.interp(1.0, Tg, G)
f_eq = lambda th: 0.5 * (1.0 + np.tanh(th / ws))
pois = NeumannPoisson(NC, h)

def faces_from_centers(u, v):
    uf = np.zeros((NC + 1, NC)); vf = np.zeros((NC, NC + 1))
    uf[1:-1] = 0.5 * (u[1:] + u[:-1]); vf[:, 1:-1] = 0.5 * (v[:, 1:] + v[:, :-1]); return uf, vf
def div(Fx, Fy): return (Fx[1:] - Fx[:-1]) / h + (Fy[:, 1:] - Fy[:, :-1]) / h
def project(uf, vf):
    phi = pois.solve(div(uf, vf)); uf = uf.copy(); vf = vf.copy()
    uf[1:-1] -= (phi[1:] - phi[:-1]) / h; vf[:, 1:-1] -= (phi[:, 1:] - phi[:, :-1]) / h; return uf, vf
def transport(q, uf, vf, scheme):
    Fx = np.zeros((NC + 1, NC)); Fy = np.zeros((NC, NC + 1))
    if scheme == 'muscl':
        L, R = muscl_faces(q, 0); Fx[1:-1] = np.where(uf[1:-1] > 0, L, R) * uf[1:-1]
        L, R = muscl_faces(q, 1); Fy[:, 1:-1] = np.where(vf[:, 1:-1] > 0, L, R) * vf[:, 1:-1]
    else:
        Fx[1:-1] = np.where(uf[1:-1] > 0, q[:-1], q[1:]) * uf[1:-1]
        Fy[:, 1:-1] = np.where(vf[:, 1:-1] > 0, q[:, :-1], q[:, 1:]) * vf[:, 1:-1]
    return Fx, Fy
def residuals(th0, th1, uf, vf, dt, St, scheme):
    T0 = 1 + DTH * th0; T1 = 1 + DTH * th1
    H0 = DTH * (th0 + f_eq(th0) / St); H1 = DTH * (th1 + f_eq(th1) / St)
    s0 = np.log(T0) + DTH / St * np.interp(T0, Tg, G); s1 = np.log(T1) + DTH / St * np.interp(T1, Tg, G)
    Th, Tc = 1 + DTH, 1 + DTH * thc
    qx = np.zeros((NC + 1, NC)); qy = np.zeros((NC, NC + 1))
    qx[0] = -(T1[0] - (2 * Th - T1[0])) / h; qx[1:-1] = -(T1[1:] - T1[:-1]) / h; qx[-1] = -((2 * Tc - T1[-1]) - T1[-1]) / h
    qy[:, 1:-1] = -(T1[:, 1:] - T1[:, :-1]) / h
    Ax, Ay = transport(H1, uf, vf, scheme); Sx, Sy = transport(s1, uf, vf, scheme)
    rE = (H1 - H0) / dt + div(Ax + qx, Ay + qy)
    Tfx = np.empty((NC + 1, NC)); Tfx[0] = Th; Tfx[-1] = Tc; Tfx[1:-1] = 0.5 * (T1[1:] + T1[:-1])
    Tfy = np.empty((NC, NC + 1)); Tfy[:, 0] = T1[:, 0]; Tfy[:, -1] = T1[:, -1]; Tfy[:, 1:-1] = 0.5 * (T1[:, 1:] + T1[:, :-1])
    sig = (0.5 * (qx[:-1] ** 2 + qx[1:] ** 2) + 0.5 * (qy[:, :-1] ** 2 + qy[:, 1:] ** 2)) / T1 ** 2
    rS = (s1 - s0) / dt + div(Sx + qx / Tfx, Sy + qy / Tfy) - sig
    return rE, rS, T1, s1, H1, div(uf, vf)

out = {}
for solver_adv in ['upwind', 'muscl']:
    for j, (ra, st) in enumerate(test):
        r = run(nc=NC, Ra=ra, Pr=1.0, Ste=st, ws=ws, Cm=1e6, eps_m=1e-3, theta_i=thc, theta_c=thc, t_end=0.25,
                out_times=times, adv=solver_adv)
        sn = r['snaps']; th = [np.full((NC, NC), thc)] + [s['th'] for s in sn]; tt = [0.0] + [s['t'] for s in sn]
        for rs in ['upwind', 'muscl']:
            acc = {k: [] for k in ['nat', 'rec', 'rec_corr', 'proj']}; den = []; dv_rec = []; dv_nat = []; U = []
            for k in range(len(sn)):
                ufn, vfn = sn[k]['uf'], sn[k]['vf']; ufr, vfr = faces_from_centers(sn[k]['u'], sn[k]['v']); ufp, vfp = project(ufr, vfr)
                dt = tt[k + 1] - tt[k]
                rEn, rSn, T1, _, _, _ = residuals(th[k], th[k + 1], ufn, vfn, dt, st, rs)
                den.append((rEn / T1) ** 2); acc['nat'].append((rSn - rEn / T1) ** 2)
                rE, rS, T1, s1, H1, rm = residuals(th[k], th[k + 1], ufr, vfr, dt, st, rs)
                acc['rec'].append((rS - rE / T1) ** 2); acc['rec_corr'].append((rS - (s1 - H1 / T1) * rm - rE / T1) ** 2)
                rE, rS, T1, _, _, _ = residuals(th[k], th[k + 1], ufp, vfp, dt, st, rs); acc['proj'].append((rS - rE / T1) ** 2)
                dv_rec.append(rm ** 2); dv_nat.append(div(ufn, vfn) ** 2); U.append(sn[k]['u'] ** 2 + sn[k]['v'] ** 2)
            d0 = np.sqrt(np.mean(den)); Ur = np.sqrt(np.mean(U))
            row = {k: float(np.sqrt(np.mean(v)) / d0) for k, v in acc.items()}
            row.update(div_rec=float(np.sqrt(np.mean(dv_rec)) / Ur), div_nat=float(np.sqrt(np.mean(dv_nat)) / Ur))
            row['ratio_rec_over_nat'] = row['rec'] / row['nat']
            out[f'solver_{solver_adv}|resid_{rs}|test{j}'] = row
        print(solver_adv, j, {k: round(v, 4) for k, v in out[f'solver_{solver_adv}|resid_upwind|test{j}'].items()}, flush=True)
summ = {}
for sa in ['upwind', 'muscl']:
    for rs in ['upwind', 'muscl']:
        rows = [v for k, v in out.items() if k.startswith(f'solver_{sa}|resid_{rs}|')]
        summ[f'solver_{sa}|resid_{rs}'] = {m: [min(r[m] for r in rows), max(r[m] for r in rows)] for m in rows[0]}
out['summary'] = summ
json.dump(out, open('/home/claude/pof/v2/audit6_muscl.json', 'w'), indent=1)
for k, v in summ.items(): print(k, {m: [round(a, 4) for a in x] for m, x in v.items()})
''')

**`v2/figs_v2a.py`** — Figures 1 and 2.

In [ ]:
writefile('v2/figs_v2a.py', r'''"""Revised Figures 1 (verification) and 2 (identity + macro-step audit)."""
import json, numpy as np
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
plt.rcParams.update({'font.size': 8.5, 'figure.dpi': 150, 'axes.linewidth': 0.7,
                     'lines.linewidth': 1.2, 'legend.frameon': False, 'font.family': 'DejaVu Sans'})
C = ['#1f5fa8', '#d1495b', '#3a8f5f', '#8a5fbf', '#e0a000', '#555555']
P = '/home/claude/pof/'; V = P + 'v2/'
v = json.load(open(P + 'verify1d.json')); mms = json.load(open(V + 'mms_kinetic.json'))
sl = json.load(open(V + 'sharp_limit.json')); r2 = json.load(open(P + 'run2d.json'))
fs = json.load(open(V + 'floors_skip.json'))

# ------------------------------------------------------------------ Figure 1
fig, ax = plt.subplots(2, 3, figsize=(8.6, 5.0), constrained_layout=True)
a = ax[0, 0]
for i, s in enumerate(v['stefan']):
    t = np.array(s['t'])
    a.plot(t, s['front_ana'], '-', color=C[i], label=f"Ste = {s['Ste']:.2f}")
    a.plot(t[::16], np.array(s['front_num'])[::16], 'o', ms=3, mfc='none', color=C[i])
a.set_xlabel('t'); a.set_ylabel(r'front position $x_\Gamma$'); a.legend(fontsize=7)
a.set_title('(a) equilibrium solver vs Neumann', fontsize=8.5)
a = ax[0, 1]; g = v['grid_refinement']
a.loglog([d['N'] for d in g], [d['rel_front_err'] for d in g], 'o-', color=C[0])
a.set_xticks([32, 64, 128, 256, 512], ['32', '64', '128', '256', '512']); a.minorticks_off()
a.set_xlabel('N'); a.set_ylabel('relative front error'); a.set_title('(b) grid refinement (w = 0.01)', fontsize=8.5)
a = ax[0, 2]; w = v['width_refinement']
a.loglog([d['w'] for d in w], [d['rel_front_err'] for d in w], 's-', color=C[1])
a.set_xticks([0.005, 0.01, 0.02, 0.04], ['0.005', '0.01', '0.02', '0.04']); a.minorticks_off()
a.set_xlabel('smoothing width w'); a.set_ylabel('relative front error'); a.set_title('(c) width refinement (N = 512)', fontsize=8.5)
a = ax[1, 0]; m = mms['mms']; Ns = np.array([d['N'] for d in m])
a.loglog(Ns, [d['eT'] for d in m], 'o-', color=C[0], label='T')
a.loglog(Ns, [d['ef'] for d in m], 's-', color=C[1], label='f')
a.loglog(Ns, m[0]['ef'] * (Ns / Ns[0]) ** -2, 'k--', lw=0.8, label='slope -2')
a.set_xticks([32, 64, 128, 256], ['32', '64', '128', '256']); a.minorticks_off()
a.set_xlabel('N'); a.set_ylabel(r'$L^2$ error'); a.legend(fontsize=7)
a.set_title('(d) kinetic solver: manufactured solution', fontsize=8.5)
a = ax[1, 1]; rows = [r for r in sl['rows'] if r['front'] > 0]
a.semilogx([r['M0'] for r in rows], [r['rel_to_neumann'] for r in rows], 'o-', color=C[2])
a.axhline(1.0, color='k', ls='--', lw=0.8)
a.set_xticks([400, 1600, 6400], ['400', '1600', '6400']); a.minorticks_off()
a.set_xlabel(r'mobility $M_0$'); a.set_ylabel(r'$x_\Gamma/x_\Gamma^{\mathrm{Neumann}}$ at $t=0.1$')
a.set_title('(e) approach to the sharp-interface limit', fontsize=8.5)
ab = a.twinx(); ab.semilogx([r['M0'] for r in rows], [r['T_interface'] - 1 for r in rows], 's--', color=C[3]); ab.minorticks_off()
ab.set_ylabel(r'interface superheat $T_\Gamma-T_m$', color=C[3])
a = ax[1, 2]; f0 = mms['f0_sensitivity']
a.semilogx([d['f0'] for d in f0], [d['front'] for d in f0], 'o-', color=C[4])
a.set_xlabel(r'seed fraction $f_0$'); a.set_ylabel(r'$x_\Gamma$ at $t=0.1$')
a.set_title(r'(f) sensitivity to the nucleation seed', fontsize=8.5)
fig.savefig(V + 'fig1_verification.png', bbox_inches='tight'); plt.close(fig)

# ------------------------------------------------------------------ Figure 2
fig, ax = plt.subplots(1, 3, figsize=(8.8, 2.8), constrained_layout=True)
a = ax[0]; pts = v['identity_manufactured']['points']
dx = np.array([p['dx'] for p in pts]); e = np.array([p['eps_id_rel'] for p in pts])
a.loglog(dx, e, 'o-', color=C[0], label='manufactured fields')
a.loglog(dx, e[0] * (dx / dx[0]) ** 2, 'k--', lw=0.8, label='slope 2')
a.set_xlabel(r'$\Delta x$ ($\Delta t=0.5\Delta x^2$)'); a.set_ylabel(r'$\|r_S-r_E/T\|/\|r_E/T\|$')
a.legend(fontsize=7); a.set_title('(a) 1D identity defect', fontsize=8.5)
a = ax[1]; s1 = v['identity_on_solver']; i2 = r2['identity_2d']
a.loglog([d['N'] for d in s1], [d['eps_id_termrel_median'] for d in s1], 'o-', color=C[1], label='1D solver pairs')
a.loglog([d['nc'] for d in i2], [d['eps_id_termrel_median'] for d in i2], 's-', color=C[2], label='2D convective pairs')
a.set_xticks([48, 96, 128, 256, 512], ['48', '96', '128', '256', '512']); a.minorticks_off()
a.set_xlabel('cells per side'); a.set_ylabel(r'$\|r_S-r_E/T\|$ / storage norm'); a.legend(fontsize=7)
a.set_title('(b) defect on consecutive solver steps', fontsize=8.5)
a = ax[2]
labels = [r'$r_f$, endpoint', r'$r_f$, trapezoid', r'$r_E$ (kin.), endpoint', r'$r_E$ (kin.), trapezoid', r'$r_E$ (eq.), endpoint', r'$r_E$ (eq.), trapezoid']
vals_dt = [fs['kin_end_skip2']['F0_rel'], fs['kin_trap_skip2']['F0_rel'], fs['kin_end_skip2']['E0_rel'], fs['kin_trap_skip2']['E0_rel'],
           fs['eq_end_skip2']['E0_rel'], fs['eq_trap_skip2']['E0_rel']]
vals_h = [fs['kin_half_end_skip2']['F0_rel'], fs['kin_half_trap_skip2']['F0_rel'], fs['kin_half_end_skip2']['E0_rel'],
          fs['kin_half_trap_skip2']['E0_rel'], np.nan, np.nan]
y = np.arange(len(labels))
a.barh(y + 0.2, vals_dt, height=0.38, color=C[0], label=r'$\Delta t_{\mathrm{macro}}$')
a.barh(y - 0.2, vals_h, height=0.38, color=C[4], label=r'$\Delta t_{\mathrm{macro}}/2$')
a.set_xscale('log'); a.set_yticks(y, labels, fontsize=7.5); a.invert_yaxis()
a.set_xlabel('residual floor on exact pairs / rate scale'); a.legend(fontsize=7, loc='upper left', bbox_to_anchor=(1.0, 1.0))
a.set_title('(c) macro-step truncation floors', fontsize=8.5)
fig.savefig(V + 'fig2_identity_macro.png', bbox_inches='tight'); plt.close(fig)
print('figs 1, 2 written')
''')

**`v2/figs_v2b.py`** — Figures 3–6.

In [ ]:
writefile('v2/figs_v2b.py', r'''"""Revised Figures 3-7."""
import json, glob, pickle, numpy as np
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
plt.rcParams.update({'font.size': 8.5, 'figure.dpi': 150, 'axes.linewidth': 0.7,
                     'lines.linewidth': 1.2, 'legend.frameon': False, 'font.family': 'DejaVu Sans'})
V = '/home/claude/pof/v2/'; P = '/home/claude/pof/'
C = {'D': '#7f7f7f', 'E': '#1f5fa8', 'ES': '#d1495b', 'EET': '#3a8f5f', 'EPH': '#8a5fbf', 'EK': '#e0a000'}
LBL = {'D': 'D', 'E': 'E', 'ES': 'E+S', 'EET': 'E+E/T', 'EPH': 'E+PH', 'EK': 'E+K'}
S = json.load(open(V + 'stats_v2.json')); A2 = json.load(open(V + 'analysis2.json'))

def load(pattern):
    runs = {}
    for f in glob.glob(pattern):
        d = json.load(open(f)); runs[d['seed']] = d
    return [runs[s] for s in sorted(runs)]
eq = {v: load(f'{P}runs/eq_{v}_s[0-3].json') + load(f'{V}runs/eq_{v}_s[4-7].json') for v in ['D', 'E', 'ES', 'EET']}
kn = {v: load(f'{V}runs/kin_{v}_s[0-7].json') for v in ['D', 'E', 'EPH', 'EK']}
seedmean = lambda runs, m: np.array([np.mean(r['per_traj'][m]) for r in runs])

def dotpanel(ax, groups, m, title, ylab=None):
    for i, v in enumerate(groups):
        y = seedmean(groups[v], m)
        ax.scatter(np.full(len(y), i) + np.linspace(-0.12, 0.12, len(y)), y, s=11, color=C[v], alpha=0.7, zorder=3)
        ax.hlines(y.mean(), i - 0.3, i + 0.3, color=C[v], lw=2)
    ax.set_xticks(range(len(groups)), [LBL[v] for v in groups]); ax.set_title(title, fontsize=8.5)
    if ylab: ax.set_ylabel(ylab)
    ax.margins(y=0.2)

def forest(ax, rows, title):
    for i, (lab, o, margin) in enumerate(rows):
        base = o['base_mean']
        mu, lo, hi = 100 * o['mean'] / base, 100 * o['ci95'][0] / base, 100 * o['ci95'][1] / base
        ax.errorbar(mu, i, xerr=[[mu - lo], [hi - mu]], fmt='o', color='k', ms=3.5, capsize=2)
        if margin is not None:
            mm = 100 * margin / base
            ax.plot([-mm, mm], [i, i], '|', color='#d1495b', ms=10, mew=1.5)
    ax.axvline(0, color='k', lw=0.6, ls=':')
    ax.set_yticks(range(len(rows)), [r[0] for r in rows], fontsize=7.5); ax.invert_yaxis()
    ax.set_xlabel('paired difference (% of baseline), 95% CI over seeds'); ax.set_title(title, fontsize=8.5)

# ------------------------------------------------------------------ Figure 3 (equilibrium 1D)
fig, ax = plt.subplots(1, 4, figsize=(9.4, 2.7), constrained_layout=True, gridspec_kw={'width_ratios': [1, 1, 1, 1.6]})
dotpanel(ax[0], eq, 'relT', 'rollout rel. error of T', 'seed mean (8 seeds)')
dotpanel(ax[1], eq, 'front', 'front error')
dotpanel(ax[2], eq, 'rE', r'rollout $\|r_E\|$ / floor')
pr = S['primary']; sc = S['secondary']
forest(ax[3], [('E+S vs E+E/T: T err.', pr['P1'], pr['P1']['margin']), ('E+S vs E+E/T: front', pr['P2'], pr['P2']['margin']),
               ('E vs D: T err.', sc['eq_E-D_relT'], None), ('E vs D: front', sc['eq_E-D_front'], None),
               ('E+S vs E: front', sc['eq_ES-E_front'], None), ('E+E/T vs E: $r_E$', sc['eq_EET-E_rE'], None)],
       'contrasts (red: equivalence margin)')
fig.savefig(V + 'fig3_ablation_eq.png', bbox_inches='tight'); plt.close(fig)

# ------------------------------------------------------------------ Figure 4 (internal-variable 1D)
fig, ax = plt.subplots(1, 4, figsize=(9.4, 2.7), constrained_layout=True, gridspec_kw={'width_ratios': [1, 1, 1.2, 1.6]})
dotpanel(ax[0], kn, 'relf', 'rollout rel. error of f', 'seed mean (8 seeds)')
dotpanel(ax[1], kn, 'rf', r'rollout $\|R_f\|$ / floor')
a = ax[2]; etas = [0.0, 0.01, 0.1, 1.0]; keys = ['Pm_0', 'Pm_001', 'Pm_01', 'Pm_1']
for v in kn:
    y = [np.mean([np.mean(r['per_traj'][k if k != 'Pm_0' else 'Pm_0']) for r in kn[v]]) for k in keys]
    a.semilogy(range(4), np.maximum(y, 1e-6), 'o-', color=C[v], label=LBL[v], ms=3)
yr = [np.mean([np.mean(r['per_traj']['ref_' + k]) for r in kn['E']]) for k in keys]
yr = np.array(yr); ok = yr > 0
a.semilogy(np.arange(4)[ok], yr[ok], 'k*', ms=6, label='reference (others 0)')
a.set_xticks(range(4), ['0', r'$0.01\Sigma_0$', r'$0.1\Sigma_0$', r'$\Sigma_0$']); a.set_xlabel(r'threshold $\eta$')
a.set_ylabel(r'$P_-(\eta)$'); a.legend(fontsize=6.5, ncol=2); a.set_title('negative phase dissipation', fontsize=8.5)
forest(ax[3], [('E+K vs E: f err. (P3)', pr['P3'], None), ('E+K vs E: T err. (P4)', pr['P4'], None),
               ('E+K vs E: $R_f$', sc['kin_EK-E_rf'], None), ('E+K vs E: $r_E$', sc['kin_EK-E_rE'], None),
               ('E+PH vs E: f err.', sc['kin_EPH-E_relf'], None), ('E+K vs D: $r_E$', sc['kin_EK-D_rE'], None),
               ('E+K vs E: $P_-(0)$', sc['kin_EK-E_sphneg'], None)], 'contrasts')
fig.savefig(V + 'fig4_ablation_kin.png', bbox_inches='tight'); plt.close(fig)

# ------------------------------------------------------------------ Figure 5 (robustness + tuning)
fig, ax = plt.subplots(1, 3, figsize=(9.0, 2.7), constrained_layout=True)
a = ax[0]; au = A2['audit']; tags = [('_end_s01', 'endpoint'), ('_trap', 'trapezoid'), ('_half', r'$\Delta t_{\mathrm{macro}}/2$'), ('_M1600', r'$M_0=1600$')]
for i, (t, lab) in enumerate(tags):
    for j, m in enumerate(['relf', 'relT']):
        d = au[t][m]; vals = 100 * np.array(d['seed_diffs']) / d['base']
        a.scatter(np.full(2, i + (j - 0.5) * 0.25), vals, color=['#e0a000', '#1f5fa8'][j], s=16, label=(['f error', 'T error'][j] if i == 0 else None))
a.axhline(0, color='k', lw=0.6, ls=':'); a.set_xticks(range(4), [l for _, l in tags], fontsize=7.5)
a.set_ylabel('E+K minus E (% of E), per seed'); a.legend(fontsize=7); a.set_title('(a) robustness of the kinetic-loss effect', fontsize=8.5)
for k, (clos, vs) in enumerate([('eq', ['E', 'ES', 'EET']), ('kin', ['E', 'EPH', 'EK'])]):
    a = ax[1 + k]
    for v in vs:
        rows = A2['tuning'][f'{clos}_{v}']['rows']
        a.semilogx([r['lam'] for r in rows], [r['val'] for r in rows], 'o-', color=C[v], label=LBL[v], ms=3.5)
    a.set_xlabel(r'penalty weight $\lambda$'); a.set_ylabel('validation data loss (seed 0)'); a.legend(fontsize=7)
    a.set_title(f"({'b' if k == 0 else 'c'}) equal-budget tuning, {'equilibrium' if k == 0 else 'internal-variable'}", fontsize=8.5)
fig.savefig(V + 'fig5_robustness.png', bbox_inches='tight'); plt.close(fig)

# ------------------------------------------------------------------ Figure 6 (2D solver audit)
snaps = pickle.load(open(V + 'snaps96.pkl', 'rb')); R = json.load(open(V + 'run2d_v2.json'))['runs']
R['up192'] = json.load(open(V + 'run2d_192.json')); v1 = json.load(open(P + 'run2d.json'))
fig = plt.figure(figsize=(9.0, 5.4), constrained_layout=True)
gs = fig.add_gridspec(2, 3, height_ratios=[1.35, 1])
xc = (np.arange(96) + 0.5) / 96
for k, idx in enumerate([0, 2, 4]):
    s = snaps[idx]; a = fig.add_subplot(gs[0, k]); X, Y = np.meshgrid(xc, xc, indexing='ij')
    cf = a.contourf(X, Y, s['th'], levels=np.linspace(-0.1, 1.0, 12), cmap='RdBu_r')
    a.contour(X, Y, s['f'], levels=[0.5], colors='k', linewidths=1.4)
    um = np.where(s['f'] > 0.5, s['u'], np.nan); vm = np.where(s['f'] > 0.5, s['v'], np.nan)
    a.streamplot(xc, xc, um.T, vm.T, density=0.7, color='w', linewidth=0.5, arrowsize=0.5)
    a.set_title(f"t = {s['t']:.2f} ($96^2$)", fontsize=8.5); a.set_aspect('equal'); a.set_xticks([0, 0.5, 1]); a.set_yticks([0, 0.5, 1])
fig.colorbar(cf, ax=fig.axes[:3], shrink=0.8, label=r'$\theta$')
a = fig.add_subplot(gs[1, 0])
for key, lab, st in [('up48', r'$48^2$', 'o-'), ('up96', r'$96^2$', 's-'), ('up192', r'$192^2$', '^-'), ('mu96', r'$96^2$ MUSCL', 'x--'), ('up96_cfl015', r'$96^2$, $\Delta t/2$', 'd:')]:
    a.plot(R[key]['t'], R[key]['Nu'], st, ms=3, label=lab)
a.set_xlabel('t'); a.set_ylabel('Nu (hot wall)'); a.legend(fontsize=6.5); a.set_title('(d) Nusselt number', fontsize=8.5)
a = fig.add_subplot(gs[1, 1]); hs = [1 / 48, 1 / 96, 1 / 192]
for q, lab, c in [('front_top', 'front, y = 0.95', '#d1495b'), ('front_mid', 'front, y = 0.5', '#1f5fa8'), ('front_bot', 'front, y = 0.05', '#3a8f5f')]:
    a.plot(hs, [R[k][q][-1] for k in ['up48', 'up96', 'up192']], 'o-', color=c, label=lab)
a.plot(hs, [R[k]['fbar'][-1] for k in ['up48', 'up96', 'up192']], 'k^--', label=r'$\bar f$')
a.set_xscale('log'); a.set_xticks(hs, ['1/48', '1/96', '1/192']); a.minorticks_off()
a.set_xlabel('grid spacing h'); a.set_ylabel('value at t = 0.25'); a.legend(fontsize=6.5); a.set_title('(e) grid refinement', fontsize=8.5)
a = fig.add_subplot(gs[1, 2])
labs, vals_f, vals_n = [], [], []
for d in v1['cm_sensitivity']:
    labs.append(f"$C_m$={d['Cm']:.0e}"); vals_f.append(d['front_mid']); vals_n.append(d['Nu'])
for key, lab in [('ws0.01', '$w_s$=0.01'), ('ws0.04', '$w_s$=0.04'), ('eps0.0001', r'$\varepsilon_m$=1e-4'), ('eps0.01', r'$\varepsilon_m$=1e-2')]:
    labs.append(lab); vals_f.append(R[key]['front_mid'][-1]); vals_n.append(R[key]['Nu'][-1])
y = np.arange(len(labs))
a.barh(y, vals_n, color='#8a5fbf'); a.set_yticks(y, labs, fontsize=7); a.invert_yaxis()
a.set_xlabel('Nu at t = 0.25 ($64^2$)'); a.set_xlim(5.0, 7.3); a.set_title('(f) parameter sensitivity', fontsize=8.5)
fig.savefig(V + 'fig6_2d_solver.png', bbox_inches='tight'); plt.close(fig)

# ------------------------------------------------------------------ Figure 7 (2D operators, weight dependence)
d2 = np.load(V + 'data_2d.npz'); ref = d2['test_th'][0]
AC = json.load(open(V + 'active2d.json'))
fig = plt.figure(figsize=(9.0, 4.8), constrained_layout=True)
gs = fig.add_gridspec(2, 4)
xc32 = (np.arange(32) + 0.5) / 32; X, Y = np.meshgrid(xc32, xc32, indexing='ij')
for k, (lab, th) in enumerate([('reference', ref[-1]), ('D', np.load(V + 'ckpt2d/D_s1_rollout0.npz')['th'][-1]),
                                ('E+S, $\\lambda=10^{-4}$', np.load(V + 'ckpt2d_r4/ES_p0_lam0.0001_s1_rollout0.npz')['th'][-1]),
                                ('E+E/T, $\\lambda=10^{-4}$', np.load(V + 'ckpt2d_r4/EET_p0_lam0.0001_s1_rollout0.npz')['th'][-1])]):
    a = fig.add_subplot(gs[0, k]); a.contourf(X, Y, np.clip(th, -0.1, 1.0), levels=np.linspace(-0.1, 1.0, 12), cmap='RdBu_r')
    a.contour(X, Y, 0.5 * (1 + np.tanh(th / 0.02)), levels=[0.5], colors='k', linewidths=1.2)
    if k > 0: a.contour(X, Y, 0.5 * (1 + np.tanh(ref[-1] / 0.02)), levels=[0.5], colors='y', linewidths=0.9, linestyles='--')
    a.set_title(f'{lab}, t = 0.25' + (' (seed 1)' if k else ''), fontsize=8); a.set_aspect('equal'); a.set_xticks([]); a.set_yticks([])
lams = [1e-4, 1e-2, 1e-1]; lk = ['1e-4', '1e-2', '1e-1']; T = AC['table']
for k, (m, t) in enumerate([('relT', r'(e) rollout rel. error of $\theta$'), ('nu', '(f) rel. Nu error')]):
    a = fig.add_subplot(gs[1, k])
    a.axhline(T['D@0'][m], color=C['D'], ls='--', lw=1, label='D ($\\lambda=0$)')
    for v in ['E', 'ES', 'EET']:
        a.semilogx(lams, [T[f'{v}@{l}'][m] for l in lk], 'o-', color=C[v], label=LBL[v], ms=3.5)
    a.set_xlabel(r'penalty weight $\lambda$'); a.set_title(t, fontsize=8.5); a.legend(fontsize=6.5)
a = fig.add_subplot(gs[1, 2])
for v, ls in [('E', '-'), ('ES', '--'), ('EET', ':')]:
    for l, c in [('1e-2', '#1f5fa8'), ('1e-1', '#d1495b')]:
        g = AC['grad_ratio'][f'{v}@{l}']
        a.semilogy(g['steps'], g['ratio'], ls, color=c, label=f'{LBL[v]}, $\\lambda$={l}')
a.set_xlabel('training step'); a.set_ylabel('physics / data gradient norm'); a.set_title('(g) loss balance during training', fontsize=8.5)
a.legend(fontsize=5.5, ncol=1)
a = fig.add_subplot(gs[1, 3])
for i, l in enumerate(['1e-4', '1e-2', '1e-1']):
    if l == '1e-4':
        o = A2['2d_contrasts']['ES-EET_relT']; base = AC['table']['EET@1e-4']['relT']
        mu = 100 * o['rel']; lo, hi = 100 * o['ci'][0] / base, 100 * o['ci'][1] / base
    else:
        o = AC['contrasts'][f'ES-EET@{l}_relT']; mu = 100 * o['rel']; lo, hi = 100 * o['ci_rel'][0], 100 * o['ci_rel'][1]
    a.errorbar(i, mu, yerr=[[mu - lo], [hi - mu]], fmt='o', color='k', capsize=3)
a.axhline(0, color='k', lw=0.6, ls=':'); a.set_xticks(range(3), [r'$10^{-4}$', r'$10^{-2}$', r'$10^{-1}$'])
a.set_xlabel(r'$\lambda$'); a.set_ylabel(r'E+S $-$ E+E/T, $\theta$ error (%)'); a.set_title('(h) entropy vs weighted energy', fontsize=8.5)
fig.savefig(V + 'fig7_2d_operator.png', bbox_inches='tight'); plt.close(fig)
print('figs 3-7 written')
''')

**`v2/figs_r4.py`** — Figures 7 and 8.

In [ ]:
writefile('v2/figs_r4.py', r'''"""Round-4 figures: revised Fig. 7 (per-seed points) and new Fig. 8 (continuity decomposition)."""
import json, numpy as np
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
plt.rcParams.update({'font.size': 8.5, 'figure.dpi': 150, 'axes.linewidth': 0.7, 'axes.spines.top': False, 'axes.spines.right': False})
V = '/home/claude/pof/v2/'
C = {'D': '#7f7f7f', 'E': '#1f5fa8', 'ES': '#d1495b', 'EET': '#3a8f5f', 'ESC': '#e0a000'}
LBL = {'D': 'D', 'E': 'E', 'ES': 'E+S', 'EET': 'E+E/T', 'ESC': 'E+S$_c$'}
A = json.load(open(V + 'active2d_r4.json')); T = A['table']; CT = A['contrasts']
AC = json.load(open(V + 'active2d.json')); A2 = json.load(open(V + 'analysis2.json'))
# ---------------- Fig 7
d2 = np.load(V + 'data_2d.npz'); ref = d2['test_th'][0]
fig = plt.figure(figsize=(9.0, 4.9), constrained_layout=True); gs = fig.add_gridspec(2, 4)
xc = (np.arange(32) + 0.5) / 32; X, Y = np.meshgrid(xc, xc, indexing='ij')
for k, (lab, th) in enumerate([('(a) reference', ref[-1]), ('(b) D', np.load(V + 'ckpt2d/D_s1_rollout0.npz')['th'][-1]),
                                ('(c) E+S, $\\lambda=10^{-4}$', np.load(V + 'ckpt2d_r4/ES_p0_lam0.0001_s1_rollout0.npz')['th'][-1]),
                                ('(d) E+E/T, $\\lambda=10^{-4}$', np.load(V + 'ckpt2d_r4/EET_p0_lam0.0001_s1_rollout0.npz')['th'][-1])]):
    a = fig.add_subplot(gs[0, k]); a.contourf(X, Y, np.clip(th, -0.1, 1.0), levels=np.linspace(-0.1, 1.0, 12), cmap='RdBu_r')
    a.contour(X, Y, 0.5 * (1 + np.tanh(th / 0.02)), levels=[0.5], colors='k', linewidths=1.2)
    if k > 0: a.contour(X, Y, 0.5 * (1 + np.tanh(ref[-1] / 0.02)), levels=[0.5], colors='y', linewidths=0.9, linestyles='--')
    a.set_title(f'{lab}, t = 0.25' + (' (seed 1)' if k else ''), fontsize=7.5); a.set_aspect('equal'); a.set_xticks([]); a.set_yticks([])
lams = [1e-4, 1e-2, 1e-1]; lk = ['1e-4', '1e-2', '1e-1']
for k, (m, t) in enumerate([('relT', r'(e) rollout rel. error of $\theta$'), ('nu', '(f) rel. Nu error')]):
    a = fig.add_subplot(gs[1, k]); dref = T['D|p0|0']
    a.axhspan(min(dref['seeds'][m]), max(dref['seeds'][m]), color=C['D'], alpha=0.15, lw=0)
    a.axhline(dref[m], color=C['D'], ls='--', lw=1, label='D (range of seeds)')
    for i, v in enumerate(['E', 'ES', 'EET']):
        mu = [T[f'{v}|p0|{l}'][m] for l in lk]
        a.semilogx(lams, mu, '-', color=C[v], lw=1, label=LBL[v])
        for l, lam in zip(lk, lams):
            a.semilogx([lam * (1 + 0.12 * (i - 1))] * 3, T[f'{v}|p0|{l}']['seeds'][m], 'o', color=C[v], ms=2.5, alpha=0.8)
    a.set_xlabel(r'penalty weight $\lambda$'); a.set_title(t, fontsize=8.5); a.legend(fontsize=6)
a = fig.add_subplot(gs[1, 2])
for v, ls in [('E', '-'), ('ES', '--'), ('EET', ':')]:
    for l, c in [('1e-2', '#1f5fa8'), ('1e-1', '#d1495b')]:
        g = AC['grad_ratio'][f'{v}@{l}']; a.semilogy(g['steps'], g['ratio'], ls, color=c, label=f'{LBL[v]}, $\\lambda$={l}')
a.set_xlabel('training step'); a.set_ylabel('physics / data gradient norm'); a.set_title('(g) loss balance (seed mean)', fontsize=8.5)
a.legend(fontsize=5.5)
a = fig.add_subplot(gs[1, 3])
for i, l in enumerate(lk):
    if l == '1e-4':
        o = A2['2d_contrasts']['ES-EET_relT']; base = AC['table']['EET@1e-4']['relT']
        mu = 100 * o['rel']; lo, hi = 100 * o['ci'][0] / base, 100 * o['ci'][1] / base; sr = None
    else:
        o = CT[f'ES-EET@{l}_relT']; mu = 100 * o['rel']; lo, hi = 100 * o['ci_rel'][0], 100 * o['ci_rel'][1]; sr = o['seed_rel']
    a.errorbar(i, mu, yerr=[[mu - lo], [hi - mu]], fmt='o', color='k', capsize=3)
    if sr: a.plot([i + 0.15] * 3, 100 * np.array(sr), '.', color='0.5', ms=4)
a.axhline(0, color='k', lw=0.6, ls=':'); a.set_xticks(range(3), [r'$10^{-4}$', r'$10^{-2}$', r'$10^{-1}$'])
a.set_xlabel(r'$\lambda$'); a.set_ylabel(r'E+S $-$ E+E/T, $\theta$ error (%)'); a.set_title('(h) entropy vs weighted energy', fontsize=8.5)
fig.savefig(V + 'fig7_2d_operator.png', bbox_inches='tight'); plt.close(fig)
# ---------------- Fig 8
AV = json.load(open(V + 'audit4_velocity.json')); tr = sorted(AV)
fig, ax = plt.subplots(2, 2, figsize=(7.0, 5.6), constrained_layout=True); ax = ax.flatten()
a = ax[0]; x = np.arange(len(tr))
for k, (f, key, lab, c, mk) in enumerate([('native', 'ident', 'native MAC', '#1f5fa8', 'o'), ('reconstructed', 'ident', 'reconstructed', '#d1495b', 's'),
        ('reconstructed', 'ident_corrected', 'reconstructed, $-(s-h/T)r_m$', '#e0a000', '^'), ('recon_projected', 'ident', 'reconstructed + projected', '#3a8f5f', 'v')]):
    a.plot(x + 0.12 * (k - 1.5), [AV[t][f][key] for t in tr], mk, color=c, ms=4, label=lab)
a.set_xticks(x, [str(i) for i in x]); a.set_xlabel('test trajectory'); a.set_ylabel(r'$\|r_S-r_E/T\|\,/\,S_0$'); a.set_yscale('log')
a.set_title('(a) identity defect, exact pairs', fontsize=8.5); a.legend(fontsize=5.5)
a = ax[1]
a.bar(x - 0.2, [max(AV[t]['native']['div_rms_dimless'], 1e-7) for t in tr], 0.4, color='#1f5fa8', label='native MAC')
a.bar(x + 0.2, [AV[t]['reconstructed']['div_rms_dimless'] for t in tr], 0.4, color='#d1495b', label='reconstructed')
a.set_yscale('log'); a.set_xticks(x, [str(i) for i in x]); a.set_xlabel('test trajectory'); a.set_ylabel(r'$\|\nabla_h\!\cdot\mathbf{u}\|\,H/U_{\mathrm{rms}}$')
a.set_title('(b) discrete divergence', fontsize=8.5); a.legend(fontsize=6)
a = ax[2]; names = [('ES-EET', 'E+S $-$ E+E/T'), ('ESC-EET', 'E+S$_c$ $-$ E+E/T'), ('ESp-EETp', 'projected:\nE+S $-$ E+E/T')]
for j, l in enumerate(['1e-2', '1e-1']):
    for i, (nm, lab) in enumerate(names):
        key = f'{nm}@{l}_relT'
        if key not in CT: continue
        o = CT[key]; xx = i + (j - 0.5) * 0.3; mu = 100 * o['rel']
        a.errorbar(xx, mu, yerr=[[mu - 100 * o['ci_rel'][0]], [100 * o['ci_rel'][1] - mu]], fmt='o' if j == 0 else 's',
                   color='#1f5fa8' if j == 0 else '#d1495b', capsize=2, ms=4, label=(f'$\\lambda$={l}' if i == 0 else None))
        a.plot([xx + 0.07] * 3, 100 * np.array(o['seed_rel']), '.', color='0.5', ms=3)
a.axhline(0, color='k', lw=0.6, ls=':'); a.set_xticks(range(3), [n[1] for n in names], fontsize=6.5)
a.set_ylabel(r'$\theta$ error difference (%)'); a.set_title('(c) matched contrasts', fontsize=8.5); a.legend(fontsize=6)
a = ax[3]; G = A['gauge']; keys = [k for k in G if k.startswith('ref')]; pk = [k for k in G if not k.startswith('ref')]
for i, term in enumerate(['E', 'S', 'EET']):
    a.plot([i - 0.1] * len(keys), [max(G[k][term]) for k in keys], 'o', color='#d1495b', ms=3.5, label='reference pairs' if i == 0 else None)
    if pk: a.plot([i + 0.1] * len(pk), [max(G[k][term]) for k in pk], 's', color='#1f5fa8', ms=3.5, label='E+S$_c$ predictions' if i == 0 else None)
a.axhline(1, color='k', lw=0.6, ls=':'); a.set_xticks(range(3), [r'$\mathcal{L}_E$', r'$\mathcal{L}_S$', r'$\mathcal{L}_{E/T}$']); a.set_yscale('log')
a.set_ylabel('loss ratio, shifted / unshifted'); a.set_title('(d) reference-state shift', fontsize=8.5); a.legend(fontsize=6)
fig.savefig(V + 'fig8_continuity.png', bbox_inches='tight'); plt.close(fig); print('figs written')
''')

**`v2/verify_headlines.py`** — Recomputes the headline numbers from the saved records.

In [ ]:
writefile('v2/verify_headlines.py', r'''"""Recompute headline numbers of the manuscript from saved archive outputs only (no training, no solver).
Usage: python verify_headlines.py ARCHIVE_ROOT  -> prints each quantity, its recomputed value, and the value in the text."""
import json, sys, os, itertools, numpy as np
from scipy import stats
R = sys.argv[1] if len(sys.argv) > 1 else '.'
J = lambda p: json.load(open(os.path.join(R, p)))
rows = []
def chk(name, val, text, tol):
    ok = abs(val - text) <= tol; rows.append(ok); print(f"{'OK ' if ok else 'BAD'} {name}: {val:.4g} (text {text})")
# --- 1D primary equivalence from exported paired arrays
z = np.load(os.path.join(R, 'stats_export/eq_per_trajectory.npz'))
def tost_ci(D, alpha):
    ms = D.mean(1); S = len(ms); se = ms.std(ddof=1) / np.sqrt(S); q = stats.t.ppf(1 - alpha, S - 1)
    return ms.mean() - q * se, ms.mean() + q * se
lo, hi = tost_ci(z['ES__relT'] - z['EET__relT'], 0.0125); chk('P1 Holm-level TOST lower', lo, -9.3e-5, 5e-7); chk('P1 upper', hi, 4.2e-5, 5e-7)
lo, hi = tost_ci(z['ES__front'] - z['EET__front'], 0.05 / 3); chk('P2 lower', lo, -3.7e-4, 5e-6); chk('P2 upper', hi, 4.4e-5, 5e-7)
post = z['seeds'] >= 4
lo, hi = tost_ci((z['ES__relT'] - z['EET__relT'])[post], 0.0125); chk('P1 post-plan lower', lo, -8.4e-5, 5e-7)
# --- 2D matched contrasts from per-run JSON records
def run(v, p, lam, s):
    if v == 'D': return J(f'runs2d_active/D_s{s}_logged.json')
    if p == 1 or v == 'ESC': return J(f'runs2d_r4/{v}_p{p}_s{s}_lam{lam}.json')
    return J(f'runs2d_active/{v}_s{s}_lam{lam}.json')
def contrast(A, B, m='relT'):
    D = np.array([np.mean(np.array(run(*A, s)['per_traj'][m]) - np.array(run(*B, s)['per_traj'][m])) for s in range(3)])
    base = np.mean([np.mean(run(*B, s)['per_traj'][m]) for s in range(3)]); se = D.std(ddof=1) / np.sqrt(3); q = stats.t.ppf(0.975, 2)
    return 100 * D.mean() / base, 100 * (D.mean() - q * se) / base, 100 * (D.mean() + q * se) / base
c = contrast(('ES', 0, '1e-2'), ('EET', 0, '1e-2')); chk('ES-EET @1e-2 (%)', c[0], -4.4, 0.05); chk('  lower', c[1], -6.9, 0.05)
c = contrast(('ESC', 0, '1e-2'), ('EET', 0, '1e-2')); chk('ESC-EET @1e-2 (%)', c[0], 0.0, 0.05); chk('  upper', c[2], 0.4, 0.05)
c = contrast(('ES', 1, '1e-2'), ('EET', 1, '1e-2')); chk('projected ES-EET @1e-2 (%)', c[0], -0.2, 0.05); chk('  lower', c[1], -2.3, 0.05)
c = contrast(('E', 0, '1e-1'), ('D', 0, '0'), 'nu'); chk('E-D Nu @1e-1 (%)', c[0], 87, 0.5); chk('  lower', c[1], 50, 0.5)
# --- audits
V = J('audit4_velocity.json')
chk('native defect max', max(V[t]['native']['ident'] for t in V), 0.108, 5e-4)
chk('reconstructed defect max', max(V[t]['reconstructed']['ident'] for t in V), 1.14, 5e-3)
chk('reconstructed div min (U/H)', min(V[t]['reconstructed']['div_rms_dimless'] for t in V), 0.80, 5e-3)
D5 = J('audit5_dg.json'); chk('DG threshold defect', D5['threshold_1e-09']['defect_abs_max'], 2.9e-13, 1e-14)
F = J('audit5_front.json'); chk('front width median (cells)', F['width_cells_q'][2], 5.7, 0.05)
chk('front profiles excluded', F['n_excluded_missing_0p1_or_0p9'], 4, 0)
chk('DG defect, threshold 1e-6', D5['threshold_1e-06']['defect_abs_max'], 5.4e-13, 1e-14)
chk('DG stable defect', D5['stable']['defect_abs_max'], 5.8e-13, 1e-14)
M6 = J('audit6_muscl.json')['summary']
chk('MUSCL: min reconstructed/native defect ratio', min(v['ratio_rec_over_nat'][0] for v in M6.values()), 5.9, 0.05)
chk('MUSCL: max reconstructed/native defect ratio', max(v['ratio_rec_over_nat'][1] for v in M6.values()), 15.3, 0.05)
S5 = J('stats_round5.json'); chk('post-plan Holm p P2', S5['P2']['p_holm'], 0.017, 5e-4)
print(f'{sum(rows)}/{len(rows)} checks agree with the manuscript')
''')

## 2. Datasets (`RUN_DATA`)

The one-dimensional sets (Table II) are split by parameter combination with shuffle seed 0, and the two-dimensional set with seed 1. The package ships the datasets, so this stage is optional.

In [ ]:
if RUN_DATA:
    run_py('datagen.py', cwd=ROOT)        # data_eq.npz, data_kin.npz
    run_py('v2/datagen2.py')              # half macro-step and M0 = 100, 1600 internal-variable sets
    run_py('v2/datagen2d.py')             # 32^2 convective set (30 runs)
else:
    print('using shipped datasets')

## 3. Solvers and verification

`RUN_FAST_CHECKS` covers the manufactured solution, the mobility and seed sweeps (Fig. 1d–f), the discrete-gradient checks, and the velocity-reconstruction audit. `RUN_SOLVERS` adds the one-dimensional Neumann and identity studies, the de Vahl Davis cavity, and the two-dimensional melting runs (Table V, Fig. 6), including the $192^2$ grid, which runs in restartable chunks.

In [ ]:
if RUN_SOLVERS:
    run_py('verify1d.py', cwd=ROOT)                       # verify1d.json (Fig. 1a-c, Fig. 2a,b)
    run_py('run2d.py', cwd=ROOT)                          # run2d.json (C_m sweep, 2D identity points)
    run_py('v2/make_vdd.py')                              # vdd.json (de Vahl Davis, Ra = 1e5)
    run_py('v2/run2d_v2.py')                              # run2d_v2.json, snaps96.pkl
    while not os.path.exists(os.path.join(V, 'run2d_192.json')):
        run_py('v2/run2d_192_chunk.py', '3000')           # 50-minute chunks of the 192^2 run
else:
    print('using shipped solver outputs: verify1d.json, run2d.json, vdd.json, v2/run2d_v2.json, v2/run2d_192.json, v2/snaps96.pkl')

In [ ]:
if RUN_FAST_CHECKS:
    run_py('v2/mms_kinetic.py')            # Appendix B, Fig. 1d,f
    run_py('v2/make_sharp_limit.py')       # Fig. 1e
    run_py('v2/discrete_gradient_check.py')
    run_py('v2/corruption_v2.py')          # Sec. VI B
    run_py('v2/audit4_velocity.py')        # Sec. VI C, Fig. 7a,b
    run_py('v2/audit5_precision.py')       # Appendix C
    run_py('v2/audit6_muscl.py')           # Section 6.3, upwind vs MUSCL

## 4. Training (`RUN_TRAINING`)

The job lists below rebuild every stored run from the settings recorded in it: 96 one-dimensional runs (eight seeds per variant, tuning sweeps, quadrature, macro-step, and mobility checks) and 62 two-dimensional runs ($\lambda=10^{-4}$, $10^{-2}$, $10^{-1}$, continuity-corrected, and projected). Existing outputs are skipped, so the stage can be interrupted and resumed.

In [ ]:
JOBS_1D = [
[
"v2/runs/eq_D_s4.json",
"v2/train2.py",
"--closure eq --variant D --seed 4 --lam 0.0003 --steps 1500 --ntrain 12 --resid end --data data_eq.npz --out {ROOT}/v2/runs/eq_D_s4.json"
],
[
"v2/runs/eq_D_s5.json",
"v2/train2.py",
"--closure eq --variant D --seed 5 --lam 0.0003 --steps 1500 --ntrain 12 --resid end --data data_eq.npz --out {ROOT}/v2/runs/eq_D_s5.json"
],
[
"v2/runs/eq_D_s6.json",
"v2/train2.py",
"--closure eq --variant D --seed 6 --lam 0.0003 --steps 1500 --ntrain 12 --resid end --data data_eq.npz --out {ROOT}/v2/runs/eq_D_s6.json"
],
[
"v2/runs/eq_D_s7.json",
"v2/train2.py",
"--closure eq --variant D --seed 7 --lam 0.0003 --steps 1500 --ntrain 12 --resid end --data data_eq.npz --out {ROOT}/v2/runs/eq_D_s7.json"
],
[
"v2/runs/eq_EET_s0_lam1e-3.json",
"v2/train2.py",
"--closure eq --variant EET --seed 0 --lam 0.001 --steps 1500 --ntrain 12 --resid end --data data_eq.npz --out {ROOT}/v2/runs/eq_EET_s0_lam1e-3.json"
],
[
"v2/runs/eq_EET_s0_lam1e-4.json",
"v2/train2.py",
"--closure eq --variant EET --seed 0 --lam 0.0001 --steps 1500 --ntrain 12 --resid end --data data_eq.npz --out {ROOT}/v2/runs/eq_EET_s0_lam1e-4.json"
],
[
"v2/runs/eq_EET_s4.json",
"v2/train2.py",
"--closure eq --variant EET --seed 4 --lam 0.0003 --steps 1500 --ntrain 12 --resid end --data data_eq.npz --out {ROOT}/v2/runs/eq_EET_s4.json"
],
[
"v2/runs/eq_EET_s5.json",
"v2/train2.py",
"--closure eq --variant EET --seed 5 --lam 0.0003 --steps 1500 --ntrain 12 --resid end --data data_eq.npz --out {ROOT}/v2/runs/eq_EET_s5.json"
],
[
"v2/runs/eq_EET_s6.json",
"v2/train2.py",
"--closure eq --variant EET --seed 6 --lam 0.0003 --steps 1500 --ntrain 12 --resid end --data data_eq.npz --out {ROOT}/v2/runs/eq_EET_s6.json"
],
[
"v2/runs/eq_EET_s7.json",
"v2/train2.py",
"--closure eq --variant EET --seed 7 --lam 0.0003 --steps 1500 --ntrain 12 --resid end --data data_eq.npz --out {ROOT}/v2/runs/eq_EET_s7.json"
],
[
"v2/runs/eq_ES_s0_lam1e-3.json",
"v2/train2.py",
"--closure eq --variant ES --seed 0 --lam 0.001 --steps 1500 --ntrain 12 --resid end --data data_eq.npz --out {ROOT}/v2/runs/eq_ES_s0_lam1e-3.json"
],
[
"v2/runs/eq_ES_s0_lam1e-4.json",
"v2/train2.py",
"--closure eq --variant ES --seed 0 --lam 0.0001 --steps 1500 --ntrain 12 --resid end --data data_eq.npz --out {ROOT}/v2/runs/eq_ES_s0_lam1e-4.json"
],
[
"v2/runs/eq_ES_s4.json",
"v2/train2.py",
"--closure eq --variant ES --seed 4 --lam 0.0003 --steps 1500 --ntrain 12 --resid end --data data_eq.npz --out {ROOT}/v2/runs/eq_ES_s4.json"
],
[
"v2/runs/eq_ES_s5.json",
"v2/train2.py",
"--closure eq --variant ES --seed 5 --lam 0.0003 --steps 1500 --ntrain 12 --resid end --data data_eq.npz --out {ROOT}/v2/runs/eq_ES_s5.json"
],
[
"v2/runs/eq_ES_s6.json",
"v2/train2.py",
"--closure eq --variant ES --seed 6 --lam 0.0003 --steps 1500 --ntrain 12 --resid end --data data_eq.npz --out {ROOT}/v2/runs/eq_ES_s6.json"
],
[
"v2/runs/eq_ES_s7.json",
"v2/train2.py",
"--closure eq --variant ES --seed 7 --lam 0.0003 --steps 1500 --ntrain 12 --resid end --data data_eq.npz --out {ROOT}/v2/runs/eq_ES_s7.json"
],
[
"v2/runs/eq_E_s0_lam1e-3.json",
"v2/train2.py",
"--closure eq --variant E --seed 0 --lam 0.001 --steps 1500 --ntrain 12 --resid end --data data_eq.npz --out {ROOT}/v2/runs/eq_E_s0_lam1e-3.json"
],
[
"v2/runs/eq_E_s0_lam1e-4.json",
"v2/train2.py",
"--closure eq --variant E --seed 0 --lam 0.0001 --steps 1500 --ntrain 12 --resid end --data data_eq.npz --out {ROOT}/v2/runs/eq_E_s0_lam1e-4.json"
],
[
"v2/runs/eq_E_s4.json",
"v2/train2.py",
"--closure eq --variant E --seed 4 --lam 0.0003 --steps 1500 --ntrain 12 --resid end --data data_eq.npz --out {ROOT}/v2/runs/eq_E_s4.json"
],
[
"v2/runs/eq_E_s5.json",
"v2/train2.py",
"--closure eq --variant E --seed 5 --lam 0.0003 --steps 1500 --ntrain 12 --resid end --data data_eq.npz --out {ROOT}/v2/runs/eq_E_s5.json"
],
[
"v2/runs/eq_E_s6.json",
"v2/train2.py",
"--closure eq --variant E --seed 6 --lam 0.0003 --steps 1500 --ntrain 12 --resid end --data data_eq.npz --out {ROOT}/v2/runs/eq_E_s6.json"
],
[
"v2/runs/eq_E_s7.json",
"v2/train2.py",
"--closure eq --variant E --seed 7 --lam 0.0003 --steps 1500 --ntrain 12 --resid end --data data_eq.npz --out {ROOT}/v2/runs/eq_E_s7.json"
],
[
"v2/runs/kin_D_s0.json",
"v2/train2.py",
"--closure kin --variant D --seed 0 --lam 0.0003 --steps 1500 --ntrain 12 --resid end --data data_kin.npz --out {ROOT}/v2/runs/kin_D_s0.json"
],
[
"v2/runs/kin_D_s1.json",
"v2/train2.py",
"--closure kin --variant D --seed 1 --lam 0.0003 --steps 1500 --ntrain 12 --resid end --data data_kin.npz --out {ROOT}/v2/runs/kin_D_s1.json"
],
[
"v2/runs/kin_D_s2.json",
"v2/train2.py",
"--closure kin --variant D --seed 2 --lam 0.0003 --steps 1500 --ntrain 12 --resid end --data data_kin.npz --out {ROOT}/v2/runs/kin_D_s2.json"
],
[
"v2/runs/kin_D_s3.json",
"v2/train2.py",
"--closure kin --variant D --seed 3 --lam 0.0003 --steps 1500 --ntrain 12 --resid end --data data_kin.npz --out {ROOT}/v2/runs/kin_D_s3.json"
],
[
"v2/runs/kin_D_s4.json",
"v2/train2.py",
"--closure kin --variant D --seed 4 --lam 0.0003 --steps 1500 --ntrain 12 --resid end --data data_kin.npz --out {ROOT}/v2/runs/kin_D_s4.json"
],
[
"v2/runs/kin_D_s5.json",
"v2/train2.py",
"--closure kin --variant D --seed 5 --lam 0.0003 --steps 1500 --ntrain 12 --resid end --data data_kin.npz --out {ROOT}/v2/runs/kin_D_s5.json"
],
[
"v2/runs/kin_D_s6.json",
"v2/train2.py",
"--closure kin --variant D --seed 6 --lam 0.0003 --steps 1500 --ntrain 12 --resid end --data data_kin.npz --out {ROOT}/v2/runs/kin_D_s6.json"
],
[
"v2/runs/kin_D_s7.json",
"v2/train2.py",
"--closure kin --variant D --seed 7 --lam 0.0003 --steps 1500 --ntrain 12 --resid end --data data_kin.npz --out {ROOT}/v2/runs/kin_D_s7.json"
],
[
"v2/runs/kin_EK_s0.json",
"v2/train2.py",
"--closure kin --variant EK --seed 0 --lam 0.0003 --steps 1500 --ntrain 12 --resid end --data data_kin.npz --out {ROOT}/v2/runs/kin_EK_s0.json"
],
[
"v2/runs/kin_EK_s0_M1600.json",
"v2/train2.py",
"--closure kin --variant EK --seed 0 --lam 0.0003 --steps 1500 --ntrain 12 --resid end --data {ROOT}/v2/data_kin_M1600.npz --out {ROOT}/v2/runs/kin_EK_s0_M1600.json"
],
[
"v2/runs/kin_EK_s0_half.json",
"v2/train2.py",
"--closure kin --variant EK --seed 0 --lam 0.0003 --steps 1500 --ntrain 12 --resid end --data {ROOT}/v2/data_kin_half.npz --out {ROOT}/v2/runs/kin_EK_s0_half.json"
],
[
"v2/runs/kin_EK_s0_lam1e-3.json",
"v2/train2.py",
"--closure kin --variant EK --seed 0 --lam 0.001 --steps 1500 --ntrain 12 --resid end --data data_kin.npz --out {ROOT}/v2/runs/kin_EK_s0_lam1e-3.json"
],
[
"v2/runs/kin_EK_s0_lam1e-4.json",
"v2/train2.py",
"--closure kin --variant EK --seed 0 --lam 0.0001 --steps 1500 --ntrain 12 --resid end --data data_kin.npz --out {ROOT}/v2/runs/kin_EK_s0_lam1e-4.json"
],
[
"v2/runs/kin_EK_s0_trap.json",
"v2/train2.py",
"--closure kin --variant EK --seed 0 --lam 0.0003 --steps 1500 --ntrain 12 --resid trap --data data_kin.npz --out {ROOT}/v2/runs/kin_EK_s0_trap.json"
],
[
"v2/runs/kin_EK_s1.json",
"v2/train2.py",
"--closure kin --variant EK --seed 1 --lam 0.0003 --steps 1500 --ntrain 12 --resid end --data data_kin.npz --out {ROOT}/v2/runs/kin_EK_s1.json"
],
[
"v2/runs/kin_EK_s1_M1600.json",
"v2/train2.py",
"--closure kin --variant EK --seed 1 --lam 0.0003 --steps 1500 --ntrain 12 --resid end --data {ROOT}/v2/data_kin_M1600.npz --out {ROOT}/v2/runs/kin_EK_s1_M1600.json"
],
[
"v2/runs/kin_EK_s1_half.json",
"v2/train2.py",
"--closure kin --variant EK --seed 1 --lam 0.0003 --steps 1500 --ntrain 12 --resid end --data {ROOT}/v2/data_kin_half.npz --out {ROOT}/v2/runs/kin_EK_s1_half.json"
],
[
"v2/runs/kin_EK_s1_trap.json",
"v2/train2.py",
"--closure kin --variant EK --seed 1 --lam 0.0003 --steps 1500 --ntrain 12 --resid trap --data data_kin.npz --out {ROOT}/v2/runs/kin_EK_s1_trap.json"
],
[
"v2/runs/kin_EK_s2.json",
"v2/train2.py",
"--closure kin --variant EK --seed 2 --lam 0.0003 --steps 1500 --ntrain 12 --resid end --data data_kin.npz --out {ROOT}/v2/runs/kin_EK_s2.json"
],
[
"v2/runs/kin_EK_s3.json",
"v2/train2.py",
"--closure kin --variant EK --seed 3 --lam 0.0003 --steps 1500 --ntrain 12 --resid end --data data_kin.npz --out {ROOT}/v2/runs/kin_EK_s3.json"
],
[
"v2/runs/kin_EK_s4.json",
"v2/train2.py",
"--closure kin --variant EK --seed 4 --lam 0.0003 --steps 1500 --ntrain 12 --resid end --data data_kin.npz --out {ROOT}/v2/runs/kin_EK_s4.json"
],
[
"v2/runs/kin_EK_s5.json",
"v2/train2.py",
"--closure kin --variant EK --seed 5 --lam 0.0003 --steps 1500 --ntrain 12 --resid end --data data_kin.npz --out {ROOT}/v2/runs/kin_EK_s5.json"
],
[
"v2/runs/kin_EK_s6.json",
"v2/train2.py",
"--closure kin --variant EK --seed 6 --lam 0.0003 --steps 1500 --ntrain 12 --resid end --data data_kin.npz --out {ROOT}/v2/runs/kin_EK_s6.json"
],
[
"v2/runs/kin_EK_s7.json",
"v2/train2.py",
"--closure kin --variant EK --seed 7 --lam 0.0003 --steps 1500 --ntrain 12 --resid end --data data_kin.npz --out {ROOT}/v2/runs/kin_EK_s7.json"
],
[
"v2/runs/kin_EPH_s0.json",
"v2/train2.py",
"--closure kin --variant EPH --seed 0 --lam 0.0003 --steps 1500 --ntrain 12 --resid end --data data_kin.npz --out {ROOT}/v2/runs/kin_EPH_s0.json"
],
[
"v2/runs/kin_EPH_s0_lam1e-3.json",
"v2/train2.py",
"--closure kin --variant EPH --seed 0 --lam 0.001 --steps 1500 --ntrain 12 --resid end --data data_kin.npz --out {ROOT}/v2/runs/kin_EPH_s0_lam1e-3.json"
],
[
"v2/runs/kin_EPH_s0_lam1e-4.json",
"v2/train2.py",
"--closure kin --variant EPH --seed 0 --lam 0.0001 --steps 1500 --ntrain 12 --resid end --data data_kin.npz --out {ROOT}/v2/runs/kin_EPH_s0_lam1e-4.json"
],
[
"v2/runs/kin_EPH_s1.json",
"v2/train2.py",
"--closure kin --variant EPH --seed 1 --lam 0.0003 --steps 1500 --ntrain 12 --resid end --data data_kin.npz --out {ROOT}/v2/runs/kin_EPH_s1.json"
],
[
"v2/runs/kin_EPH_s2.json",
"v2/train2.py",
"--closure kin --variant EPH --seed 2 --lam 0.0003 --steps 1500 --ntrain 12 --resid end --data data_kin.npz --out {ROOT}/v2/runs/kin_EPH_s2.json"
],
[
"v2/runs/kin_EPH_s3.json",
"v2/train2.py",
"--closure kin --variant EPH --seed 3 --lam 0.0003 --steps 1500 --ntrain 12 --resid end --data data_kin.npz --out {ROOT}/v2/runs/kin_EPH_s3.json"
],
[
"v2/runs/kin_EPH_s4.json",
"v2/train2.py",
"--closure kin --variant EPH --seed 4 --lam 0.0003 --steps 1500 --ntrain 12 --resid end --data data_kin.npz --out {ROOT}/v2/runs/kin_EPH_s4.json"
],
[
"v2/runs/kin_EPH_s5.json",
"v2/train2.py",
"--closure kin --variant EPH --seed 5 --lam 0.0003 --steps 1500 --ntrain 12 --resid end --data data_kin.npz --out {ROOT}/v2/runs/kin_EPH_s5.json"
],
[
"v2/runs/kin_EPH_s6.json",
"v2/train2.py",
"--closure kin --variant EPH --seed 6 --lam 0.0003 --steps 1500 --ntrain 12 --resid end --data data_kin.npz --out {ROOT}/v2/runs/kin_EPH_s6.json"
],
[
"v2/runs/kin_EPH_s7.json",
"v2/train2.py",
"--closure kin --variant EPH --seed 7 --lam 0.0003 --steps 1500 --ntrain 12 --resid end --data data_kin.npz --out {ROOT}/v2/runs/kin_EPH_s7.json"
],
[
"v2/runs/kin_E_s0.json",
"v2/train2.py",
"--closure kin --variant E --seed 0 --lam 0.0003 --steps 1500 --ntrain 12 --resid end --data data_kin.npz --out {ROOT}/v2/runs/kin_E_s0.json"
],
[
"v2/runs/kin_E_s0_M1600.json",
"v2/train2.py",
"--closure kin --variant E --seed 0 --lam 0.0003 --steps 1500 --ntrain 12 --resid end --data {ROOT}/v2/data_kin_M1600.npz --out {ROOT}/v2/runs/kin_E_s0_M1600.json"
],
[
"v2/runs/kin_E_s0_half.json",
"v2/train2.py",
"--closure kin --variant E --seed 0 --lam 0.0003 --steps 1500 --ntrain 12 --resid end --data {ROOT}/v2/data_kin_half.npz --out {ROOT}/v2/runs/kin_E_s0_half.json"
],
[
"v2/runs/kin_E_s0_lam1e-3.json",
"v2/train2.py",
"--closure kin --variant E --seed 0 --lam 0.001 --steps 1500 --ntrain 12 --resid end --data data_kin.npz --out {ROOT}/v2/runs/kin_E_s0_lam1e-3.json"
],
[
"v2/runs/kin_E_s0_lam1e-4.json",
"v2/train2.py",
"--closure kin --variant E --seed 0 --lam 0.0001 --steps 1500 --ntrain 12 --resid end --data data_kin.npz --out {ROOT}/v2/runs/kin_E_s0_lam1e-4.json"
],
[
"v2/runs/kin_E_s0_trap.json",
"v2/train2.py",
"--closure kin --variant E --seed 0 --lam 0.0003 --steps 1500 --ntrain 12 --resid trap --data data_kin.npz --out {ROOT}/v2/runs/kin_E_s0_trap.json"
],
[
"v2/runs/kin_E_s1.json",
"v2/train2.py",
"--closure kin --variant E --seed 1 --lam 0.0003 --steps 1500 --ntrain 12 --resid end --data data_kin.npz --out {ROOT}/v2/runs/kin_E_s1.json"
],
[
"v2/runs/kin_E_s1_M1600.json",
"v2/train2.py",
"--closure kin --variant E --seed 1 --lam 0.0003 --steps 1500 --ntrain 12 --resid end --data {ROOT}/v2/data_kin_M1600.npz --out {ROOT}/v2/runs/kin_E_s1_M1600.json"
],
[
"v2/runs/kin_E_s1_half.json",
"v2/train2.py",
"--closure kin --variant E --seed 1 --lam 0.0003 --steps 1500 --ntrain 12 --resid end --data {ROOT}/v2/data_kin_half.npz --out {ROOT}/v2/runs/kin_E_s1_half.json"
],
[
"v2/runs/kin_E_s1_trap.json",
"v2/train2.py",
"--closure kin --variant E --seed 1 --lam 0.0003 --steps 1500 --ntrain 12 --resid trap --data data_kin.npz --out {ROOT}/v2/runs/kin_E_s1_trap.json"
],
[
"v2/runs/kin_E_s2.json",
"v2/train2.py",
"--closure kin --variant E --seed 2 --lam 0.0003 --steps 1500 --ntrain 12 --resid end --data data_kin.npz --out {ROOT}/v2/runs/kin_E_s2.json"
],
[
"v2/runs/kin_E_s3.json",
"v2/train2.py",
"--closure kin --variant E --seed 3 --lam 0.0003 --steps 1500 --ntrain 12 --resid end --data data_kin.npz --out {ROOT}/v2/runs/kin_E_s3.json"
],
[
"v2/runs/kin_E_s4.json",
"v2/train2.py",
"--closure kin --variant E --seed 4 --lam 0.0003 --steps 1500 --ntrain 12 --resid end --data data_kin.npz --out {ROOT}/v2/runs/kin_E_s4.json"
],
[
"v2/runs/kin_E_s5.json",
"v2/train2.py",
"--closure kin --variant E --seed 5 --lam 0.0003 --steps 1500 --ntrain 12 --resid end --data data_kin.npz --out {ROOT}/v2/runs/kin_E_s5.json"
],
[
"v2/runs/kin_E_s6.json",
"v2/train2.py",
"--closure kin --variant E --seed 6 --lam 0.0003 --steps 1500 --ntrain 12 --resid end --data data_kin.npz --out {ROOT}/v2/runs/kin_E_s6.json"
],
[
"v2/runs/kin_E_s7.json",
"v2/train2.py",
"--closure kin --variant E --seed 7 --lam 0.0003 --steps 1500 --ntrain 12 --resid end --data data_kin.npz --out {ROOT}/v2/runs/kin_E_s7.json"
],
[
"runs/eq_D_s0.json",
"v2/train2.py",
"--closure eq --variant D --seed 0 --lam 0.0003 --steps 1500 --ntrain 12 --out {ROOT}/runs/eq_D_s0.json"
],
[
"runs/eq_D_s1.json",
"v2/train2.py",
"--closure eq --variant D --seed 1 --lam 0.0003 --steps 1500 --ntrain 12 --out {ROOT}/runs/eq_D_s1.json"
],
[
"runs/eq_D_s2.json",
"v2/train2.py",
"--closure eq --variant D --seed 2 --lam 0.0003 --steps 1500 --ntrain 12 --out {ROOT}/runs/eq_D_s2.json"
],
[
"runs/eq_D_s3.json",
"v2/train2.py",
"--closure eq --variant D --seed 3 --lam 0.0003 --steps 1500 --ntrain 12 --out {ROOT}/runs/eq_D_s3.json"
],
[
"runs/eq_EET_s0.json",
"v2/train2.py",
"--closure eq --variant EET --seed 0 --lam 0.0003 --steps 1500 --ntrain 12 --out {ROOT}/runs/eq_EET_s0.json"
],
[
"runs/eq_EET_s1.json",
"v2/train2.py",
"--closure eq --variant EET --seed 1 --lam 0.0003 --steps 1500 --ntrain 12 --out {ROOT}/runs/eq_EET_s1.json"
],
[
"runs/eq_EET_s2.json",
"v2/train2.py",
"--closure eq --variant EET --seed 2 --lam 0.0003 --steps 1500 --ntrain 12 --out {ROOT}/runs/eq_EET_s2.json"
],
[
"runs/eq_EET_s3.json",
"v2/train2.py",
"--closure eq --variant EET --seed 3 --lam 0.0003 --steps 1500 --ntrain 12 --out {ROOT}/runs/eq_EET_s3.json"
],
[
"runs/eq_ES_s0.json",
"v2/train2.py",
"--closure eq --variant ES --seed 0 --lam 0.0003 --steps 1500 --ntrain 12 --out {ROOT}/runs/eq_ES_s0.json"
],
[
"runs/eq_ES_s1.json",
"v2/train2.py",
"--closure eq --variant ES --seed 1 --lam 0.0003 --steps 1500 --ntrain 12 --out {ROOT}/runs/eq_ES_s1.json"
],
[
"runs/eq_ES_s2.json",
"v2/train2.py",
"--closure eq --variant ES --seed 2 --lam 0.0003 --steps 1500 --ntrain 12 --out {ROOT}/runs/eq_ES_s2.json"
],
[
"runs/eq_ES_s3.json",
"v2/train2.py",
"--closure eq --variant ES --seed 3 --lam 0.0003 --steps 1500 --ntrain 12 --out {ROOT}/runs/eq_ES_s3.json"
],
[
"runs/eq_E_s0.json",
"v2/train2.py",
"--closure eq --variant E --seed 0 --lam 0.0003 --steps 1500 --ntrain 12 --out {ROOT}/runs/eq_E_s0.json"
],
[
"runs/eq_E_s1.json",
"v2/train2.py",
"--closure eq --variant E --seed 1 --lam 0.0003 --steps 1500 --ntrain 12 --out {ROOT}/runs/eq_E_s1.json"
],
[
"runs/eq_E_s2.json",
"v2/train2.py",
"--closure eq --variant E --seed 2 --lam 0.0003 --steps 1500 --ntrain 12 --out {ROOT}/runs/eq_E_s2.json"
],
[
"runs/eq_E_s3.json",
"v2/train2.py",
"--closure eq --variant E --seed 3 --lam 0.0003 --steps 1500 --ntrain 12 --out {ROOT}/runs/eq_E_s3.json"
],
[
"runs/sweep_E_3e-2.json",
"v2/train2.py",
"--closure eq --variant E --seed 0 --lam 0.03 --steps 1500 --out {ROOT}/runs/sweep_E_3e-2.json"
],
[
"runs/sweep_E_3e-3.json",
"v2/train2.py",
"--closure eq --variant E --seed 0 --lam 0.003 --steps 1500 --out {ROOT}/runs/sweep_E_3e-3.json"
],
[
"runs/sweep_E_3e-4.json",
"v2/train2.py",
"--closure eq --variant E --seed 0 --lam 0.0003 --steps 1500 --out {ROOT}/runs/sweep_E_3e-4.json"
],
[
"runs/sweepsp_D.json",
"v2/train2.py",
"--closure eq --variant D --seed 0 --lam 0.003 --steps 1500 --ntrain 12 --out {ROOT}/runs/sweepsp_D.json"
],
[
"runs/sweepsp_E_3e-2.json",
"v2/train2.py",
"--closure eq --variant E --seed 0 --lam 0.03 --steps 1500 --ntrain 12 --out {ROOT}/runs/sweepsp_E_3e-2.json"
],
[
"runs/sweepsp_E_3e-3.json",
"v2/train2.py",
"--closure eq --variant E --seed 0 --lam 0.003 --steps 1500 --ntrain 12 --out {ROOT}/runs/sweepsp_E_3e-3.json"
],
[
"runs/sweepsp_E_3e-4.json",
"v2/train2.py",
"--closure eq --variant E --seed 0 --lam 0.0003 --steps 1500 --ntrain 12 --out {ROOT}/runs/sweepsp_E_3e-4.json"
],
[
"runs/dense/eq_D_s0.json",
"v2/train2.py",
"--closure eq --variant D --seed 0 --lam 0.003 --steps 1500 --out {ROOT}/runs/dense/eq_D_s0.json"
]
]
JOBS_2D = [
[
"v2/runs2d/D_s0.json",
"v2/train2d.py",
"--variant D --seed 0 --lam 0.001 --out {ROOT}/v2/runs2d/D_s0.json"
],
[
"v2/runs2d/D_s1.json",
"v2/train2d.py",
"--variant D --seed 1 --lam 0.0001 --out {ROOT}/v2/runs2d/D_s1.json"
],
[
"v2/runs2d/D_s2.json",
"v2/train2d.py",
"--variant D --seed 2 --lam 0.0001 --out {ROOT}/v2/runs2d/D_s2.json"
],
[
"v2/runs2d/EET_s0.json",
"v2/train2d.py",
"--variant EET --seed 0 --lam 0.0001 --out {ROOT}/v2/runs2d/EET_s0.json"
],
[
"v2/runs2d/EET_s1.json",
"v2/train2d.py",
"--variant EET --seed 1 --lam 0.0001 --out {ROOT}/v2/runs2d/EET_s1.json"
],
[
"v2/runs2d/EET_s2.json",
"v2/train2d.py",
"--variant EET --seed 2 --lam 0.0001 --out {ROOT}/v2/runs2d/EET_s2.json"
],
[
"v2/runs2d/ES_s0.json",
"v2/train2d.py",
"--variant ES --seed 0 --lam 0.0001 --out {ROOT}/v2/runs2d/ES_s0.json"
],
[
"v2/runs2d/ES_s1.json",
"v2/train2d.py",
"--variant ES --seed 1 --lam 0.0001 --out {ROOT}/v2/runs2d/ES_s1.json"
],
[
"v2/runs2d/ES_s2.json",
"v2/train2d.py",
"--variant ES --seed 2 --lam 0.0001 --out {ROOT}/v2/runs2d/ES_s2.json"
],
[
"v2/runs2d/E_s0.json",
"v2/train2d.py",
"--variant E --seed 0 --lam 0.0001 --out {ROOT}/v2/runs2d/E_s0.json"
],
[
"v2/runs2d/E_s1.json",
"v2/train2d.py",
"--variant E --seed 1 --lam 0.0001 --out {ROOT}/v2/runs2d/E_s1.json"
],
[
"v2/runs2d/E_s2.json",
"v2/train2d.py",
"--variant E --seed 2 --lam 0.0001 --out {ROOT}/v2/runs2d/E_s2.json"
],
[
"v2/runs2d/sweep_E_1e-2.json",
"v2/train2d.py",
"--variant E --seed 0 --lam 0.01 --out {ROOT}/v2/runs2d/sweep_E_1e-2.json"
],
[
"v2/runs2d/sweep_E_1e-3.json",
"v2/train2d.py",
"--variant E --seed 0 --lam 0.001 --out {ROOT}/v2/runs2d/sweep_E_1e-3.json"
],
[
"v2/runs2d/sweep_E_1e-4.json",
"v2/train2d.py",
"--variant E --seed 0 --lam 0.0001 --out {ROOT}/v2/runs2d/sweep_E_1e-4.json"
],
[
"v2/runs2d_active/D_s0_logged.json",
"v2/train2d.py",
"--variant D --seed 0 --lam 0.1 --out {ROOT}/v2/runs2d_active/D_s0_logged.json"
],
[
"v2/runs2d_active/D_s1_logged.json",
"v2/train2d.py",
"--variant D --seed 1 --lam 0.1 --out {ROOT}/v2/runs2d_active/D_s1_logged.json"
],
[
"v2/runs2d_active/D_s2_logged.json",
"v2/train2d.py",
"--variant D --seed 2 --lam 0.1 --out {ROOT}/v2/runs2d_active/D_s2_logged.json"
],
[
"v2/runs2d_active/EET_s0_lam1e-1.json",
"v2/train2d.py",
"--variant EET --seed 0 --lam 0.1 --out {ROOT}/v2/runs2d_active/EET_s0_lam1e-1.json"
],
[
"v2/runs2d_active/EET_s0_lam1e-2.json",
"v2/train2d.py",
"--variant EET --seed 0 --lam 0.01 --out {ROOT}/v2/runs2d_active/EET_s0_lam1e-2.json"
],
[
"v2/runs2d_active/EET_s1_lam1e-1.json",
"v2/train2d.py",
"--variant EET --seed 1 --lam 0.1 --out {ROOT}/v2/runs2d_active/EET_s1_lam1e-1.json"
],
[
"v2/runs2d_active/EET_s1_lam1e-2.json",
"v2/train2d.py",
"--variant EET --seed 1 --lam 0.01 --out {ROOT}/v2/runs2d_active/EET_s1_lam1e-2.json"
],
[
"v2/runs2d_active/EET_s2_lam1e-1.json",
"v2/train2d.py",
"--variant EET --seed 2 --lam 0.1 --out {ROOT}/v2/runs2d_active/EET_s2_lam1e-1.json"
],
[
"v2/runs2d_active/EET_s2_lam1e-2.json",
"v2/train2d.py",
"--variant EET --seed 2 --lam 0.01 --out {ROOT}/v2/runs2d_active/EET_s2_lam1e-2.json"
],
[
"v2/runs2d_active/ES_s0_lam1e-1.json",
"v2/train2d.py",
"--variant ES --seed 0 --lam 0.1 --out {ROOT}/v2/runs2d_active/ES_s0_lam1e-1.json"
],
[
"v2/runs2d_active/ES_s0_lam1e-2.json",
"v2/train2d.py",
"--variant ES --seed 0 --lam 0.01 --out {ROOT}/v2/runs2d_active/ES_s0_lam1e-2.json"
],
[
"v2/runs2d_active/ES_s1_lam1e-1.json",
"v2/train2d.py",
"--variant ES --seed 1 --lam 0.1 --out {ROOT}/v2/runs2d_active/ES_s1_lam1e-1.json"
],
[
"v2/runs2d_active/ES_s1_lam1e-2.json",
"v2/train2d.py",
"--variant ES --seed 1 --lam 0.01 --out {ROOT}/v2/runs2d_active/ES_s1_lam1e-2.json"
],
[
"v2/runs2d_active/ES_s2_lam1e-1.json",
"v2/train2d.py",
"--variant ES --seed 2 --lam 0.1 --out {ROOT}/v2/runs2d_active/ES_s2_lam1e-1.json"
],
[
"v2/runs2d_active/ES_s2_lam1e-2.json",
"v2/train2d.py",
"--variant ES --seed 2 --lam 0.01 --out {ROOT}/v2/runs2d_active/ES_s2_lam1e-2.json"
],
[
"v2/runs2d_active/E_s0_lam1e-1.json",
"v2/train2d.py",
"--variant E --seed 0 --lam 0.1 --out {ROOT}/v2/runs2d_active/E_s0_lam1e-1.json"
],
[
"v2/runs2d_active/E_s0_lam1e-2.json",
"v2/train2d.py",
"--variant E --seed 0 --lam 0.01 --out {ROOT}/v2/runs2d_active/E_s0_lam1e-2.json"
],
[
"v2/runs2d_active/E_s1_lam1e-1.json",
"v2/train2d.py",
"--variant E --seed 1 --lam 0.1 --out {ROOT}/v2/runs2d_active/E_s1_lam1e-1.json"
],
[
"v2/runs2d_active/E_s1_lam1e-2.json",
"v2/train2d.py",
"--variant E --seed 1 --lam 0.01 --out {ROOT}/v2/runs2d_active/E_s1_lam1e-2.json"
],
[
"v2/runs2d_active/E_s2_lam1e-1.json",
"v2/train2d.py",
"--variant E --seed 2 --lam 0.1 --out {ROOT}/v2/runs2d_active/E_s2_lam1e-1.json"
],
[
"v2/runs2d_active/E_s2_lam1e-2.json",
"v2/train2d.py",
"--variant E --seed 2 --lam 0.01 --out {ROOT}/v2/runs2d_active/E_s2_lam1e-2.json"
],
[
"v2/runs2d_r4/EET_p1_s0_lam1e-1.json",
"v2/train2d_r4.py",
"--variant EET --proj 1 --seed 0 --lam 0.1 --out {ROOT}/v2/runs2d_r4/EET_p1_s0_lam1e-1.json"
],
[
"v2/runs2d_r4/EET_p1_s0_lam1e-2.json",
"v2/train2d_r4.py",
"--variant EET --proj 1 --seed 0 --lam 0.01 --out {ROOT}/v2/runs2d_r4/EET_p1_s0_lam1e-2.json"
],
[
"v2/runs2d_r4/EET_p1_s1_lam1e-1.json",
"v2/train2d_r4.py",
"--variant EET --proj 1 --seed 1 --lam 0.1 --out {ROOT}/v2/runs2d_r4/EET_p1_s1_lam1e-1.json"
],
[
"v2/runs2d_r4/EET_p1_s1_lam1e-2.json",
"v2/train2d_r4.py",
"--variant EET --proj 1 --seed 1 --lam 0.01 --out {ROOT}/v2/runs2d_r4/EET_p1_s1_lam1e-2.json"
],
[
"v2/runs2d_r4/EET_p1_s2_lam1e-1.json",
"v2/train2d_r4.py",
"--variant EET --proj 1 --seed 2 --lam 0.1 --out {ROOT}/v2/runs2d_r4/EET_p1_s2_lam1e-1.json"
],
[
"v2/runs2d_r4/EET_p1_s2_lam1e-2.json",
"v2/train2d_r4.py",
"--variant EET --proj 1 --seed 2 --lam 0.01 --out {ROOT}/v2/runs2d_r4/EET_p1_s2_lam1e-2.json"
],
[
"v2/runs2d_r4/ESC_p0_s0_lam1e-1.json",
"v2/train2d_r4.py",
"--variant ESC --proj 0 --seed 0 --lam 0.1 --out {ROOT}/v2/runs2d_r4/ESC_p0_s0_lam1e-1.json"
],
[
"v2/runs2d_r4/ESC_p0_s0_lam1e-2.json",
"v2/train2d_r4.py",
"--variant ESC --proj 0 --seed 0 --lam 0.01 --out {ROOT}/v2/runs2d_r4/ESC_p0_s0_lam1e-2.json"
],
[
"v2/runs2d_r4/ESC_p0_s1_lam1e-1.json",
"v2/train2d_r4.py",
"--variant ESC --proj 0 --seed 1 --lam 0.1 --out {ROOT}/v2/runs2d_r4/ESC_p0_s1_lam1e-1.json"
],
[
"v2/runs2d_r4/ESC_p0_s1_lam1e-2.json",
"v2/train2d_r4.py",
"--variant ESC --proj 0 --seed 1 --lam 0.01 --out {ROOT}/v2/runs2d_r4/ESC_p0_s1_lam1e-2.json"
],
[
"v2/runs2d_r4/ESC_p0_s2_lam1e-1.json",
"v2/train2d_r4.py",
"--variant ESC --proj 0 --seed 2 --lam 0.1 --out {ROOT}/v2/runs2d_r4/ESC_p0_s2_lam1e-1.json"
],
[
"v2/runs2d_r4/ESC_p0_s2_lam1e-2.json",
"v2/train2d_r4.py",
"--variant ESC --proj 0 --seed 2 --lam 0.01 --out {ROOT}/v2/runs2d_r4/ESC_p0_s2_lam1e-2.json"
],
[
"v2/runs2d_r4/ES_p1_s0_lam1e-1.json",
"v2/train2d_r4.py",
"--variant ES --proj 1 --seed 0 --lam 0.1 --out {ROOT}/v2/runs2d_r4/ES_p1_s0_lam1e-1.json"
],
[
"v2/runs2d_r4/ES_p1_s0_lam1e-2.json",
"v2/train2d_r4.py",
"--variant ES --proj 1 --seed 0 --lam 0.01 --out {ROOT}/v2/runs2d_r4/ES_p1_s0_lam1e-2.json"
],
[
"v2/runs2d_r4/ES_p1_s1_lam1e-1.json",
"v2/train2d_r4.py",
"--variant ES --proj 1 --seed 1 --lam 0.1 --out {ROOT}/v2/runs2d_r4/ES_p1_s1_lam1e-1.json"
],
[
"v2/runs2d_r4/ES_p1_s1_lam1e-2.json",
"v2/train2d_r4.py",
"--variant ES --proj 1 --seed 1 --lam 0.01 --out {ROOT}/v2/runs2d_r4/ES_p1_s1_lam1e-2.json"
],
[
"v2/runs2d_r4/ES_p1_s2_lam1e-1.json",
"v2/train2d_r4.py",
"--variant ES --proj 1 --seed 2 --lam 0.1 --out {ROOT}/v2/runs2d_r4/ES_p1_s2_lam1e-1.json"
],
[
"v2/runs2d_r4/ES_p1_s2_lam1e-2.json",
"v2/train2d_r4.py",
"--variant ES --proj 1 --seed 2 --lam 0.01 --out {ROOT}/v2/runs2d_r4/ES_p1_s2_lam1e-2.json"
],
[
"v2/runs2d_r4/E_p1_s0_lam1e-1.json",
"v2/train2d_r4.py",
"--variant E --proj 1 --seed 0 --lam 0.1 --out {ROOT}/v2/runs2d_r4/E_p1_s0_lam1e-1.json"
],
[
"v2/runs2d_r4/E_p1_s0_lam1e-2.json",
"v2/train2d_r4.py",
"--variant E --proj 1 --seed 0 --lam 0.01 --out {ROOT}/v2/runs2d_r4/E_p1_s0_lam1e-2.json"
],
[
"v2/runs2d_r4/E_p1_s1_lam1e-1.json",
"v2/train2d_r4.py",
"--variant E --proj 1 --seed 1 --lam 0.1 --out {ROOT}/v2/runs2d_r4/E_p1_s1_lam1e-1.json"
],
[
"v2/runs2d_r4/E_p1_s1_lam1e-2.json",
"v2/train2d_r4.py",
"--variant E --proj 1 --seed 1 --lam 0.01 --out {ROOT}/v2/runs2d_r4/E_p1_s1_lam1e-2.json"
],
[
"v2/runs2d_r4/E_p1_s2_lam1e-1.json",
"v2/train2d_r4.py",
"--variant E --proj 1 --seed 2 --lam 0.1 --out {ROOT}/v2/runs2d_r4/E_p1_s2_lam1e-1.json"
],
[
"v2/runs2d_r4/E_p1_s2_lam1e-2.json",
"v2/train2d_r4.py",
"--variant E --proj 1 --seed 2 --lam 0.01 --out {ROOT}/v2/runs2d_r4/E_p1_s2_lam1e-2.json"
],
[
"v2/runs2d_r4check/EET_p0_s1_lam1e-4.json",
"v2/train2d_r4.py",
"--variant EET --proj 0 --seed 1 --lam 0.0001 --out {ROOT}/v2/runs2d_r4check/EET_p0_s1_lam1e-4.json"
],
[
"v2/runs2d_r4check/ES_p0_s1_lam1e-4.json",
"v2/train2d_r4.py",
"--variant ES --proj 0 --seed 1 --lam 0.0001 --out {ROOT}/v2/runs2d_r4check/ES_p0_s1_lam1e-4.json"
]
]
if RUN_TRAINING:
    run_jobs(JOBS_1D)
    run_jobs(JOBS_2D)
else:
    print(len(JOBS_1D), '1D and', len(JOBS_2D), '2D run records present:',
          sum(os.path.exists(os.path.join(ROOT, j[0])) for j in JOBS_1D + JOBS_2D))

## 5. Analysis

Every statistic, audit, and summary file used in the paper is recomputed here from the run records, checkpoints, and solver outputs.

In [ ]:
for s, a in [('v2/baseline_floors.py', ''), ('v2/make_floors_skip.py', ''), ('v2/make_dissipation.py', ''),
             ('v2/stats_v2.py', ''), ('v2/stats_round3.py', ''), ('v2/stats_round4.py', ''), ('v2/stats_round5.py', ''),
             ('v2/make_analysis2.py', ''), ('v2/make_active2d.py', ''), ('v2/agg_r4.py', ''),
             ('v2/audit3.py', ''), ('v2/audit4_dg.py', ''), ('v2/audit4_dg.py', '--table'), ('v2/audit4_kin.py', ''),
             ('v2/audit5_dg.py', ''), ('v2/audit5_front.py', '')]:
    run_py(s, a, tail=4)

## 6. Figures

In [ ]:
from IPython.display import Image, display, Markdown
run_py('v2/figs_v2a.py', tail=2); run_py('v2/figs_v2b.py', tail=2); run_py('v2/figs_r4.py', tail=2)
FIGS = [('fig1_verification.png', 'Figure 1. Solver verification.'),
        ('fig2_identity_macro.png', 'Figure 2. Discrete identity and macro-step floors.'),
        ('fig3_ablation_eq.png', 'Figure 3. One-dimensional equilibrium ablation.'),
        ('fig4_ablation_kin.png', 'Figure 4. One-dimensional internal-variable ablation.'),
        ('fig5_robustness.png', 'Figure 5. Robustness of the kinetic penalty and penalty-weight tuning.'),
        ('fig6_2d_solver.png', 'Figure 6. Two-dimensional melting.'),
        ('fig8_continuity.png', 'Figure 7. Continuity decomposition.'),
        ('fig7_2d_operator.png', 'Figure 8. Two-dimensional operator ablation.')]
for i, (f, cap) in enumerate(FIGS, 1):
    shutil.copy(os.path.join(V, f), os.path.join(ROOT, 'figures', f'Fig{i}.png'))
    display(Markdown(f'**{cap}**')); display(Image(os.path.join(V, f), width=900))

## 7. Tables

In [ ]:
import numpy as np, pandas as pd
pd.set_option('display.precision', 4); pd.set_option('display.width', 160)
J = lambda f: json.load(open(os.path.join(V, f)))
def save(df, name):
    import re as _re
    df.to_csv(os.path.join(ROOT, 'tables', _re.sub(r'[^A-Za-z0-9]+', '_', name).strip('_') + '.csv')); display(Markdown(f'**{name}**')); display(df)

save(pd.DataFrame([['D', '', '', '', '', '', 'equilibrium (1D, 2D); internal-variable (1D)'],
                   ['E', 'x', '', '', '', '', 'equilibrium (1D, 2D); internal-variable (1D)'],
                   ['E+S', 'x', 'x', '', '', '', 'equilibrium (1D, 2D)'],
                   ['E+E/T', 'x', '', 'x', '', '', 'equilibrium (1D, 2D)'],
                   ['E+PH', 'x', '', '', '', 'x', 'internal-variable (1D)'],
                   ['E+K', 'x', '', '', 'x', '', 'internal-variable (1D)']],
                  columns=['Variant', 'L_E', 'L_S', 'L_E/T', 'L_f', 'L_ph', 'Closure']).set_index('Variant'), 'Table I  Loss variants')

rows = []
for name, f in [('1D equilibrium', os.path.join(ROOT, 'data_eq.npz')), ('1D internal-variable', os.path.join(ROOT, 'data_kin.npz'))]:
    d = np.load(f)
    rows.append([name, f"N={int(d['N'])}, t<={d['train_t'][0, -1]:.2f}",
                 f"T_h in [{min(d['train_th'].min(), d['test_th'].min()):.2f},{max(d['train_th'].max(), d['test_th'].max()):.2f}] x "
                 f"L in [{min(d['train_L'].min(), d['test_L'].min()):.1f},{max(d['train_L'].max(), d['test_L'].max()):.1f}]",
                 f"{d['train_T'].shape[0]}/{d['val_T'].shape[0]}/{d['test_T'].shape[0]} (12 used)", d['train_T'].shape[1]])
d = np.load(os.path.join(V, 'data_2d.npz'))
rows.append(['2D equilibrium (convective)', f"{int(d['NC'])}^2, t<={d['test_t'][0, -1]:.2f}",
             f"Ra in [{d['train_Ra'].min():.0e},{d['train_Ra'].max():.0e}] x Ste in [{min(d['train_Ste'].min(), d['test_Ste'].min()):.2f},{max(d['train_Ste'].max(), d['test_Ste'].max()):.2f}]",
             f"{d['train_th'].shape[0]}/{d['val_th'].shape[0]}/{d['test_th'].shape[0]}", d['train_th'].shape[1]])
save(pd.DataFrame(rows, columns=['Dataset', 'Grid, horizon', 'Varied', 'Train/val/test', 'States']).set_index('Dataset'), 'Table II  Datasets')

bf = J('baseline_floors.json')['floors']; fs = J('floors_skip.json')
sc = pd.DataFrame({'E_0': [bf['eq_end']['E0'], bf['kin_end']['E0']], 'S_0': [bf['eq_end']['S0'], np.nan],
                   'F_0': [np.nan, bf['kin_end']['F0']], 'Sigma_0': [np.nan, bf['kin_end']['SIG0']]},
                  index=['1D equilibrium', '1D internal-variable'])
save(sc, 'Table III (a)  Loss scales, endpoint rule')
pair = lambda a, b: f'{a:.3g} / {b:.3g}'
fl = pd.DataFrame({
    'endpoint, dt': [pair(fs['kin_end_skip0']['F0_rel'], fs['kin_end_skip2']['F0_rel']), pair(fs['kin_end_skip0']['E0_rel'], fs['kin_end_skip2']['E0_rel']),
                     pair(fs['eq_end_skip0']['E0_rel'], fs['eq_end_skip2']['E0_rel']), pair(fs['eq_end_skip0']['ident'], fs['eq_end_skip2']['ident'])],
    'trapezoid, dt': [pair(fs['kin_trap_skip0']['F0_rel'], fs['kin_trap_skip2']['F0_rel']), pair(fs['kin_trap_skip0']['E0_rel'], fs['kin_trap_skip2']['E0_rel']),
                      pair(fs['eq_trap_skip0']['E0_rel'], fs['eq_trap_skip2']['E0_rel']), pair(fs['eq_trap_skip0']['ident'], fs['eq_trap_skip2']['ident'])],
    'endpoint, dt/2': [pair(fs['kin_half_end_skip0']['F0_rel'], fs['kin_half_end_skip2']['F0_rel']), pair(fs['kin_half_end_skip0']['E0_rel'], fs['kin_half_end_skip2']['E0_rel']), '--', '--'],
    'trapezoid, dt/2': [pair(fs['kin_half_trap_skip0']['F0_rel'], fs['kin_half_trap_skip2']['F0_rel']), pair(fs['kin_half_trap_skip0']['E0_rel'], fs['kin_half_trap_skip2']['E0_rel']), '--', '--']},
    index=['r_f (internal-variable)', 'r_E (internal-variable)', 'r_E (equilibrium)', '|r_S - r_E/T| / S_0 (equilibrium)'])
save(fl, 'Table III (b)  Relative floors, all pairs / excluding first two macro-steps')

S = J('stats_v2.json')['primary']; S4 = J('stats_round4.json')
from scipy import stats as _st
p = {'P1': S4['P1_plan_margin_primary']['p_tost'], 'P2': S4['P2']['p_tost'], 'P3': S['P3']['p_perm'], 'P4': S['P4']['p_perm']}
order = sorted(p, key=p.get); adj, run_ = {}, 0
for i, k in enumerate(order): run_ = max(run_, min(1, (4 - i) * p[k])); adj[k] = run_
names = {'P1': ('E+S - E+E/T', 'T error'), 'P2': ('E+S - E+E/T', 'front'), 'P3': ('E+K - E', 'f error'), 'P4': ('E+K - E', 'T error')}
t4 = pd.DataFrame([[*names[k], f"{100 * S[k]['rel']:.1f}%", f"[{100 * S[k]['ci95'][0] / S[k]['base_mean']:.1f}, {100 * S[k]['ci95'][1] / S[k]['base_mean']:.1f}]%",
                    f"{100 * S[k]['mde_rel']:.1f}%", f'{adj[k]:.2g}'] for k in ['P1', 'P2', 'P3', 'P4']],
                  columns=['Contrast', 'Metric', 'Difference', '95% CI', 'MDE', 'p (Holm)'], index=['P1', 'P2', 'P3', 'P4'])
save(t4, 'Table IV  Primary contrasts')

R = J('run2d_v2.json')['runs']; R['up192'] = J('run2d_192.json')
t5 = pd.DataFrame([[R[k]['Nu'][-1], R[k]['fbar'][-1], R[k]['front_top'][-1], R[k]['front_mid'][-1], R[k]['front_bot'][-1], R[k]['usolid']]
                   for k in ['up48', 'up96', 'up192', 'up96_cfl015', 'mu96']],
                  columns=['Nu', 'mean f', 'front y=0.95', 'front y=0.5', 'front y=0.05', 'max solid speed'],
                  index=['48^2', '96^2', '192^2', '96^2, dt/2', '96^2, MUSCL'])
save(t5.round(3), 'Table V  Two-dimensional melting at t = 0.25')

## 8. Numbers quoted in the text and appendices

In [ ]:
v1 = json.load(open(os.path.join(ROOT, 'verify1d.json'))); vdd = json.load(open(os.path.join(ROOT, 'vdd.json')))
mms = J('mms_kinetic.json'); sl = J('sharp_limit.json'); a3 = J('audit3.json'); av = J('audit4_velocity.json')
ak = J('audit4_kin.json'); d5 = J('audit5_dg.json'); fr = J('audit5_front.json'); pr = J('audit5_precision.json')
s5 = J('stats_round5.json'); s3 = J('stats_round3.json'); cor = J('corruption_v2.json'); dis = J('dissipation.json')
A2 = J('analysis2.json'); A4 = J('active2d_r4.json'); bfj = J('baseline_floors.json')
show = lambda title, d: (display(Markdown(f'**{title}**')), print(json.dumps(d, indent=1, default=float)))
show('Sec. IV A  verification', dict(neumann_rel_err=[s['rel_err_final'] for s in v1['stefan']],
     mms_orders=[r.get('order_T') for r in mms['mms']], seed_sweep=[(r['f0'], r['front']) for r in mms['f0_sensitivity']],
     mobility_sweep=[(r['M0'], r['rel_to_neumann'], r['T_interface'] - 1) for r in sl['rows']], de_vahl_davis=vdd))
show('Sec. IV B  cellwise ledger (first and last intervals)', [{k: r[k] for k in ('interval', 'exact_rel', 'end_rel', 'trap_rel')} for r in a3['macro_ledger'][:3] + a3['macro_ledger'][-2:]])
show('Sec. V C / Appendix E  admissibility', {v: {k: a3['kinetic_admissibility'][v][k] for k in ('out_frac', 'neg_in_inadm', 'P0_adm', 'Rf_adm', 'Rf_inadm', 'fmin', 'fmax')} for v in ['D', 'E', 'EPH', 'EK']})
show('Appendix E  mobility in inadmissible cells', ak)
show('Sec. V C  interpolation baselines', {k: {m: v['mean'] for m, v in bfj[k].items() if m in ('linear', 'idw')} for k in bfj if 'interp' in k})
show('Sec. VI B  corruption', cor)
show('Sec. VI C  velocity reconstruction', {t: {f: {k: av[t][f][k] for k in ('div_rms_dimless', 'ident', 'ident_corrected')} for f in ('native', 'reconstructed', 'recon_projected')} for t in av})
show('Sec. VI C  reference shifts', A4['gauge'])
show('Sec. VI D  matched contrasts (%)', {k: [round(100 * v['rel'], 1), [round(100 * x, 1) for x in v['ci_rel']]] for k, v in A4['contrasts'].items() if k.endswith('relT') or 'nu' in k})
show('Appendix C  precision', pr); show('Appendix D  equivalence and post-plan family', dict(round3=s3['equivalence_adjusted'], post_plan=s5, front_width=fr))
show('Appendix F  heating', dict(model_mapping=s3['model_mapping_delta0.2'], illustrative=s3['illustrative_10K_300K']))
show('Appendix G  discrete-gradient pair', d5)
show('Fig. 5  tuning and robustness', dict(tuning={k: v['selected'] for k, v in A2['tuning'].items()},
     robustness={k: [round(100 * x / v['relf']['base'], 1) for x in v['relf']['seed_diffs']] for k, v in A2['audit'].items()}))

## 9. Check against the manuscript

In [ ]:
run_py('v2/verify_headlines.py', V, tail=30)

All checks should report `OK`. Outputs: figures in `figures/` (numbered as in the paper), tables as CSV in `tables/`, and every summary file in `v2/`.